# Chapter 16: Storage

> *"Future users of large data banks must be protected from having to
> know how the data is organised in the machine (the internal
> representation)."*  
> — **Edgar F. Codd**, *A Relational Model of Data for Large Shared Data
> Banks* (1970)

------------------------------------------------------------------------

### 16.1 The Ephemeral Machine and the Persistent Domain

Throughout the preceding chapters, our computation has operated almost
exclusively within the transient theatre of volatile memory. When you
assign a new variable in Python or allocate a buffer on the C++ free
store via `std::make_unique`, you are leasing silicon real estate from
the operating system's virtual memory subsystem. The nanosecond access
times of DRAM provide an exhilarating environment for raw computation,
but this speed comes at an absolute cost: **ephemerality**. Cut the
power rails, crash the operating system, or terminate the process host,
and the runtime state dissolves instantaneously into thermal entropy.

Software systems cannot function in a vacuum of transience. Banking
ledgers, social graphs, compiler symbol caches, and flight avionics
demand durability. When state transitions occur, they must endure
machine restarts, operating system panics, and catastrophic hardware
interrupts.

Historically, software engineers stored state by serialising in-memory
structures directly into flat files on disk. However, flat files
collapse the moment multiple processes read and write concurrently. Flat
files force the application layer to understand the byte-for-byte
physical layout of the hard drive: record offsets, padding bytes,
endianness, and disk block alignments. If the disk structure changes,
every line of application code breaks.

In 1970, an Oxford-educated mathematician working at IBM named **Edgar
F. Codd** published a landmark paper that severed this dependency
forever. Codd proposed the **Relational Model of Data**. He argued that
application programmers should never interact with pointers, disk
offsets, or physical memory layouts. Instead, data should be modelled
strictly as **mathematical relations** (sets of tuples), manipulated
through a formal **relational algebra**, and declared via high-level
non-procedural query languages. The underlying database engine was left
entirely free to lay out bytes on disk, balance index trees, and
optimise execution plans in whatever manner yielded the highest
throughput.

(See Figure 16–1 for an outline of this structure.)

<figure id="fig:storage-abstraction">
<img src="data:image/svg+xml;base64,PD94bWwgdmVyc2lvbj0iMS4wIiBlbmNvZGluZz0iVVRGLTgiIHN0YW5kYWxvbmU9Im5vIj8+Cjwh
RE9DVFlQRSBzdmcgUFVCTElDICItLy9XM0MvL0RURCBTVkcgMS4xLy9FTiIKICJodHRwOi8vd3d3
LnczLm9yZy9HcmFwaGljcy9TVkcvMS4xL0RURC9zdmcxMS5kdGQiPgo8IS0tIEdlbmVyYXRlZCBi
eSBncmFwaHZpeiB2ZXJzaW9uIDIuNDMuMCAoMCkKIC0tPgo8IS0tIFRpdGxlOiBHIFBhZ2VzOiAx
IC0tPgo8c3ZnIHdpZHRoPSIzNjhwdCIgaGVpZ2h0PSIxOTNwdCIKIHZpZXdCb3g9IjAuMDAgMC4w
MCAzNjguMDAgMTkzLjAwIiB4bWxucz0iaHR0cDovL3d3dy53My5vcmcvMjAwMC9zdmciIHhtbG5z
OnhsaW5rPSJodHRwOi8vd3d3LnczLm9yZy8xOTk5L3hsaW5rIj4KPGcgaWQ9ImdyYXBoMCIgY2xh
c3M9ImdyYXBoIiB0cmFuc2Zvcm09InNjYWxlKDEgMSkgcm90YXRlKDApIHRyYW5zbGF0ZSg0IDE4
OSkiPgo8dGl0bGU+RzwvdGl0bGU+CjwhLS0gYXBwbGljYXRpb24gLS0+CjxnIGlkPSJub2RlMSIg
Y2xhc3M9Im5vZGUiPgo8dGl0bGU+YXBwbGljYXRpb248L3RpdGxlPgo8cG9seWdvbiBmaWxsPSJu
b25lIiBzdHJva2U9ImJsYWNrIiBwb2ludHM9IjM2MCwtMTg1IDAsLTE4NSAwLC0xNDkgMzYwLC0x
NDkgMzYwLC0xODUiLz4KPHRleHQgdGV4dC1hbmNob3I9Im1pZGRsZSIgeD0iMTgwIiB5PSItMTcw
IiBmb250LWZhbWlseT0iQ291cmllciBOZXcsIERlamFWdSBTYW5zIE1vbm8sIG1vbm9zcGFjZSIg
Zm9udC1zaXplPSIxMC4wMCI+QXBwbGljYXRpb24gRG9tYWluPC90ZXh0Pgo8dGV4dCB0ZXh0LWFu
Y2hvcj0ibWlkZGxlIiB4PSIxODAiIHk9Ii0xNTkiIGZvbnQtZmFtaWx5PSJDb3VyaWVyIE5ldywg
RGVqYVZ1IFNhbnMgTW9ubywgbW9ub3NwYWNlIiBmb250LXNpemU9IjEwLjAwIj4oT2JqZWN0cywg
U3RydWN0cywgQWxnZWJyYWljIERhdGEgVHlwZXMsIER5bmFtaWMgR3JhcGhzKTwvdGV4dD4KPC9n
Pgo8IS0tIHJlbGF0aW9uYWwgLS0+CjxnIGlkPSJub2RlMiIgY2xhc3M9Im5vZGUiPgo8dGl0bGU+
cmVsYXRpb25hbDwvdGl0bGU+Cjxwb2x5Z29uIGZpbGw9Im5vbmUiIHN0cm9rZT0iYmxhY2siIHBv
aW50cz0iMzYwLC0xMDUgMCwtMTA1IDAsLTY5IDM2MCwtNjkgMzYwLC0xMDUiLz4KPHRleHQgdGV4
dC1hbmNob3I9Im1pZGRsZSIgeD0iMTgwIiB5PSItOTAiIGZvbnQtZmFtaWx5PSJDb3VyaWVyIE5l
dywgRGVqYVZ1IFNhbnMgTW9ubywgbW9ub3NwYWNlIiBmb250LXNpemU9IjEwLjAwIj5SZWxhdGlv
bmFsIEVuZ2luZSBDb3JlPC90ZXh0Pgo8dGV4dCB0ZXh0LWFuY2hvcj0ibWlkZGxlIiB4PSIxODAi
IHk9Ii03OSIgZm9udC1mYW1pbHk9IkNvdXJpZXIgTmV3LCBEZWphVnUgU2FucyBNb25vLCBtb25v
c3BhY2UiIGZvbnQtc2l6ZT0iMTAuMDAiPihUdXBsZXMsIFJlbGF0aW9ucywgUHJlZGljYXRlcywg
QWxnZWJyYWljIFRyYW5zZm9ybXMpPC90ZXh0Pgo8L2c+CjwhLS0gYXBwbGljYXRpb24mIzQ1OyZn
dDtyZWxhdGlvbmFsIC0tPgo8ZyBpZD0iZWRnZTEiIGNsYXNzPSJlZGdlIj4KPHRpdGxlPmFwcGxp
Y2F0aW9uJiM0NTsmZ3Q7cmVsYXRpb25hbDwvdGl0bGU+CjxwYXRoIGZpbGw9Im5vbmUiIHN0cm9r
ZT0iYmxhY2siIGQ9Ik0xODAsLTE0OC42OUMxODAsLTEzOC44OSAxODAsLTEyNi40MiAxODAsLTEx
NS4zMyIvPgo8cG9seWdvbiBmaWxsPSJibGFjayIgc3Ryb2tlPSJibGFjayIgcG9pbnRzPSIxODMu
NSwtMTE1LjI2IDE4MCwtMTA1LjI2IDE3Ni41LC0xMTUuMjYgMTgzLjUsLTExNS4yNiIvPgo8dGV4
dCB0ZXh0LWFuY2hvcj0ic3RhcnQiIHg9IjE4MCIgeT0iLTEzMCIgZm9udC1mYW1pbHk9IkNvdXJp
ZXIgTmV3LCBEZWphVnUgU2FucyBNb25vLCBtb25vc3BhY2UiIGZvbnQtc2l6ZT0iMTAuMDAiPiAm
IzE2MDtPYmplY3QmIzQ1O1JlbGF0aW9uYWwgLzwvdGV4dD4KPHRleHQgdGV4dC1hbmNob3I9InN0
YXJ0IiB4PSIxODAiIHk9Ii0xMTkiIGZvbnQtZmFtaWx5PSJDb3VyaWVyIE5ldywgRGVqYVZ1IFNh
bnMgTW9ubywgbW9ub3NwYWNlIiBmb250LXNpemU9IjEwLjAwIj4gJiMxNjA7TWVtb3J5IEltcGVk
YW5jZSBNaXNtYXRjaDwvdGV4dD4KPC9nPgo8IS0tIHBoeXNpY2FsIC0tPgo8ZyBpZD0ibm9kZTMi
IGNsYXNzPSJub2RlIj4KPHRpdGxlPnBoeXNpY2FsPC90aXRsZT4KPHBvbHlnb24gZmlsbD0ibm9u
ZSIgc3Ryb2tlPSJibGFjayIgcG9pbnRzPSIzNjAsLTM2IDAsLTM2IDAsMCAzNjAsMCAzNjAsLTM2
Ii8+Cjx0ZXh0IHRleHQtYW5jaG9yPSJtaWRkbGUiIHg9IjE4MCIgeT0iLTIxIiBmb250LWZhbWls
eT0iQ291cmllciBOZXcsIERlamFWdSBTYW5zIE1vbm8sIG1vbm9zcGFjZSIgZm9udC1zaXplPSIx
MC4wMCI+UGh5c2ljYWwgU3RvcmFnZSBTdWJzeXN0ZW08L3RleHQ+Cjx0ZXh0IHRleHQtYW5jaG9y
PSJtaWRkbGUiIHg9IjE4MCIgeT0iLTEwIiBmb250LWZhbWlseT0iQ291cmllciBOZXcsIERlamFW
dSBTYW5zIE1vbm8sIG1vbm9zcGFjZSIgZm9udC1zaXplPSIxMC4wMCI+QmxvY2sgQnVmZmVycywg
V0FMLCBJbmRpY2VzOiBCJiM0NTtUcmVlcyAvIEJTVHMsIERpc2tzKTwvdGV4dD4KPC9nPgo8IS0t
IHJlbGF0aW9uYWwmIzQ1OyZndDtwaHlzaWNhbCAtLT4KPGcgaWQ9ImVkZ2UyIiBjbGFzcz0iZWRn
ZSI+Cjx0aXRsZT5yZWxhdGlvbmFsJiM0NTsmZ3Q7cGh5c2ljYWw8L3RpdGxlPgo8cGF0aCBmaWxs
PSJub25lIiBzdHJva2U9ImJsYWNrIiBkPSJNMTgwLC02OC43NUMxODAsLTYyIDE4MCwtNTQuMDkg
MTgwLC00Ni41OSIvPgo8cG9seWdvbiBmaWxsPSJibGFjayIgc3Ryb2tlPSJibGFjayIgcG9pbnRz
PSIxODMuNSwtNDYuNDMgMTgwLC0zNi40MyAxNzYuNSwtNDYuNDMgMTgzLjUsLTQ2LjQzIi8+Cjx0
ZXh0IHRleHQtYW5jaG9yPSJzdGFydCIgeD0iMTgwIiB5PSItNTAiIGZvbnQtZmFtaWx5PSJDb3Vy
aWVyIE5ldywgRGVqYVZ1IFNhbnMgTW9ubywgbW9ub3NwYWNlIiBmb250LXNpemU9IjEwLjAwIj4g
JiMxNjA7UGFnZWQgLyBSZWNvcmQgQWNjZXNzPC90ZXh0Pgo8L2c+CjwvZz4KPC9zdmc+Cg==
" alt="" />
<figcaption>Figure 16–1: Levels of Storage Abstraction</figcaption>
</figure>

In this chapter, we bridge the gap between high-level language semantics
and low-level data persistence. We will explore:

1.  The mathematical underpinnings of **Relational Algebra**.
2.  **Indexing Theory**: transitioning from $O(N)$ full-scan sweeps to
    logarithmic $O(\log N)$ tree lookups.
3.  The **ACID** transaction model and the mechanics of atomic
    rollbacks.
4.  An end-to-end experimental implementation: constructing an in-memory
    relational database engine in **Modern C++20**, featuring a custom
    lexical analyser, recursive descent SQL parser, an indexing engine,
    and an algebraic execution pipeline.

### 16.2 Relational Algebra: The Mathematical Foundations

Before SQL was standardised by ANSI in 1986, relational databases were
grounded entirely in set theory and predicate logic. A database is not
fundamentally a collection of "tables" and "spreadsheets"; it is a set
of formal mathematical relations.

#### 16.2.1 Tuples, Relations, and Schemas

The relational model is defined rigorously:

- **Domain ($D$):** A set of atomic values of a uniform data type (e.g.,
  $\mathbb{Z}$, $\mathbb{R}$, strings of ASCII characters, booleans).
- **Attribute ($A$):** A named label associated with a specific domain.
  For example, $\text{Age} \in \mathbb{N}$,
  $\text{Name} \in \text{String}$.
- **Relation Schema ($R$):** A finite list of attributes
  $R = (A_1: D_1, A_2: D_2, \dots, A_k: D_k)$. The integer $k$ is the
  **arity** (or degree) of the relation.
- **Tuple ($t$):** An ordered list of values $(v_1, v_2, \dots, v_k)$
  such that each $v_i \in D_i$. A tuple represents a single proposition
  or record.
- **Relation ($r(R)$):** A finite set of tuples sharing the schema $R$.
  In mathematical terms:
  $$r(R) \subseteq D_1 \times D_2 \times \dots \times D_k$$ Because a
  relation is fundamentally a **set**, two crucial mathematical
  properties follow:
  1.  **No Duplicates:** A relation cannot contain two identical tuples
      ($t_a = t_b \iff a = b$).
  2.  **Unordered:** The ordering of tuples within a relation has no
      semantic meaning.

#### 16.2.2 Fundamental Operators of Relational Algebra

Relational algebra is a **closed formal system**. Every relational
operator takes one or more relations as input and produces a brand-new
relation as output. This property of closure enables arbitrarily deep
nesting and composition of data queries (see Figure 16–2).

<figure id="fig:relational-algebra">
<img src="data:image/svg+xml;base64,PD94bWwgdmVyc2lvbj0iMS4wIiBlbmNvZGluZz0iVVRGLTgiIHN0YW5kYWxvbmU9Im5vIj8+Cjwh
RE9DVFlQRSBzdmcgUFVCTElDICItLy9XM0MvL0RURCBTVkcgMS4xLy9FTiIKICJodHRwOi8vd3d3
LnczLm9yZy9HcmFwaGljcy9TVkcvMS4xL0RURC9zdmcxMS5kdGQiPgo8IS0tIEdlbmVyYXRlZCBi
eSBncmFwaHZpeiB2ZXJzaW9uIDIuNDMuMCAoMCkKIC0tPgo8IS0tIFRpdGxlOiBHIFBhZ2VzOiAx
IC0tPgo8c3ZnIHdpZHRoPSIzMDJwdCIgaGVpZ2h0PSIyMzlwdCIKIHZpZXdCb3g9IjAuMDAgMC4w
MCAzMDEuNTAgMjM5LjAwIiB4bWxucz0iaHR0cDovL3d3dy53My5vcmcvMjAwMC9zdmciIHhtbG5z
OnhsaW5rPSJodHRwOi8vd3d3LnczLm9yZy8xOTk5L3hsaW5rIj4KPGcgaWQ9ImdyYXBoMCIgY2xh
c3M9ImdyYXBoIiB0cmFuc2Zvcm09InNjYWxlKDEgMSkgcm90YXRlKDApIHRyYW5zbGF0ZSg0IDIz
NSkiPgo8dGl0bGU+RzwvdGl0bGU+CjwhLS0gciAtLT4KPGcgaWQ9Im5vZGUxIiBjbGFzcz0ibm9k
ZSI+Cjx0aXRsZT5yPC90aXRsZT4KPHBvbHlnb24gZmlsbD0ibm9uZSIgc3Ryb2tlPSJibGFjayIg
cG9pbnRzPSIxOTguNSwtMjE2IDEzMy41LC0yMTYgMTMzLjUsLTE5NCAxOTguNSwtMTk0IDE5OC41
LC0yMTYiLz4KPHBvbHlnb24gZmlsbD0ibm9uZSIgc3Ryb2tlPSJibGFjayIgcG9pbnRzPSIyMDIu
NSwtMjIwIDEyOS41LC0yMjAgMTI5LjUsLTE5MCAyMDIuNSwtMTkwIDIwMi41LC0yMjAiLz4KPHRl
eHQgdGV4dC1hbmNob3I9Im1pZGRsZSIgeD0iMTY2IiB5PSItMjAyLjUiIGZvbnQtZmFtaWx5PSJD
b3VyaWVyIE5ldywgRGVqYVZ1IFNhbnMgTW9ubywgbW9ub3NwYWNlIiBmb250LXNpemU9IjEwLjAw
Ij5BIHwgQiB8IEM8L3RleHQ+Cjx0ZXh0IHRleHQtYW5jaG9yPSJtaWRkbGUiIHg9Ijk5IiB5PSIt
MjIzIiBmb250LWZhbWlseT0iQ291cmllciBOZXcsIERlamFWdSBTYW5zIE1vbm8sIG1vbm9zcGFj
ZSIgZm9udC1zaXplPSIxMC4wMCI+UmVsYXRpb24gUjwvdGV4dD4KPC9nPgo8IS0tIGpvaW4gLS0+
CjxnIGlkPSJub2RlMiIgY2xhc3M9Im5vZGUiPgo8dGl0bGU+am9pbjwvdGl0bGU+Cjxwb2x5Z29u
IGZpbGw9Im5vbmUiIHN0cm9rZT0iYmxhY2siIHBvaW50cz0iMjY1LC0xNjggMTU3LC0xNjggMTU3
LC0xNDYgMjY1LC0xNDYgMjY1LC0xNjgiLz4KPHRleHQgdGV4dC1hbmNob3I9Im1pZGRsZSIgeD0i
MjExIiB5PSItMTU0LjUiIGZvbnQtZmFtaWx5PSJDb3VyaWVyIE5ldywgRGVqYVZ1IFNhbnMgTW9u
bywgbW9ub3NwYWNlIiBmb250LXNpemU9IjEwLjAwIj5OYXR1cmFsIEpvaW4gKOKLiCk8L3RleHQ+
CjwvZz4KPCEtLSByJiM0NTsmZ3Q7am9pbiAtLT4KPGcgaWQ9ImVkZ2UxIiBjbGFzcz0iZWRnZSI+
Cjx0aXRsZT5yJiM0NTsmZ3Q7am9pbjwvdGl0bGU+CjxwYXRoIGZpbGw9Im5vbmUiIHN0cm9rZT0i
YmxhY2siIGQ9Ik0xNzkuNzUsLTE4OS45NUMxODQuMjksLTE4NS4zIDE4OS40LC0xODAuMDggMTk0
LjE1LC0xNzUuMjIiLz4KPHBvbHlnb24gZmlsbD0iYmxhY2siIHN0cm9rZT0iYmxhY2siIHBvaW50
cz0iMTk2LjcsLTE3Ny42MiAyMDEuMTksLTE2OC4wMyAxOTEuNywtMTcyLjczIDE5Ni43LC0xNzcu
NjIiLz4KPC9nPgo8IS0tIHNlbGVjdGlvbiAtLT4KPGcgaWQ9Im5vZGU0IiBjbGFzcz0ibm9kZSI+
Cjx0aXRsZT5zZWxlY3Rpb248L3RpdGxlPgo8cG9seWdvbiBmaWxsPSJub25lIiBzdHJva2U9ImJs
YWNrIiBwb2ludHM9IjI2NSwtMTI0IDE1NywtMTI0IDE1NywtMTAyIDI2NSwtMTAyIDI2NSwtMTI0
Ii8+Cjx0ZXh0IHRleHQtYW5jaG9yPSJtaWRkbGUiIHg9IjIxMSIgeT0iLTExMC41IiBmb250LWZh
bWlseT0iQ291cmllciBOZXcsIERlamFWdSBTYW5zIE1vbm8sIG1vbm9zcGFjZSIgZm9udC1zaXpl
PSIxMC4wMCI+U2VsZWN0aW9uICjPgyk8L3RleHQ+Cjx0ZXh0IHRleHQtYW5jaG9yPSJtaWRkbGUi
IHg9Ijc4LjUiIHk9Ii0xMjciIGZvbnQtZmFtaWx5PSJDb3VyaWVyIE5ldywgRGVqYVZ1IFNhbnMg
TW9ubywgbW9ub3NwYWNlIiBmb250LXNpemU9IjEwLjAwIj4oRmlsdGVyIFJvd3MgYnkgUHJlZGlj
YXRlKTwvdGV4dD4KPC9nPgo8IS0tIGpvaW4mIzQ1OyZndDtzZWxlY3Rpb24gLS0+CjxnIGlkPSJl
ZGdlMyIgY2xhc3M9ImVkZ2UiPgo8dGl0bGU+am9pbiYjNDU7Jmd0O3NlbGVjdGlvbjwvdGl0bGU+
CjxwYXRoIGZpbGw9Im5vbmUiIHN0cm9rZT0iYmxhY2siIGQ9Ik0yMTEsLTE0NS44M0MyMTEsLTE0
Mi4zMiAyMTEsLTEzOC4yNiAyMTEsLTEzNC4yNCIvPgo8cG9seWdvbiBmaWxsPSJibGFjayIgc3Ry
b2tlPSJibGFjayIgcG9pbnRzPSIyMTQuNSwtMTM0LjA3IDIxMSwtMTI0LjA3IDIwNy41LC0xMzQu
MDcgMjE0LjUsLTEzNC4wNyIvPgo8L2c+CjwhLS0gcyAtLT4KPGcgaWQ9Im5vZGUzIiBjbGFzcz0i
bm9kZSI+Cjx0aXRsZT5zPC90aXRsZT4KPHBvbHlnb24gZmlsbD0ibm9uZSIgc3Ryb2tlPSJibGFj
ayIgcG9pbnRzPSIyODkuNSwtMjE2IDIyNC41LC0yMTYgMjI0LjUsLTE5NCAyODkuNSwtMTk0IDI4
OS41LC0yMTYiLz4KPHBvbHlnb24gZmlsbD0ibm9uZSIgc3Ryb2tlPSJibGFjayIgcG9pbnRzPSIy
OTMuNSwtMjIwIDIyMC41LC0yMjAgMjIwLjUsLTE5MCAyOTMuNSwtMTkwIDI5My41LC0yMjAiLz4K
PHRleHQgdGV4dC1hbmNob3I9Im1pZGRsZSIgeD0iMjU3IiB5PSItMjAyLjUiIGZvbnQtZmFtaWx5
PSJDb3VyaWVyIE5ldywgRGVqYVZ1IFNhbnMgTW9ubywgbW9ub3NwYWNlIiBmb250LXNpemU9IjEw
LjAwIj5DIHwgRCB8IEU8L3RleHQ+Cjx0ZXh0IHRleHQtYW5jaG9yPSJtaWRkbGUiIHg9IjE5MCIg
eT0iLTIyMyIgZm9udC1mYW1pbHk9IkNvdXJpZXIgTmV3LCBEZWphVnUgU2FucyBNb25vLCBtb25v
c3BhY2UiIGZvbnQtc2l6ZT0iMTAuMDAiPlJlbGF0aW9uIFM8L3RleHQ+CjwvZz4KPCEtLSBzJiM0
NTsmZ3Q7am9pbiAtLT4KPGcgaWQ9ImVkZ2UyIiBjbGFzcz0iZWRnZSI+Cjx0aXRsZT5zJiM0NTsm
Z3Q7am9pbjwvdGl0bGU+CjxwYXRoIGZpbGw9Im5vbmUiIHN0cm9rZT0iYmxhY2siIGQ9Ik0yNDIu
OTUsLTE4OS45NUMyMzguMywtMTg1LjMgMjMzLjA4LC0xODAuMDggMjI4LjIyLC0xNzUuMjIiLz4K
PHBvbHlnb24gZmlsbD0iYmxhY2siIHN0cm9rZT0iYmxhY2siIHBvaW50cz0iMjMwLjU3LC0xNzIu
NjIgMjIxLjAzLC0xNjguMDMgMjI1LjYyLC0xNzcuNTcgMjMwLjU3LC0xNzIuNjIiLz4KPC9nPgo8
IS0tIHByb2plY3Rpb24gLS0+CjxnIGlkPSJub2RlNSIgY2xhc3M9Im5vZGUiPgo8dGl0bGU+cHJv
amVjdGlvbjwvdGl0bGU+Cjxwb2x5Z29uIGZpbGw9Im5vbmUiIHN0cm9rZT0iYmxhY2siIHBvaW50
cz0iMjY1LC04MCAxNTcsLTgwIDE1NywtNTggMjY1LC01OCAyNjUsLTgwIi8+Cjx0ZXh0IHRleHQt
YW5jaG9yPSJtaWRkbGUiIHg9IjIxMSIgeT0iLTY2LjUiIGZvbnQtZmFtaWx5PSJDb3VyaWVyIE5l
dywgRGVqYVZ1IFNhbnMgTW9ubywgbW9ub3NwYWNlIiBmb250LXNpemU9IjEwLjAwIj5Qcm9qZWN0
aW9uICjPgCk8L3RleHQ+Cjx0ZXh0IHRleHQtYW5jaG9yPSJtaWRkbGUiIHg9Ijc4LjUiIHk9Ii04
MyIgZm9udC1mYW1pbHk9IkNvdXJpZXIgTmV3LCBEZWphVnUgU2FucyBNb25vLCBtb25vc3BhY2Ui
IGZvbnQtc2l6ZT0iMTAuMDAiPihTZWxlY3QgU3Vic2V0IG9mIENvbHVtbnMpPC90ZXh0Pgo8L2c+
CjwhLS0gc2VsZWN0aW9uJiM0NTsmZ3Q7cHJvamVjdGlvbiAtLT4KPGcgaWQ9ImVkZ2U0IiBjbGFz
cz0iZWRnZSI+Cjx0aXRsZT5zZWxlY3Rpb24mIzQ1OyZndDtwcm9qZWN0aW9uPC90aXRsZT4KPHBh
dGggZmlsbD0ibm9uZSIgc3Ryb2tlPSJibGFjayIgZD0iTTIxMSwtMTAxLjgzQzIxMSwtOTguMzIg
MjExLC05NC4yNiAyMTEsLTkwLjI0Ii8+Cjxwb2x5Z29uIGZpbGw9ImJsYWNrIiBzdHJva2U9ImJs
YWNrIiBwb2ludHM9IjIxNC41LC05MC4wNyAyMTEsLTgwLjA3IDIwNy41LC05MC4wNyAyMTQuNSwt
OTAuMDciLz4KPC9nPgo8IS0tIG5ldyAtLT4KPGcgaWQ9Im5vZGU2IiBjbGFzcz0ibm9kZSI+Cjx0
aXRsZT5uZXc8L3RpdGxlPgo8dGV4dCB0ZXh0LWFuY2hvcj0ibWlkZGxlIiB4PSIyMTEiIHk9Ii0x
NS41IiBmb250LWZhbWlseT0iQ291cmllciBOZXcsIERlamFWdSBTYW5zIE1vbm8sIG1vbm9zcGFj
ZSIgZm9udC1zaXplPSIxMC4wMCI+TmV3IFJlbGF0aW9uPC90ZXh0Pgo8L2c+CjwhLS0gcHJvamVj
dGlvbiYjNDU7Jmd0O25ldyAtLT4KPGcgaWQ9ImVkZ2U1IiBjbGFzcz0iZWRnZSI+Cjx0aXRsZT5w
cm9qZWN0aW9uJiM0NTsmZ3Q7bmV3PC90aXRsZT4KPHBhdGggZmlsbD0ibm9uZSIgc3Ryb2tlPSJi
bGFjayIgZD0iTTIxMSwtNTcuNjFDMjExLC01NC4zMyAyMTEsLTUwLjUzIDIxMSwtNDYuNjMiLz4K
PHBvbHlnb24gZmlsbD0iYmxhY2siIHN0cm9rZT0iYmxhY2siIHBvaW50cz0iMjE0LjUsLTQ2LjMy
IDIxMSwtMzYuMzIgMjA3LjUsLTQ2LjMyIDIxNC41LC00Ni4zMiIvPgo8L2c+CjwvZz4KPC9zdmc+
Cg==
" alt="" />
<figcaption>Figure 16–2: Relational Operations as Data
Queries</figcaption>
</figure>

There are six primitive operators defined by Codd:

**1. Selection ($\sigma$):**

The selection operator filters tuples from a relation $R$ that satisfy a
Boolean propositional formula $P$.

<span id="eq:eq16-1">$$\sigma_P(R) = \{ t \in R \mid P(t) \text{ is true} \}\qquad{(16–1)}$$</span>

*Example:* Given an `Engineers` relation,
$\sigma_{\text{Age} \ge 21}(\text{Engineers})$ extracts all tuples where
the attribute `Age` is at least 21. In SQL, this corresponds directly to
the `WHERE` clause.

**2. Projection ($\pi$):**

The projection operator selects a subset of attributes from the relation
and discards the rest. If duplicate tuples emerge due to discarded
attributes, they are collapsed.

<span id="eq:eq16-2">$$\pi_{A_1, A_2, \dots, A_m}(R) = \{ (t[A_1], t[A_2], \dots, t[A_m]) \mid t \in R \}\qquad{(16–2)}$$</span>

*Example:* $\pi_{\text{Name}, \text{Salary}}(\text{Engineers})$ outputs
a relation with an arity of 2, stripping away addresses, IDs, and ages.
In SQL, this corresponds to `SELECT Name, Salary`.

**3. Cartesian Product ($\times$):**

The Cartesian product (or cross product) combines every tuple of
relation $R$ with every tuple of relation $S$.

<span id="eq:eq16-3">$$R \times S = \{ t \circ s \mid t \in R \land s \in S \}\qquad{(16–3)}$$</span>

If $|R| = N$ and $|S| = M$, then $|R \times S| = N \times M$. The schema
arity becomes $\text{arity}(R) + \text{arity}(S)$.

**4. Set Union ($\cup$):**

Combines all tuples from two relations. Both relations must be
**union-compatible** (they must have identical arity and matching
attribute domains).

<span id="eq:eq16-4">$$R \cup S = \{ t \mid t \in R \lor t \in S \}\qquad{(16–4)}$$</span>

**5. Set Difference ($-$):**

Finds all tuples present in relation $R$ that do not exist in relation
$S$. Both relations must be union-compatible.

<span id="eq:eq16-5">$$R - S = \{ t \mid t \in R \land t \notin S \}\qquad{(16–5)}$$</span>

**6. Rename ($\rho$):**

Renames attributes or the relation itself, avoiding naming collisions
during self-joins.

<span id="eq:eq16-6">$$\rho_{S(B_1, \dots, B_k)}(R)\qquad{(16–6)}$$</span>

#### 16.2.3 Derived Operators: The Natural Join ($\bowtie$)

From these six primitive operators, we derive the most computationally
critical operator in database theory: the **Natural Join**.

A natural join ($R \bowtie S$) computes the Cartesian product
$R \times S$, selects tuples where all shared attributes match, and
projects away the duplicate common columns:

<span id="eq:eq16-7">$$R \bowtie S = \pi_{\text{Attributes}}\left(\sigma_{R.A_1 = S.A_1 \land \dots \land R.A_m = S.A_m}(R \times S)\right)\qquad{(16–7)}$$</span>

### Relational Algebra to SQL Equivalence

The SQL language is simply a syntactic skin over relational algebra
expressions (see Table 16–1).

| Relational Algebra | SQL Construct | Time Complexity (Naive) |
|------------------------|------------------------|------------------------|
| $\sigma_{\text{Predicate}}(R)$ | `SELECT * FROM R WHERE Predicate` | $O(N)$ scan |
| $\pi_{\text{Cols}}(R)$ | `SELECT Cols FROM R` | $O(N)$ scan |
| $R \times S$ | `SELECT * FROM R CROSS JOIN S` | $O(N \times M)$ |
| $R \bowtie S$ | `SELECT * FROM R NATURAL JOIN S` | $O(N \times M)$ naive / $O(N \log M)$ indexed |
| $R \cup S$ | `SELECT * FROM R UNION SELECT * FROM S` | $O((N+M) \log(N+M))$ |

Table 16–1: Relation Algebra Expressions as SQL {#tbl:sq-language}

#### 16.3 The Anatomy of a Relational Database Engine

To translate declarative SQL text into concrete memory transformations,
a database engine operates as a multi-stage pipeline (see Figure 16–3).

<figure id="fig:db-engine">
<img src="data:image/svg+xml;base64,PHN2ZyB2aWV3Qm94PSIyMCA0MCA4ODAgMTMwIiB3aWR0aD0iMTAwJSIgaGVpZ2h0PSIxMzAiIHht
bG5zPSJodHRwOi8vd3d3LnczLm9yZy8yMDAwL3N2ZyI+CiAgPGRlZnM+CiAgICA8c3R5bGU+CiAg
ICAgIDpyb290IHsKICAgICAgICAtLXN2Zy1iZzogdmFyKC0tYmcsICNmZmZmZmYpOwogICAgICAg
IC0tc3ZnLWZnOiB2YXIoLS1mZywgIzAwMDAwMCk7CiAgICAgICAgLS1zdmctdGludC0xLWZhbGxi
YWNrOiAjY2NjY2NjOyAvKiAyMCUgYmxhY2sgKyA4MCUgd2hpdGUgKi8KICAgICAgICAtLXN2Zy10
aW50LTItZmFsbGJhY2s6ICM5OTk5OTk7IC8qIDQwJSBibGFjayArIDYwJSB3aGl0ZSAqLwogICAg
ICAgIC0tc3ZnLXRpbnQtMTogY29sb3ItbWl4KGluIHNyZ2IsIHZhcigtLXN2Zy1mZykgMjAlLCB2
YXIoLS1zdmctYmcpIDgwJSk7CiAgICAgICAgLS1zdmctdGludC0yOiBjb2xvci1taXgoaW4gc3Jn
YiwgdmFyKC0tc3ZnLWZnKSA0MCUsIHZhcigtLXN2Zy1iZykgNjAlKTsKICAgICAgfQogICAgICB0
ZXh0ICAgeyBmb250LWZhbWlseTogIkFyaWFsIE5hcnJvdyIsIEZyZWVTYW5zLCBzYW5zLXNlcmlm
OyB9CiAgICA8L3N0eWxlPgogICAgPGxpbmVhckdyYWRpZW50IGlkPSJib3hHcmFkIiB4MT0iMCUi
IHkxPSIwJSIgeDI9IjEwMCUiIHkyPSIxMDAlIj4KICAgICAgPHN0b3Agb2Zmc2V0PSIwJSIgc3Rv
cC1jb2xvcj0iIzFlMjkzYiIgLz4KICAgICAgPHN0b3Agb2Zmc2V0PSIxMDAlIiBzdG9wLWNvbG9y
PSIjMGYxNzJhIiAvPgogICAgPC9saW5lYXJHcmFkaWVudD4KICAgIDxsaW5lYXJHcmFkaWVudCBp
ZD0iYWNjZW50R3JhZCIgeDE9IjAlIiB5MT0iMCUiIHgyPSIxMDAlIiB5Mj0iMTAwJSI+CiAgICAg
IDxzdG9wIG9mZnNldD0iMCUiIHN0b3AtY29sb3I9IiMzOGJkZjgiIC8+CiAgICAgIDxzdG9wIG9m
ZnNldD0iMTAwJSIgc3RvcC1jb2xvcj0iIzAyODRjNyIgLz4KICAgIDwvbGluZWFyR3JhZGllbnQ+
CiAgICA8ZmlsdGVyIGlkPSJkcm9wU2hhZG93IiB4PSItMTAlIiB5PSItMTAlIiB3aWR0aD0iMTIw
JSIgaGVpZ2h0PSIxMzAlIj4KICAgICAgPGZlRHJvcFNoYWRvdyBkeD0iMiIgZHk9IjQiIHN0ZERl
dmlhdGlvbj0iMyIgZmxvb2QtY29sb3I9IiMwMDAwMDAiIGZsb29kLW9wYWNpdHk9IjAuNCIvPgog
ICAgPC9maWx0ZXI+CiAgPC9kZWZzPgogIDwhLS0gU3RhZ2UgMTogTGV4ZXIgJiBQYXJzZXIgLS0+
CiAgPGcgZmlsdGVyPSJ1cmwoI2Ryb3BTaGFkb3cpIj4KICAgIDxyZWN0IHg9IjMwIiB5PSI1MCIg
d2lkdGg9IjE1MCIgaGVpZ2h0PSIxMTAiIHJ4PSI4IiBmaWxsPSJ2YXIoLS1zdmctdGludC0xLCB2
YXIoLS1zdmctdGludC0xLWZhbGxiYWNrKSkiIHN0cm9rZT0idmFyKC0tc3ZnLWZnKSIgc3Ryb2tl
LXdpZHRoPSIyIi8+CiAgICA8dGV4dCB4PSIxMDUiIHk9Ijg1IiBmaWxsPSJ2YXIoLS1zdmctZmcp
IiBmb250LXNpemU9IjE2IiBmb250LXdlaWdodD0iYm9sZCIgdGV4dC1hbmNob3I9Im1pZGRsZSI+
U1FMIFN0cmluZzwvdGV4dD4KICAgIDx0ZXh0IHg9IjEwNSIgeT0iMTE1IiBmaWxsPSJ2YXIoLS1z
dmctZmcpIiBmb250LXNpemU9IjE0IiB0ZXh0LWFuY2hvcj0ibWlkZGxlIj5MZXhlciArIFBhcnNl
cjwvdGV4dD4KICAgIDx0ZXh0IHg9IjEwNSIgeT0iMTM1IiBmaWxsPSJ2YXIoLS1zdmctZmcpIiBm
b250LXNpemU9IjE0IiB0ZXh0LWFuY2hvcj0ibWlkZGxlIj5Qcm9kdWNlcyBBU1Q8L3RleHQ+CiAg
PC9nPgogIDwhLS0gQXJyb3cgMSAtLT4KICA8cGF0aCBkPSJNIDE4MCAxMDUgTCAyMzAgMTA1IiBz
dHJva2U9InZhcigtLXN2Zy1mZykiIHN0cm9rZS13aWR0aD0iMyIgZmlsbD0ibm9uZSIgbWFya2Vy
LWVuZD0idXJsKCNhcnJvd2hlYWQpIi8+CiAgPHBvbHlnb24gcG9pbnRzPSIyNDAsMTA1IDIzMCwx
MDAgMjMwLDExMCIgZmlsbD0idmFyKC0tc3ZnLWZnIi8+CiAgPCEtLSBTdGFnZSAyOiBMb2dpY2Fs
IFF1ZXJ5IFBsYW5uZXIgLS0+CiAgPGcgZmlsdGVyPSJ1cmwoI2Ryb3BTaGFkb3cpIj4KICAgIDxy
ZWN0IHg9IjI0MCIgeT0iNTAiIHdpZHRoPSIxNjAiIGhlaWdodD0iMTEwIiByeD0iOCIgZmlsbD0i
dmFyKC0tc3ZnLXRpbnQtMSwgdmFyKC0tc3ZnLXRpbnQtMS1mYWxsYmFjaykpIiBzdHJva2U9InZh
cigtLXN2Zy1mZykiIHN0cm9rZS13aWR0aD0iMiIvPgogICAgPHRleHQgeD0iMzIwIiB5PSI4NSIg
ZmlsbD0idmFyKC0tc3ZnLWZnKSIgZm9udC1zaXplPSIxNiIgZm9udC13ZWlnaHQ9ImJvbGQiIHRl
eHQtYW5jaG9yPSJtaWRkbGUiPlF1ZXJ5IFBsYW5uZXI8L3RleHQ+CiAgICA8dGV4dCB4PSIzMjAi
IHk9IjExNSIgZmlsbD0idmFyKC0tc3ZnLWZnKSIgZm9udC1zaXplPSIxNCIgdGV4dC1hbmNob3I9
Im1pZGRsZSI+TG9naWNhbCBQbGFuPC90ZXh0PgogICAgPHRleHQgeD0iMzIwIiB5PSIxMzUiIGZp
bGw9InZhcigtLXN2Zy1mZykiIGZvbnQtc2l6ZT0iMTQiIHRleHQtYW5jaG9yPSJtaWRkbGUiPlJl
bGF0aW9uYWwgQWxnIFRyZWU8L3RleHQ+CiAgPC9nPgogIDwhLS0gQXJyb3cgMiAtLT4KICA8cGF0
aCBkPSJNIDQwMCAxMDUgTCA0NTAgMTA1IiBzdHJva2U9InZhcigtLXN2Zy1mZykiIHN0cm9rZS13
aWR0aD0iMyIgZmlsbD0ibm9uZSIvPgogIDxwb2x5Z29uIHBvaW50cz0iNDYwLDEwNSA0NTAsMTAw
IDQ1MCwxMTAiIGZpbGw9InZhcigtLXN2Zy1mZykiLz4KICA8IS0tIFN0YWdlIDM6IEV4ZWN1dGlv
biBFbmdpbmUgLS0+CiAgPGcgZmlsdGVyPSJ1cmwoI2Ryb3BTaGFkb3cpIj4KICAgIDxyZWN0IHg9
IjQ2MCIgeT0iNTAiIHdpZHRoPSIxODAiIGhlaWdodD0iMTEwIiByeD0iOCIgZmlsbD0idmFyKC0t
c3ZnLXRpbnQtMSwgdmFyKC0tc3ZnLXRpbnQtMS1mYWxsYmFjaykpIiBzdHJva2U9InZhcigtLXN2
Zy1mZykiIHN0cm9rZS13aWR0aD0iMiIvPgogICAgPHRleHQgeD0iNTUwIiB5PSI4NSIgZmlsbD0i
dmFyKC0tc3ZnLWZnKSIgZm9udC1zaXplPSIxNiIgZm9udC13ZWlnaHQ9ImJvbGQiIHRleHQtYW5j
aG9yPSJtaWRkbGUiPkV4ZWN1dGlvbiBFbmdpbmU8L3RleHQ+CiAgICA8dGV4dCB4PSI1NTAiIHk9
IjExNSIgZmlsbD0idmFyKC0tc3ZnLWZnKSIgZm9udC1zaXplPSIxNCIgdGV4dC1hbmNob3I9Im1p
ZGRsZSI+SXRlcmF0b3JzIC8gUGlwZWxpbmluZzwvdGV4dD4KICAgIDx0ZXh0IHg9IjU1MCIgeT0i
MTM1IiBmaWxsPSJ2YXIoLS1zdmctZmcpIiBmb250LXNpemU9IjE0IiB0ZXh0LWFuY2hvcj0ibWlk
ZGxlIj5TY2FuIC8gRmlsdGVyIC8gUHJvamVjdDwvdGV4dD4KICA8L2c+CiAgPCEtLSBBcnJvdyAz
IC0tPgogIDxwYXRoIGQ9Ik0gNjQwIDEwNSBMIDY5MCAxMDUiIHN0cm9rZT0idmFyKC0tc3ZnLWZn
KSIgc3Ryb2tlLXdpZHRoPSIzIiBmaWxsPSJub25lIi8+CiAgPHBvbHlnb24gcG9pbnRzPSI3MDAs
MTA1IDY5MCwxMDAgNjkwLDExMCIgZmlsbD0idmFyKC0tc3ZnLWZnKSIvPgogIDwhLS0gU3RhZ2Ug
NDogU3RvcmFnZSBNYW5hZ2VyICYgSW5kZXggLS0+CiAgPGcgZmlsdGVyPSJ1cmwoI2Ryb3BTaGFk
b3cpIj4KICAgIDxyZWN0IHg9IjcwMCIgeT0iNTAiIHdpZHRoPSIxNzAiIGhlaWdodD0iMTEwIiBy
eD0iOCIgZmlsbD0idmFyKC0tc3ZnLXRpbnQtMSwgdmFyKC0tc3ZnLXRpbnQtMS1mYWxsYmFjaykp
IiBzdHJva2U9InZhcigtLXN2Zy1mZykiIHN0cm9rZS13aWR0aD0iMiIvPgogICAgPHRleHQgeD0i
Nzg1IiB5PSI4NSIgZmlsbD0idmFyKC0tc3ZnLWZnKSIgZm9udC1zaXplPSIxNiIgZm9udC13ZWln
aHQ9ImJvbGQiIHRleHQtYW5jaG9yPSJtaWRkbGUiPlN0b3JhZ2UgYW5kIEluZGljZXM8L3RleHQ+
CiAgICA8dGV4dCB4PSI3NzUiIHk9IjExNSIgZmlsbD0idmFyKC0tc3ZnLWZnKSIgZm9udC1zaXpl
PSIxNCIgdGV4dC1hbmNob3I9Im1pZGRsZSI+Um93IEhlYXAgLyBUYWJsZXM8L3RleHQ+CiAgICA8
dGV4dCB4PSI3ODUiIHk9IjEzNSIgZmlsbD0idmFyKC0tc3ZnLWZnKSIgZm9udC1zaXplPSIxNCIg
dGV4dC1hbmNob3I9Im1pZGRsZSI+VHJlZSBJbmRpY2VzIChPKGxvZyBOKSk8L3RleHQ+CiAgPC9n
Pgo8L3N2Zz4=
" alt="Figure 16–3: Relational Query Processor Execution Pipeline" />
<figcaption>Figure 16–3: Relational Query Processor Execution
Pipeline</figcaption>
</figure>

1.  **Frontend Lexer and Parser:** The raw SQL string is converted into
    discrete tokens (keywords, literals, identifiers) and parsed into an
    **Abstract Syntax Tree (AST)** using formal grammar rules.

2.  **Planner and Optimiser:** Converts the AST into a directed acyclic
    graph (DAG) of relational algebra operators. An optimiser rearranges
    operations (e.g., pushing selections down before joins to minimise
    intermediate relation cardinality).

3.  **Execution Engine:** Evaluates the plan against physical tables.
    Many modern engines use the **Volcano Iterator Model**, where each
    operator node implements a standard interface:

    ``` cpp
    struct ExecutionNode {
        virtual void open() = 0;
        virtual std::optional<Row> next() = 0;
        virtual void close() = 0;
        virtual ~ExecutionNode() = default;
    };
    ```

4.  **Storage and Indexing Subsystem:** Manages raw in-memory or on-disk
    byte arrays, handles schema validation, allocates row IDs, and
    maintains search structures.

### 16.4 Indexing Structures: Accelerating Lookups

Consider a table `Users` with $N = 10,000,000$ rows. If we execute:

``` sql
SELECT * FROM Users WHERE id = 842101;
```

Without an auxiliary search structure, the engine has no choice but to
perform a **Sequential Scan** (Full Table Scan):

$$\text{Time Complexity} = O(N)$$

At $10^7$ rows, evaluating memory locations sequentially takes tens to
hundreds of milliseconds. When chained across joins, this $O(N)$ cost
cascades into catastrophic $O(N \times M)$ operational deadlocks.

An **Index** is an auxiliary data structure that maps an attribute's
value directly to the physical storage location (or row identifier) of
the tuple containing that value (Figure 16–4).

<figure id="fig:db-index">
<img src="data:image/svg+xml;base64,PHN2ZyB2aWV3Qm94PSI0MCAyMCA4MTAgMjMwIiB3aWR0aD0iMTAwJSIgaGVpZ2h0PSIyMzAiIHZl
cnNpb249IjEuMSIgaWQ9InN2ZzE1IiB4bWxucz0iaHR0cDovL3d3dy53My5vcmcvMjAwMC9zdmci
PgogIDxkZWZzPgogICAgPHN0eWxlPgogICAgICA6cm9vdCB7CiAgICAgICAgLS1zdmctYmc6IHZh
cigtLWJnLCAjZmZmZmZmKTsKICAgICAgICAtLXN2Zy1mZzogdmFyKC0tZmcsICMwMDAwMDApOwog
ICAgICAgIC0tc3ZnLXRpbnQtMS1mYWxsYmFjazogI2NjY2NjYzsgLyogMjAlIGJsYWNrICsgODAl
IHdoaXRlICovCiAgICAgICAgLS1zdmctdGludC0yLWZhbGxiYWNrOiAjOTk5OTk5OyAvKiA0MCUg
YmxhY2sgKyA2MCUgd2hpdGUgKi8KICAgICAgICAtLXN2Zy10aW50LTE6IGNvbG9yLW1peChpbiBz
cmdiLCB2YXIoLS1zdmctZmcpIDIwJSwgdmFyKC0tc3ZnLWJnKSA4MCUpOwogICAgICAgIC0tc3Zn
LXRpbnQtMjogY29sb3ItbWl4KGluIHNyZ2IsIHZhcigtLXN2Zy1mZykgNDAlLCB2YXIoLS1zdmct
YmcpIDYwJSk7CiAgICAgIH0KICAgICAgdGV4dCAgIHsgZm9udC1mYW1pbHk6ICJBcmlhbCBOYXJy
b3ciLCBGcmVlU2Fucywgc2Fucy1zZXJpZjsgfQogICAgPC9zdHlsZT4KICA8L2RlZnM+CiAgPCEt
LSBJbmRleCBUcmVlIFZpc3VhbGl6YXRpb24gKExlZnQgU2lkZSkgLS0+CiAgPGcgdHJhbnNmb3Jt
PSJ0cmFuc2xhdGUoNDAsIDIwKSIgaWQ9Imc1Ij4KICAgIDx0ZXh0IHg9IjE4MCIgeT0iMjAiIGZp
bGw9InZhcigtLXN2Zy1mZykiIGZvbnQtc2l6ZT0iMTYiIGZvbnQtd2VpZ2h0PSJib2xkIiB0ZXh0
LWFuY2hvcj0ibWlkZGxlIiBpZD0idGV4dDEiPkJTVCAvIEItVHJlZSBJbmRleCAoS2V5OiBVc2Vy
IElEKTwvdGV4dD4KICAgIDwhLS0gUm9vdCBOb2RlICg1MCkgLS0+CiAgICA8cmVjdCB4PSIxNDAi
IHk9IjQwIiB3aWR0aD0iODAiIGhlaWdodD0iMzAiIHJ4PSI2IiBmaWxsPSJ2YXIoLS1zdmctdGlu
dC0xLCB2YXIoLS1zdmctdGludC0xLWZhbGxiYWNrKSkiIHN0cm9rZT0idmFyKC0tc3ZnLWZnKSIg
c3Ryb2tlLXdpZHRoPSIyIiBpZD0icmVjdDIiIC8+CiAgICA8dGV4dCB4PSIxODAiIHk9IjYwIiBm
aWxsPSJ2YXIoLS1zdmctZmcpIiBmb250LXNpemU9IjE0IiB0ZXh0LWFuY2hvcj0ibWlkZGxlIiBp
ZD0idGV4dDIiPktleTogNTA8L3RleHQ+CiAgICA8IS0tIEJyYW5jaCBMZWZ0ICgyMCkgJiBSaWdo
dCAoNzUpIC0tPgogICAgPGxpbmUgeDE9IjE1NSIgeTE9IjcwIiB4Mj0iOTAiIHkyPSIxMDUiIHN0
cm9rZT0idmFyKC0tc3ZnLWZnKSIgc3Ryb2tlLXdpZHRoPSIyIiBpZD0ibGluZTIiIC8+CiAgICA8
bGluZSB4MT0iMjA1IiB5MT0iNzAiIHgyPSIyNzAiIHkyPSIxMDUiIHN0cm9rZT0idmFyKC0tc3Zn
LWZnKSIgc3Ryb2tlLXdpZHRoPSIyIiBpZD0ibGluZTMiIC8+CiAgICA8cmVjdCB4PSI1MCIgeT0i
MTA1IiB3aWR0aD0iODAiIGhlaWdodD0iMzAiIHJ4PSI2IiBmaWxsPSJ2YXIoLS1zdmctdGludC0x
LCB2YXIoLS1zdmctdGludC0xLWZhbGxiYWNrKSkiIHN0cm9rZT0idmFyKC0tc3ZnLWZnKSIgc3Ry
b2tlLXdpZHRoPSIxLjUiIGlkPSJyZWN0MyIgLz4KICAgIDx0ZXh0IHg9IjkwIiB5PSIxMjUiIGZp
bGw9InZhcigtLXN2Zy1mZykiIGZvbnQtc2l6ZT0iMTQiIHRleHQtYW5jaG9yPSJtaWRkbGUiIGlk
PSJ0ZXh0MyI+S2V5OiAyMDwvdGV4dD4KICAgIDxyZWN0IHg9IjIzMCIgeT0iMTA1IiB3aWR0aD0i
ODAiIGhlaWdodD0iMzAiIHJ4PSI2IiBmaWxsPSJ2YXIoLS1zdmctdGludC0xLCB2YXIoLS1zdmct
dGludC0xLWZhbGxiYWNrKSkiIHN0cm9rZT0idmFyKC0tc3ZnLWZnKSIgc3Ryb2tlLXdpZHRoPSIy
IiBpZD0icmVjdDQiIC8+CiAgICA8dGV4dCB4PSIyNzAiIHk9IjEyNSIgZmlsbD0idmFyKC0tc3Zn
LWZnKSIgZm9udC1zaXplPSIxNCIgZm9udC13ZWlnaHQ9ImJvbGQiIHRleHQtYW5jaG9yPSJtaWRk
bGUiIGlkPSJ0ZXh0NCI+S2V5OiA3NTwvdGV4dD4KICAgIDwhLS0gQnJhbmNoIFJpZ2h0LVJpZ2h0
ICg5MCkgLS0+CiAgICA8bGluZSB4MT0iMjkwIiB5MT0iMTM1IiB4Mj0iMzMwIiB5Mj0iMTcwIiBz
dHJva2U9InZhcigtLXN2Zy1mZykiIHN0cm9rZS13aWR0aD0iMiIgaWQ9ImxpbmU0IiAvPgogICAg
PHJlY3QgeD0iMjkwIiB5PSIxNzAiIHdpZHRoPSI4MCIgaGVpZ2h0PSIzMCIgcng9IjYiIGZpbGw9
InZhcigtLXN2Zy10aW50LTEsIHZhcigtLXN2Zy10aW50LTEtZmFsbGJhY2spKSIgc3Ryb2tlPSJ2
YXIoLS1zdmctZmcpIiBzdHJva2Utd2lkdGg9IjIiIGlkPSJyZWN0NSIgLz4KICAgIDx0ZXh0IHg9
IjMzMCIgeT0iMTkwIiBmaWxsPSJ2YXIoLS1zdmctZmcpIiBmb250LXNpemU9IjE0IiBmb250LXdl
aWdodD0iYm9sZCIgdGV4dC1hbmNob3I9Im1pZGRsZSIgaWQ9InRleHQ1Ij5LZXk6IDkwPC90ZXh0
PgogIDwvZz4KICA8IS0tIENvbm5lY3Rpb24gUG9pbnRlciBmcm9tIEtleSA5MCB0byBIZWFwIFJv
dyAyIC0tPgogIDxwYXRoIGQ9Ik0gNDEwIDIwNSBDIDQ3MCAyMDUsIDQ4MCAxNDUsIDUyMCAxNDUi
IGZpbGw9Im5vbmUiIHN0cm9rZT0idmFyKC0tc3ZnLWZnKSIgc3Ryb2tlLXdpZHRoPSIyLjUiIHN0
cm9rZS1kYXNoYXJyYXk9IjYsNCIgaWQ9InBhdGg1IiAvPgogIDxwb2x5Z29uIHBvaW50cz0iNTIw
LDE0NSA1MTAsMTQwIDUxMCwxNTAiIGZpbGw9InZhcigtLXN2Zy1mZykiIGlkPSJwb2x5Z29uNSIg
Lz4KICA8dGV4dCB4PSI0NTUiIHk9IjEzNSIgZmlsbD0idmFyKC0tc3ZnLWZnKSIgZm9udC1zaXpl
PSIxNCIgdGV4dC1hbmNob3I9Im1pZGRsZSIgaWQ9InRleHQ2Ij5EaXJlY3QgUm93IFBvaW50ZXI8
L3RleHQ+CiAgPCEtLSBQaHlzaWNhbCBEYXRhIFRhYmxlIC8gUm93IFN0b3JhZ2UgKFJpZ2h0IFNp
ZGUpIC0tPgogIDxnIHRyYW5zZm9ybT0idHJhbnNsYXRlKDUzMCwgMzUpIiBpZD0iZzE1Ij4KICAg
IDx0ZXh0IHg9IjE0MCIgeT0iMTUiIGZpbGw9InZhcigtLXN2Zy1mZykiIGZvbnQtc2l6ZT0iMTYi
IGZvbnQtd2VpZ2h0PSJib2xkIiB0ZXh0LWFuY2hvcj0ibWlkZGxlIiBpZD0idGV4dDciPlRhYmxl
IFN0b3JhZ2UgKFJvdyBIZWFwKTwvdGV4dD4KICAgIDwhLS0gUm93IDAgLS0+CiAgICA8cmVjdCB4
PSIwIiB5PSIzMCIgd2lkdGg9IjI4MCIgaGVpZ2h0PSIzMCIgZmlsbD0idmFyKC0tc3ZnLXRpbnQt
MSwgdmFyKC0tc3ZnLXRpbnQtMS1mYWxsYmFjaykpIiBzdHJva2U9InZhcigtLXN2Zy1mZykiIGlk
PSJyZWN0NyIgLz4KICAgIDx0ZXh0IHg9IjE1IiB5PSI1MCIgZmlsbD0idmFyKC0tc3ZnLWZnKSIg
Zm9udC1zaXplPSIxNCIgaWQ9InRleHQ4Ij5Sb3dJRDogMDwvdGV4dD4KICAgIDx0ZXh0IHg9IjEw
MCIgeT0iNTAiIGZpbGw9InZhcigtLXN2Zy1mZykiIGZvbnQtc2l6ZT0iMTQiIGlkPSJ0ZXh0OSI+
MjAgfCAmcXVvdDtBZGEmcXVvdDsmI3hhMDsmI3hhMDsmI3hhMDsmI3hhMDt8IDQ1MDAuMDwvdGV4
dD4KICAgIDwhLS0gUm93IDEgLS0+CiAgICA8cmVjdCB4PSIwIiB5PSI2NSIgd2lkdGg9IjI4MCIg
aGVpZ2h0PSIzMCIgZmlsbD0idmFyKC0tc3ZnLXRpbnQtMSwgdmFyKC0tc3ZnLXRpbnQtMS1mYWxs
YmFjaykpIiBzdHJva2U9InZhcigtLXN2Zy1mZykiIGlkPSJyZWN0OSIgLz4KICAgIDx0ZXh0IHg9
IjE1IiB5PSI4NSIgZmlsbD0idmFyKC0tc3ZnLWZnKSIgZm9udC1zaXplPSIxNCIgaWQ9InRleHQx
MCI+Um93SUQ6IDE8L3RleHQ+CiAgICA8dGV4dCB4PSIxMDAiIHk9Ijg1IiBmaWxsPSJ2YXIoLS1z
dmctZmcpIiBmb250LXNpemU9IjE0IiBpZD0idGV4dDExIj41MCB8ICZxdW90O0FsYW4mcXVvdDsm
I3hhMDsmI3hhMDsmI3hhMDt8IDUyMDAuMDwvdGV4dD4KICAgIDwhLS0gUm93IDIgKFRhcmdldCkg
LS0+CiAgICA8cmVjdCB4PSIwIiB5PSIxMDAiIHdpZHRoPSIyODAiIGhlaWdodD0iMzAiIGZpbGw9
InZhcigtLXN2Zy10aW50LTEsIHZhcigtLXN2Zy10aW50LTEtZmFsbGJhY2spKSIgc3Ryb2tlPSJ2
YXIoLS1zdmctZmcpIiBzdHJva2Utd2lkdGg9IjIiIGlkPSJyZWN0MTEiIC8+CiAgICA8dGV4dCB4
PSIxNSIgeT0iMTIwIiBmaWxsPSJ2YXIoLS1zdmctZmcpIiBmb250LXNpemU9IjE0IiBmb250LXdl
aWdodD0iYm9sZCIgaWQ9InRleHQxMiI+Um93SUQ6IDI8L3RleHQ+CiAgICA8dGV4dCB4PSIxMDAi
IHk9IjEyMCIgZmlsbD0idmFyKC0tc3ZnLWZnKSIgZm9udC1zaXplPSIxNCIgZm9udC13ZWlnaHQ9
ImJvbGQiIGlkPSJ0ZXh0MTMiPjkwIHwgJnF1b3Q7R3JhY2UmcXVvdDsmI3hhMDsmI3hhMDt8IDYx
MDAuMDwvdGV4dD4KICAgIDwhLS0gUm93IDMgLS0+CiAgICA8cmVjdCB4PSIwIiB5PSIxMzUiIHdp
ZHRoPSIyODAiIGhlaWdodD0iMzAiIGZpbGw9InZhcigtLXN2Zy10aW50LTEsIHZhcigtLXN2Zy10
aW50LTEtZmFsbGJhY2spKSIgc3Ryb2tlPSJ2YXIoLS1zdmctZmcpIiBpZD0icmVjdDEzIiAvPgog
ICAgPHRleHQgeD0iMTUiIHk9IjE1NSIgZmlsbD0idmFyKC0tc3ZnLWZnKSIgZm9udC1zaXplPSIx
NCIgaWQ9InRleHQxNCI+Um93SUQ6IDM8L3RleHQ+CiAgICA8dGV4dCB4PSIxMDAiIHk9IjE1NSIg
ZmlsbD0idmFyKC0tc3ZnLWZnKSIgZm9udC1zaXplPSIxNCIgaWQ9InRleHQxNSI+NzUgfCAmcXVv
dDtFZHNnZXImcXVvdDsmI3hhMDt8IDQ4MDAuMDwvdGV4dD4KICA8L2c+Cjwvc3ZnPg==
" alt="Figure 16–4: BST/B-Tree Index" />
<figcaption>Figure 16–4: BST/B-Tree Index</figcaption>
</figure>

By building a **Balanced Binary Search Tree** or **B-Tree** index over
the attribute `id`, lookup time drops exponentially:

$$\text{Time Complexity} = O(\log N)$$

For $10,000,000$ rows, $\log_2(10^7) \approx 24$ comparisons. An
operation that previously took $10,000,000$ checks now completes in
roughly two dozen pointer traversals.

### 16.5 Transactions and the ACID Guarantees

Databases operate under hostile conditions: power rails can drop, memory
can become exhausted, assertions can fail, and concurrent threads can
attempt to mutate the exact same record simultaneously. To maintain
operational correctness, database transactions must uphold four
invariant guarantees, formally known as **ACID**:

1.  **Atomicity:** The "all-or-nothing" guarantee. A transaction
    consists of a discrete sequence of operations. If every operation
    succeeds, the transaction commits. If a single constraint check,
    system call, or arithmetic assertion fails, the entire transaction
    is **aborted** and **rolled back**. The database state must appear
    as though the transaction never began.
2.  **Consistency:** A transaction may only transition the database from
    one valid state to another valid state. All declared
    constraints—such as `NOT NULL`, `UNIQUE`, and `PRIMARY KEY`—must
    hold invariant at the point of commit.
3.  **Isolation:** Concurrent transactions must execute without leaking
    intermediate states to one another. The observable result of
    executing multiple transactions concurrently must be identical to
    running them in some strict sequential order (**Serialisability**).
4.  **Durability:** Once a transaction commits, its mutations are
    permanent. Even if the hardware loses power immediately following a
    commit acknowledgment, the state changes persist (historically
    achieved via non-volatile **Write-Ahead Logging (WAL)**).

In our Modern C++ engine implementation, we will explicitly enforce
**Atomicity** and **Consistency** through an in-memory **Undo-Log /
Rollback Journal** coupled with strict constraint validators.

### 16.6 The In-Memory Architecture in Modern C++

A relational database engine is to be constructed from first principles
in Modern C++20. We will organise our system into four cohesive layers
(as also outlined in Figure 16–5):

1.  **Algebraic Type Representation (`Value`):** A type-safe,
    variant-based storage model with strong comparison semantics.
2.  **Schema and Table Structure (`Schema`, `Table`):** Value-semantic
    storage with primary key and constraint enforcement.
3.  **Binary Search Tree Index (`Index`):** An associative $O(\log N)$
    search structure mapping column values to internal `RowID` sets.
4.  **SQL Frontend (`Lexer`, `Parser`, `QueryEngine`):** A full
    recursive-descent parser executing mini-SQL queries over our
    relational algebra primitives.

<figure id="fig:statement-to-result">
<img src="data:image/svg+xml;base64,PD94bWwgdmVyc2lvbj0iMS4wIiBlbmNvZGluZz0iVVRGLTgiIHN0YW5kYWxvbmU9Im5vIj8+Cjwh
RE9DVFlQRSBzdmcgUFVCTElDICItLy9XM0MvL0RURCBTVkcgMS4xLy9FTiIKICJodHRwOi8vd3d3
LnczLm9yZy9HcmFwaGljcy9TVkcvMS4xL0RURC9zdmcxMS5kdGQiPgo8IS0tIEdlbmVyYXRlZCBi
eSBncmFwaHZpeiB2ZXJzaW9uIDIuNDMuMCAoMCkKIC0tPgo8IS0tIFRpdGxlOiBHIFBhZ2VzOiAx
IC0tPgo8c3ZnIHdpZHRoPSIzNjhwdCIgaGVpZ2h0PSIzMzRwdCIKIHZpZXdCb3g9IjAuMDAgMC4w
MCAzNjguMDAgMzM0LjAwIiB4bWxucz0iaHR0cDovL3d3dy53My5vcmcvMjAwMC9zdmciIHhtbG5z
OnhsaW5rPSJodHRwOi8vd3d3LnczLm9yZy8xOTk5L3hsaW5rIj4KPGcgaWQ9ImdyYXBoMCIgY2xh
c3M9ImdyYXBoIiB0cmFuc2Zvcm09InNjYWxlKDEgMSkgcm90YXRlKDApIHRyYW5zbGF0ZSg0IDMz
MCkiPgo8dGl0bGU+RzwvdGl0bGU+CjwhLS0gc3FsIC0tPgo8ZyBpZD0ibm9kZTEiIGNsYXNzPSJu
b2RlIj4KPHRpdGxlPnNxbDwvdGl0bGU+Cjxwb2x5Z29uIGZpbGw9Im5vbmUiIHN0cm9rZT0iYmxh
Y2siIHBvaW50cz0iMzYwLC0zMjYgMCwtMzI2IDAsLTI5MCAzNjAsLTI5MCAzNjAsLTMyNiIvPgo8
dGV4dCB0ZXh0LWFuY2hvcj0ibWlkZGxlIiB4PSIxODAiIHk9Ii0zMTEiIGZvbnQtZmFtaWx5PSJD
b3VyaWVyIE5ldywgRGVqYVZ1IFNhbnMgTW9ubywgbW9ub3NwYWNlIiBmb250LXNpemU9IjEwLjAw
Ij5TUUwgU3RhdGVtZW50PC90ZXh0Pgo8dGV4dCB0ZXh0LWFuY2hvcj0ibWlkZGxlIiB4PSIxODAi
IHk9Ii0zMDAiIGZvbnQtZmFtaWx5PSJDb3VyaWVyIE5ldywgRGVqYVZ1IFNhbnMgTW9ubywgbW9u
b3NwYWNlIiBmb250LXNpemU9IjEwLjAwIj4mcXVvdDtTRUxFQ1QgbmFtZSwgc2FsYXJ5IEZST00g
ZW5naW5lZXJzIFdIRVJFIGFnZSAmZ3Q7PSAyMTsmcXVvdDs8L3RleHQ+CjwvZz4KPCEtLSBsZXhl
ciAtLT4KPGcgaWQ9Im5vZGUyIiBjbGFzcz0ibm9kZSI+Cjx0aXRsZT5sZXhlcjwvdGl0bGU+Cjxw
b2x5Z29uIGZpbGw9Im5vbmUiIHN0cm9rZT0iYmxhY2siIHBvaW50cz0iMzYwLC0yNjggMCwtMjY4
IDAsLTIzMiAzNjAsLTIzMiAzNjAsLTI2OCIvPgo8dGV4dCB0ZXh0LWFuY2hvcj0ibWlkZGxlIiB4
PSIxODAiIHk9Ii0yNTMiIGZvbnQtZmFtaWx5PSJDb3VyaWVyIE5ldywgRGVqYVZ1IFNhbnMgTW9u
bywgbW9ub3NwYWNlIiBmb250LXNpemU9IjEwLjAwIj5MZXhlciAoVG9rZW4gU3RyZWFtKTwvdGV4
dD4KPHRleHQgdGV4dC1hbmNob3I9Im1pZGRsZSIgeD0iMTgwIiB5PSItMjQyIiBmb250LWZhbWls
eT0iQ291cmllciBOZXcsIERlamFWdSBTYW5zIE1vbm8sIG1vbm9zcGFjZSIgZm9udC1zaXplPSIx
MC4wMCI+W1NFTEVDVCwgSURFTlQoJnF1b3Q7bmFtZSZxdW90OyksIENPTU1BLCBJREVOVCgmcXVv
dDtzYWxhcnkmcXVvdDspLCAuLi5dPC90ZXh0Pgo8L2c+CjwhLS0gc3FsJiM0NTsmZ3Q7bGV4ZXIg
LS0+CjxnIGlkPSJlZGdlMSIgY2xhc3M9ImVkZ2UiPgo8dGl0bGU+c3FsJiM0NTsmZ3Q7bGV4ZXI8
L3RpdGxlPgo8cGF0aCBmaWxsPSJub25lIiBzdHJva2U9ImJsYWNrIiBkPSJNMTgwLC0yODkuODlD
MTgwLC0yODYuMiAxODAsLTI4Mi4yMSAxODAsLTI3OC4yNCIvPgo8cG9seWdvbiBmaWxsPSJibGFj
ayIgc3Ryb2tlPSJibGFjayIgcG9pbnRzPSIxODMuNSwtMjc4IDE4MCwtMjY4IDE3Ni41LC0yNzgg
MTgzLjUsLTI3OCIvPgo8L2c+CjwhLS0gcGFyc2VyIC0tPgo8ZyBpZD0ibm9kZTMiIGNsYXNzPSJu
b2RlIj4KPHRpdGxlPnBhcnNlcjwvdGl0bGU+Cjxwb2x5Z29uIGZpbGw9Im5vbmUiIHN0cm9rZT0i
YmxhY2siIHBvaW50cz0iMzYwLC0yMTAgMCwtMjEwIDAsLTE3NCAzNjAsLTE3NCAzNjAsLTIxMCIv
Pgo8dGV4dCB0ZXh0LWFuY2hvcj0ibWlkZGxlIiB4PSIxODAiIHk9Ii0xOTUiIGZvbnQtZmFtaWx5
PSJDb3VyaWVyIE5ldywgRGVqYVZ1IFNhbnMgTW9ubywgbW9ub3NwYWNlIiBmb250LXNpemU9IjEw
LjAwIj5SZWN1cnNpdmUgRGVzY2VudCBTUUwgUGFyc2VyPC90ZXh0Pgo8dGV4dCB0ZXh0LWFuY2hv
cj0ibWlkZGxlIiB4PSIxODAiIHk9Ii0xODQiIGZvbnQtZmFtaWx5PSJDb3VyaWVyIE5ldywgRGVq
YVZ1IFNhbnMgTW9ubywgbW9ub3NwYWNlIiBmb250LXNpemU9IjEwLjAwIj5BU1QgTm9kZTogU2Vs
ZWN0U3RhdGVtZW50IEFTVCBOb2RlPC90ZXh0Pgo8L2c+CjwhLS0gbGV4ZXImIzQ1OyZndDtwYXJz
ZXIgLS0+CjxnIGlkPSJlZGdlMiIgY2xhc3M9ImVkZ2UiPgo8dGl0bGU+bGV4ZXImIzQ1OyZndDtw
YXJzZXI8L3RpdGxlPgo8cGF0aCBmaWxsPSJub25lIiBzdHJva2U9ImJsYWNrIiBkPSJNMTgwLC0y
MzEuODlDMTgwLC0yMjguMiAxODAsLTIyNC4yMSAxODAsLTIyMC4yNCIvPgo8cG9seWdvbiBmaWxs
PSJibGFjayIgc3Ryb2tlPSJibGFjayIgcG9pbnRzPSIxODMuNSwtMjIwIDE4MCwtMjEwIDE3Ni41
LC0yMjAgMTgzLjUsLTIyMCIvPgo8L2c+CjwhLS0gcGxhbm5lciAtLT4KPGcgaWQ9Im5vZGU0IiBj
bGFzcz0ibm9kZSI+Cjx0aXRsZT5wbGFubmVyPC90aXRsZT4KPHBvbHlnb24gZmlsbD0ibm9uZSIg
c3Ryb2tlPSJibGFjayIgcG9pbnRzPSIzNjAsLTE1MiAwLC0xNTIgMCwtMTE2IDM2MCwtMTE2IDM2
MCwtMTUyIi8+Cjx0ZXh0IHRleHQtYW5jaG9yPSJtaWRkbGUiIHg9IjE4MCIgeT0iLTE0Mi41IiBm
b250LWZhbWlseT0iQ291cmllciBOZXcsIERlamFWdSBTYW5zIE1vbm8sIG1vbm9zcGFjZSIgZm9u
dC1zaXplPSIxMC4wMCI+UXVlcnkgUGxhbm5lcjwvdGV4dD4KPHRleHQgdGV4dC1hbmNob3I9InN0
YXJ0IiB4PSI4IiB5PSItMTMxLjUiIGZvbnQtZmFtaWx5PSJDb3VyaWVyIE5ldywgRGVqYVZ1IFNh
bnMgTW9ubywgbW9ub3NwYWNlIiBmb250LXNpemU9IjEwLjAwIj5JbmRleCBMb29rdXAgQXZhaWxh
YmxlPyAmIzQ1OyYjNDU7Jmd0OyBCU1QgUmFuZ2UgUXVlcnk8L3RleHQ+Cjx0ZXh0IHRleHQtYW5j
aG9yPSJzdGFydCIgeD0iOCIgeT0iLTEyMC41IiBmb250LWZhbWlseT0iQ291cmllciBOZXcsIERl
amFWdSBTYW5zIE1vbm8sIG1vbm9zcGFjZSIgZm9udC1zaXplPSIxMC4wMCI+RWxzZSAmIzE2MDsm
IzE2MDsmIzE2MDsmIzE2MDsmIzE2MDsmIzE2MDsmIzE2MDsmIzE2MDsmIzE2MDsmIzE2MDsmIzE2
MDsmIzE2MDsmIzE2MDsmIzE2MDsmIzE2MDsmIzE2MDsmIzE2MDsmIzE2MDsmIzE2MDsmIzQ1OyYj
NDU7Jmd0OyBGdWxsIFRhYmxlIFNlcXVlbnRpYWwgU2NhbjwvdGV4dD4KPC9nPgo8IS0tIHBhcnNl
ciYjNDU7Jmd0O3BsYW5uZXIgLS0+CjxnIGlkPSJlZGdlMyIgY2xhc3M9ImVkZ2UiPgo8dGl0bGU+
cGFyc2VyJiM0NTsmZ3Q7cGxhbm5lcjwvdGl0bGU+CjxwYXRoIGZpbGw9Im5vbmUiIHN0cm9rZT0i
YmxhY2siIGQ9Ik0xODAsLTE3My44OUMxODAsLTE3MC4yIDE4MCwtMTY2LjIxIDE4MCwtMTYyLjI0
Ii8+Cjxwb2x5Z29uIGZpbGw9ImJsYWNrIiBzdHJva2U9ImJsYWNrIiBwb2ludHM9IjE4My41LC0x
NjIgMTgwLC0xNTIgMTc2LjUsLTE2MiAxODMuNSwtMTYyIi8+CjwvZz4KPCEtLSBub2RlcyAtLT4K
PGcgaWQ9Im5vZGU1IiBjbGFzcz0ibm9kZSI+Cjx0aXRsZT5ub2RlczwvdGl0bGU+Cjxwb2x5Z29u
IGZpbGw9Im5vbmUiIHN0cm9rZT0iYmxhY2siIHBvaW50cz0iMzYwLC05NCAwLC05NCAwLC01OCAz
NjAsLTU4IDM2MCwtOTQiLz4KPHRleHQgdGV4dC1hbmNob3I9Im1pZGRsZSIgeD0iMTgwIiB5PSIt
NzkiIGZvbnQtZmFtaWx5PSJDb3VyaWVyIE5ldywgRGVqYVZ1IFNhbnMgTW9ubywgbW9ub3NwYWNl
IiBmb250LXNpemU9IjEwLjAwIj5FeGVjdXRpb24gTm9kZXM8L3RleHQ+Cjx0ZXh0IHRleHQtYW5j
aG9yPSJtaWRkbGUiIHg9IjE4MCIgeT0iLTY4IiBmb250LWZhbWlseT0iQ291cmllciBOZXcsIERl
amFWdSBTYW5zIE1vbm8sIG1vbm9zcGFjZSIgZm9udC1zaXplPSIxMC4wMCI+U2VsZWN0aW9uTm9k
ZSAoz4MpICYjNDU7JiM0NTsmZ3Q7IFByb2plY3Rpb25Ob2RlICjPgCk8L3RleHQ+CjwvZz4KPCEt
LSBwbGFubmVyJiM0NTsmZ3Q7bm9kZXMgLS0+CjxnIGlkPSJlZGdlNCIgY2xhc3M9ImVkZ2UiPgo8
dGl0bGU+cGxhbm5lciYjNDU7Jmd0O25vZGVzPC90aXRsZT4KPHBhdGggZmlsbD0ibm9uZSIgc3Ry
b2tlPSJibGFjayIgZD0iTTE4MCwtMTE1Ljg5QzE4MCwtMTEyLjIgMTgwLC0xMDguMjEgMTgwLC0x
MDQuMjQiLz4KPHBvbHlnb24gZmlsbD0iYmxhY2siIHN0cm9rZT0iYmxhY2siIHBvaW50cz0iMTgz
LjUsLTEwNCAxODAsLTk0IDE3Ni41LC0xMDQgMTgzLjUsLTEwNCIvPgo8L2c+CjwhLS0gcmVzdWx0
IC0tPgo8ZyBpZD0ibm9kZTYiIGNsYXNzPSJub2RlIj4KPHRpdGxlPnJlc3VsdDwvdGl0bGU+Cjxw
b2x5Z29uIGZpbGw9Im5vbmUiIHN0cm9rZT0iYmxhY2siIHBvaW50cz0iMzYwLC0zNiAwLC0zNiAw
LDAgMzYwLDAgMzYwLC0zNiIvPgo8dGV4dCB0ZXh0LWFuY2hvcj0ibWlkZGxlIiB4PSIxODAiIHk9
Ii0xNS41IiBmb250LWZhbWlseT0iQ291cmllciBOZXcsIERlamFWdSBTYW5zIE1vbm8sIG1vbm9z
cGFjZSIgZm9udC1zaXplPSIxMC4wMCI+UmVzdWx0IFRhYmxlPC90ZXh0Pgo8L2c+CjwhLS0gbm9k
ZXMmIzQ1OyZndDtyZXN1bHQgLS0+CjxnIGlkPSJlZGdlNSIgY2xhc3M9ImVkZ2UiPgo8dGl0bGU+
bm9kZXMmIzQ1OyZndDtyZXN1bHQ8L3RpdGxlPgo8cGF0aCBmaWxsPSJub25lIiBzdHJva2U9ImJs
YWNrIiBkPSJNMTgwLC01Ny44OUMxODAsLTU0LjIgMTgwLC01MC4yMSAxODAsLTQ2LjI0Ii8+Cjxw
b2x5Z29uIGZpbGw9ImJsYWNrIiBzdHJva2U9ImJsYWNrIiBwb2ludHM9IjE4My41LC00NiAxODAs
LTM2IDE3Ni41LC00NiAxODMuNSwtNDYiLz4KPC9nPgo8L2c+Cjwvc3ZnPgo=
" alt="" />
<figcaption>Figure 16–5: Layers of SQL Statement Processing</figcaption>
</figure>

- Listing 16–1 defines the core data model. Rather than untyped raw
  memory blocks, we use `std::variant` to represent values cleanly and
  safely.
- Listing 16–2 implements the Binary Search Tree Index; an index
  accelerates lookups from an attribute value to matching `RowID`
  instances. The template-free balanced binary tree indexing structure
  handles `Value` keys directly.
- Listing 16–3 creates the `Table` class, which holds the row heap,
  executes constraint validation, maintains indices, and tracks an
  **Undo Log** for atomic rollback.
- Listing 16–4 defines a lexical analyser (lexer) to be used with a
  recursive-descent parser, supporting the following grammar:
  `plaintext   Statement     := CreateTable | InsertInto | SelectStmt   CreateTable   := "CREATE" "TABLE" Ident "(" ColumnDefList ")"   ColumnDefList := ColumnDef ( "," ColumnDef )*   ColumnDef     := Ident Type [ "PRIMARY" "KEY" ] [ "NOT" "NULL" ]   InsertInto    := "INSERT" "INTO" Ident "VALUES" "(" LiteralList ")"   SelectStmt    := "SELECT" ProjectionList "FROM" Ident [ "WHERE" Predicate ]   Predicate     := Ident BinaryOp Literal   BinaryOp      := "=" | "!=" | "<" | "<=" | ">" | ">="`
- Listing 16–5 defines the relational algebra nodes: `PhysicalScanNode`
  and `ProjectionNode` ($\pi$).
- Listing 16–6 constructs an end-to-end driver showing a test database
  in action:
  1.  Creating relational schemas with constraints.
  2.  Indexing attributes and verifying lookups.
  3.  Executing relational queries via SQL.
  4.  Testing transactional atomicity with rollback on constraint
      violations.

Listing 16–1: Core Implementation: Type Safety and Schemas (CoreDB.hpp):

In [ ]:
#ifndef COREDB_HPP
#define COREDB_HPP

#include <iostream>
#include <string>
#include <vector>
#include <variant>
#include <optional>
#include <memory>
#include <stdexcept>
#include <functional>
#include <sstream>
#include <map>
#include <algorithm>
#include <cstdint>
#include <string_view>
#include <iomanip>

namespace CoreDB {

// -----------------------------------------------------------------------------
// 1. Primitive Value System
// -----------------------------------------------------------------------------

enum class DataType {
    Integer,
    Float,
    Text,
    Null
};

// Represents a concrete relational value
class Value {
public:
    using VariantType = std::variant<std::monostate, int64_t, double, std::string>;

    Value() : m_data(std::monostate{}) {}
    Value(int64_t v) : m_data(v) {}
    Value(int v) : m_data(static_cast<int64_t>(v)) {}
    Value(double v) : m_data(v) {}
    Value(std::string v) : m_data(std::move(v)) {}
    Value(const char* v) : m_data(std::string(v)) {}

    [[nodiscard]] DataType type() const noexcept {
        return std::visit([](const auto& val) -> DataType {
            using T = std::decay_t<decltype(val)>;
            if constexpr (std::is_same_v<T, std::monostate>) return DataType::Null;
            else if constexpr (std::is_same_v<T, int64_t>)    return DataType::Integer;
            else if constexpr (std::is_same_v<T, double>)     return DataType::Float;
            else if constexpr (std::is_same_v<T, std::string>)return DataType::Text;
        }, m_data);
    }

    [[nodiscard]] bool is_null() const noexcept {
        return std::holds_alternative<std::monostate>(m_data);
    }

    [[nodiscard]] int64_t as_int() const {
        if (auto p = std::get_if<int64_t>(&m_data)) return *p;
        throw std::runtime_error("Type error: Value is not an Integer.");
    }

    [[nodiscard]] double as_float() const {
        if (auto p = std::get_if<double>(&m_data)) return *p;
        if (auto p = std::get_if<int64_t>(&m_data)) return static_cast<double>(*p);
        throw std::runtime_error("Type error: Value is not numeric.");
    }

    [[nodiscard]] const std::string& as_text() const {
        if (auto p = std::get_if<std::string>(&m_data)) return *p;
        throw std::runtime_error("Type error: Value is not Text.");
    }

    // Equality operator
    bool operator==(const Value& other) const {
        if (type() != other.type()) {
            // Support transparent float-to-int comparison
            if ((type() == DataType::Integer && other.type() == DataType::Float) ||
                (type() == DataType::Float && other.type() == DataType::Integer)) {
                return as_float() == other.as_float();
            }
            return false;
        }
        return m_data == other.m_data;
    }

    // Ordering operator for BST indexing and sorting
    bool operator<(const Value& other) const {
        if (type() != other.type()) {
            if ((type() == DataType::Integer && other.type() == DataType::Float) ||
                (type() == DataType::Float && other.type() == DataType::Integer)) {
                return as_float() < other.as_float();
            }
            return type() < other.type();
        }
        return m_data < other.m_data;
    }

    bool operator<=(const Value& other) const { return *this < other || *this == other; }
    bool operator>(const Value& other) const { return !(*this <= other); }
    bool operator>=(const Value& other) const { return !(*this < other); }
    bool operator!=(const Value& other) const { return !(*this == other); }

    friend std::ostream& operator<<(std::ostream& os, const Value& val) {
        std::visit([&os](const auto& v) {
            using T = std::decay_t<decltype(v)>;
            if constexpr (std::is_same_v<T, std::monostate>) {
                os << "NULL";
            } else if constexpr (std::is_same_v<T, std::string>) {
                os << "'" << v << "'";
            } else {
                os << v;
            }
        }, val.m_data);
        return os;
    }

private:
    VariantType m_data;
};

// -----------------------------------------------------------------------------
// 2. Schema Definition and Constraints
// -----------------------------------------------------------------------------

struct Column {
    std::string name;
    DataType type;
    bool is_primary_key{false};
    bool is_not_null{false};
};

class Schema {
public:
    Schema() = default;
    Schema(std::initializer_list<Column> cols) : m_columns(cols) {}
    explicit Schema(std::vector<Column> cols) : m_columns(std::move(cols)) {}

    [[nodiscard]] const std::vector<Column>& columns() const noexcept { return m_columns; }

    [[nodiscard]] std::optional<size_t> index_of(std::string_view column_name) const noexcept {
        for (size_t i = 0; i < m_columns.size(); ++i) {
            if (m_columns[i].name == column_name) return i;
        }
        return std::nullopt;
    }

    [[nodiscard]] const Column& get_column(size_t idx) const {
        if (idx >= m_columns.size()) throw std::out_of_range("Column index out of bounds.");
        return m_columns[idx];
    }

    [[nodiscard]] size_t arity() const noexcept { return m_columns.size(); }

private:
    std::vector<Column> m_columns;
};

using Row = std::vector<Value>;
using RowID = size_t;

} // namespace CoreDB

#endif // COREDB_HPP

Listing 16–2: Binary Search Tree Index (BSTIndex.hpp):

In [ ]:
#ifndef BST_INDEX_HPP
#define BST_INDEX_HPP

//#include "CoreDB.hpp"

namespace CoreDB {

class BSTIndex {
private:
    struct Node {
        Value key;
        std::vector<RowID> row_ids;
        std::unique_ptr<Node> left;
        std::unique_ptr<Node> right;
        int height{1};

        explicit Node(Value k, RowID id) : key(std::move(k)) {
            row_ids.push_back(id);
        }
    };

    std::unique_ptr<Node> m_root;

    int get_height(const Node* n) const noexcept {
        return n ? n->height : 0;
    }

    int balance_factor(const Node* n) const noexcept {
        return n ? get_height(n->left.get()) - get_height(n->right.get()) : 0;
    }

    void update_height(Node* n) noexcept {
        if (n) {
            n->height = 1 + std::max(get_height(n->left.get()), get_height(n->right.get()));
        }
    }

    std::unique_ptr<Node> rotate_right(std::unique_ptr<Node> y) {
        auto x = std::move(y->left);
        y->left = std::move(x->right);
        update_height(y.get());
        x->right = std::move(y);
        update_height(x.get());
        return x;
    }

    std::unique_ptr<Node> rotate_left(std::unique_ptr<Node> x) {
        auto y = std::move(x->right);
        x->right = std::move(y->left);
        update_height(x.get());
        y->left = std::move(x);
        update_height(y.get());
        return y;
    }

    std::unique_ptr<Node> insert_impl(std::unique_ptr<Node> node, const Value& key, RowID id) {
        if (!node) {
            return std::make_unique<Node>(key, id);
        }

        if (key == node->key) {
            node->row_ids.push_back(id);
            return node;
        }

        if (key < node->key) {
            node->left = insert_impl(std::move(node->left), key, id);
        } else {
            node->right = insert_impl(std::move(node->right), key, id);
        }

        update_height(node.get());
        int balance = balance_factor(node.get());

        // AVL Rebalancing
        if (balance > 1 && key < node->left->key) {
            return rotate_right(std::move(node));
        }
        if (balance < -1 && key > node->right->key) {
            return rotate_left(std::move(node));
        }
        if (balance > 1 && key > node->left->key) {
            node->left = rotate_left(std::move(node->left));
            return rotate_right(std::move(node));
        }
        if (balance < -1 && key < node->right->key) {
            node->right = rotate_right(std::move(node->right));
            return rotate_left(std::move(node));
        }

        return node;
    }

    void find_range_impl(const Node* node, const Value& low, const Value& high, 
                         std::vector<RowID>& results) const {
        if (!node) return;

        if (low < node->key) {
            find_range_impl(node->left.get(), low, high, results);
        }
        if (node->key >= low && node->key <= high) {
            results.insert(results.end(), node->row_ids.begin(), node->row_ids.end());
        }
        if (high > node->key) {
            find_range_impl(node->right.get(), low, high, results);
        }
    }

public:
    BSTIndex() = default;

    void insert(const Value& key, RowID id) {
        m_root = insert_impl(std::move(m_root), key, id);
    }

    [[nodiscard]] std::vector<RowID> find_exact(const Value& key) const {
        const Node* curr = m_root.get();
        while (curr) {
            if (key == curr->key) {
                return curr->row_ids;
            }
            if (key < curr->key) {
                curr = curr->left.get();
            } else {
                curr = curr->right.get();
            }
        }
        return {};
    }

    [[nodiscard]] std::vector<RowID> find_range(const Value& low, const Value& high) const {
        std::vector<RowID> results;
        find_range_impl(m_root.get(), low, high, results);
        return results;
    }
};

} // namespace CoreDB

#endif // BST_INDEX_HPP

Listing 16–3: Transactional Table Storage with Atomic Rollback
(Table.hpp):

In [ ]:
#ifndef TABLE_HPP
#define TABLE_HPP

//#include "CoreDB.hpp"
//#include "BSTIndex.hpp"

namespace CoreDB {

class Table {
public:
    Table(std::string name, Schema schema)
        : m_name(std::move(name)), m_schema(std::move(schema)) {
        // Automatically index the primary key column
        for (size_t i = 0; i < m_schema.arity(); ++i) {
            if (m_schema.get_column(i).is_primary_key) {
                m_pk_index_col = i;
                m_indices[i] = std::make_unique<BSTIndex>();
            }
        }
    }

    [[nodiscard]] const std::string& name() const noexcept { return m_name; }
    [[nodiscard]] const Schema& schema() const noexcept { return m_schema; }
    [[nodiscard]] size_t size() const noexcept { return m_active_row_count; }

    void create_index_on(const std::string& column_name) {
        auto col_idx = m_schema.index_of(column_name);
        if (!col_idx) {
            throw std::runtime_error("Column '" + column_name + "' does not exist.");
        }
        size_t idx = *col_idx;
        if (m_indices.contains(idx)) return; // Already indexed

        auto new_index = std::make_unique<BSTIndex>();
        for (size_t r = 0; r < m_rows.size(); ++r) {
            if (m_row_active[r]) {
                new_index->insert(m_rows[r][idx], r);
            }
        }
        m_indices[idx] = std::move(new_index);
    }

    [[nodiscard]] const BSTIndex* get_index(size_t col_idx) const {
        auto it = m_indices.find(col_idx);
        if (it != m_indices.end()) return it->second.get();
        return nullptr;
    }

    // -------------------------------------------------------------------------
    // Transactional Mutations
    // -------------------------------------------------------------------------

    void begin_transaction() {
        m_in_transaction = true;
        m_undo_log.clear();
    }

    void commit() {
        if (!m_in_transaction) return;
        m_undo_log.clear();
        m_in_transaction = false;
    }

    void rollback() {
        if (!m_in_transaction) return;
        
        // Revert mutations in reverse order
        for (auto it = m_undo_log.rbegin(); it != m_undo_log.rend(); ++it) {
            RowID id = it->row_id;
            if (it->type == UndoType::Insert) {
                m_row_active[id] = false;
                --m_active_row_count;
            }
        }
        
        // Rebuild indices to reflect restored state
        for (auto& [col_idx, index_ptr] : m_indices) {
            auto fresh_index = std::make_unique<BSTIndex>();
            for (size_t r = 0; r < m_rows.size(); ++r) {
                if (m_row_active[r]) {
                    fresh_index->insert(m_rows[r][col_idx], r);
                }
            }
            index_ptr = std::move(fresh_index);
        }

        m_undo_log.clear();
        m_in_transaction = false;
    }

    void insert_record(Row row) {
        validate_row(row);

        // Check primary key uniqueness
        if (m_pk_index_col.has_value()) {
            size_t pk_col = *m_pk_index_col;
            const Value& pk_val = row[pk_col];
            auto existing = m_indices[pk_col]->find_exact(pk_val);
            if (!existing.empty()) {
                throw std::runtime_error("Constraint Violation: Duplicate Primary Key value " + 
                                         [pk_val]() { 
                                             std::ostringstream ss; 
                                             ss << pk_val; 
                                             return ss.str(); 
                                         }());
            }
        }

        RowID new_id = m_rows.size();
        m_rows.push_back(std::move(row));
        m_row_active.push_back(true);
        ++m_active_row_count;

        // Update all associated indices
        for (auto& [col_idx, index_ptr] : m_indices) {
            index_ptr->insert(m_rows[new_id][col_idx], new_id);
        }

        if (m_in_transaction) {
            m_undo_log.push_back({UndoType::Insert, new_id});
        }
    }

    [[nodiscard]] const Row& get_row(RowID id) const {
        if (id >= m_rows.size() || !m_row_active[id]) {
            throw std::out_of_range("Invalid RowID.");
        }
        return m_rows[id];
    }

    [[nodiscard]] std::vector<RowID> all_row_ids() const {
        std::vector<RowID> ids;
        ids.reserve(m_active_row_count);
        for (size_t i = 0; i < m_rows.size(); ++i) {
            if (m_row_active[i]) ids.push_back(i);
        }
        return ids;
    }

private:
    enum class UndoType { Insert };
    struct UndoEntry {
        UndoType type;
        RowID row_id;
    };

    std::string m_name;
    Schema m_schema;
    std::vector<Row> m_rows;
    std::vector<bool> m_row_active;
    size_t m_active_row_count{0};

    std::optional<size_t> m_pk_index_col;
    std::map<size_t, std::unique_ptr<BSTIndex>> m_indices;

    bool m_in_transaction{false};
    std::vector<UndoEntry> m_undo_log;

    void validate_row(const Row& row) const {
        if (row.size() != m_schema.arity()) {
            throw std::runtime_error("Schema Mismatch: Expected " + 
                                     std::to_string(m_schema.arity()) + 
                                     " attributes, got " + std::to_string(row.size()));
        }
        for (size_t i = 0; i < m_schema.arity(); ++i) {
            const auto& col = m_schema.get_column(i);
            const auto& val = row[i];

            if (col.is_not_null && val.is_null()) {
                throw std::runtime_error("Constraint Violation: Column '" + 
                                         col.name + "' cannot be NULL.");
            }
            if (!val.is_null() && val.type() != col.type) {
                // Allow safe int-to-float promotion
                if (col.type == DataType::Float && val.type() == DataType::Integer) {
                    continue;
                }
                throw std::runtime_error("Type Mismatch on Column '" + col.name + "'.");
            }
        }
    }
};

} // namespace CoreDB

#endif // TABLE_HPP

Listing 16–4: Lexical Tokenisation (Lexer.hpp):

In [ ]:
#ifndef LEXER_HPP
#define LEXER_HPP

//#include "CoreDB.hpp"

namespace CoreDB {

enum class TokenType {
    Keyword_CREATE, Keyword_TABLE, Keyword_INSERT, Keyword_INTO,
    Keyword_VALUES, Keyword_SELECT, Keyword_FROM, Keyword_WHERE,
    Keyword_PRIMARY, Keyword_KEY, Keyword_NOT, Keyword_NULL,
    Type_INT, Type_FLOAT, Type_TEXT,
    Identifier,
    Literal_Integer, Literal_Float, Literal_String,
    Comma, LeftParen, RightParen, Asterisk,
    Op_Equal, Op_NotEqual, Op_Less, Op_LessEqual, Op_Greater, Op_GreaterEqual,
    SemiColon,
    EndOfFile
};

struct Token {
    TokenType type;
    std::string text;
    size_t line{1};
};

class Lexer {
public:
    explicit Lexer(std::string source) : m_src(std::move(source)), m_pos(0) {}

    std::vector<Token> tokenise() {
        std::vector<Token> tokens;
        while (!is_eof()) {
            char c = peek();
            if (std::isspace(c)) {
                advance();
                continue;
            }
            if (std::isalpha(c) || c == '_') {
                tokens.push_back(lex_identifier());
                continue;
            }
            if (std::isdigit(c) || (c == '-' && std::isdigit(peek_next()))) {
                tokens.push_back(lex_number());
                continue;
            }
            if (c == '\'') {
                tokens.push_back(lex_string());
                continue;
            }
            tokens.push_back(lex_symbol());
        }
        tokens.push_back({TokenType::EndOfFile, "", m_line});
        return tokens;
    }

private:
    std::string m_src;
    size_t m_pos{0};
    size_t m_line{1};

    [[nodiscard]] bool is_eof() const noexcept { return m_pos >= m_src.size(); }
    [[nodiscard]] char peek() const noexcept { return is_eof() ? '\0' : m_src[m_pos]; }
    [[nodiscard]] char peek_next() const noexcept { 
        return (m_pos + 1 >= m_src.size()) ? '\0' : m_src[m_pos + 1]; 
    }
    char advance() noexcept { return m_src[m_pos++]; }

    Token lex_identifier() {
        size_t start = m_pos;
        while (!is_eof() && (std::isalnum(peek()) || peek() == '_')) {
            advance();
        }
        std::string raw = m_src.substr(start, m_pos - start);
        std::string upper = raw;
        std::transform(upper.begin(), upper.end(), upper.begin(), ::toupper);

        if (upper == "CREATE")   return {TokenType::Keyword_CREATE, raw, m_line};
        if (upper == "TABLE")    return {TokenType::Keyword_TABLE, raw, m_line};
        if (upper == "INSERT")   return {TokenType::Keyword_INSERT, raw, m_line};
        if (upper == "INTO")     return {TokenType::Keyword_INTO, raw, m_line};
        if (upper == "VALUES")   return {TokenType::Keyword_VALUES, raw, m_line};
        if (upper == "SELECT")   return {TokenType::Keyword_SELECT, raw, m_line};
        if (upper == "FROM")     return {TokenType::Keyword_FROM, raw, m_line};
        if (upper == "WHERE")    return {TokenType::Keyword_WHERE, raw, m_line};
        if (upper == "PRIMARY")  return {TokenType::Keyword_PRIMARY, raw, m_line};
        if (upper == "KEY")      return {TokenType::Keyword_KEY, raw, m_line};
        if (upper == "NOT")      return {TokenType::Keyword_NOT, raw, m_line};
        if (upper == "NULL")     return {TokenType::Keyword_NULL, raw, m_line};
        if (upper == "INT")      return {TokenType::Type_INT, raw, m_line};
        if (upper == "FLOAT")    return {TokenType::Type_FLOAT, raw, m_line};
        if (upper == "TEXT")     return {TokenType::Type_TEXT, raw, m_line};

        return {TokenType::Identifier, raw, m_line};
    }

    Token lex_number() {
        size_t start = m_pos;
        if (peek() == '-') advance();
        bool is_float = false;
        while (!is_eof() && std::isdigit(peek())) advance();
        if (!is_eof() && peek() == '.') {
            is_float = true;
            advance();
            while (!is_eof() && std::isdigit(peek())) advance();
        }
        std::string val = m_src.substr(start, m_pos - start);
        return {is_float ? TokenType::Literal_Float : TokenType::Literal_Integer, val, m_line};
    }

    Token lex_string() {
        advance(); // Skip opening '
        size_t start = m_pos;
        while (!is_eof() && peek() != '\'') {
            advance();
        }
        if (is_eof()) throw std::runtime_error("Lexer error: Unterminated string literal.");
        std::string val = m_src.substr(start, m_pos - start);
        advance(); // Skip closing '
        return {TokenType::Literal_String, val, m_line};
    }

    Token lex_symbol() {
        char c = advance();
        switch (c) {
            case '(': return {TokenType::LeftParen, "(", m_line};
            case ')': return {TokenType::RightParen, ")", m_line};
            case ',': return {TokenType::Comma, ",", m_line};
            case '*': return {TokenType::Asterisk, "*", m_line};
            case '=': return {TokenType::Op_Equal, "=", m_line};
            case '<':
                if (peek() == '=') { advance(); return {TokenType::Op_LessEqual, "<=", m_line}; }
                if (peek() == '>') { advance(); return {TokenType::Op_NotEqual, "<>", m_line}; }
                return {TokenType::Op_Less, "<", m_line};
            case '>':
                if (peek() == '=') { advance(); return {TokenType::Op_GreaterEqual, ">=", m_line}; }
                return {TokenType::Op_Greater, ">", m_line};
            case '!':
                if (peek() == '=') { advance(); return {TokenType::Op_NotEqual, "!=", m_line}; }
                break;
            case ';': return {TokenType::SemiColon, ";", m_line};
            default: break;
        }
        throw std::runtime_error(std::string("Lexer error: Unknown symbol '") + c + "'");
    }
};

} // namespace CoreDB

#endif // LEXER_HPP

Listing 16–5: Query Engine: Relational Operations (Database.hpp):

In [ ]:
#ifndef DATABASE_HPP
#define DATABASE_HPP

//#include "CoreDB.hpp"
//#include "BSTIndex.hpp"
//#include "Table.hpp"
//#include "Lexer.hpp"

namespace CoreDB {

enum class ComparisonOp {
    Equal,
    NotEqual,
    LessThan,
    LessOrEqual,
    GreaterThan,
    GreaterOrEqual
};

struct Predicate {
    std::string column_name;
    ComparisonOp op;
    Value value;

    [[nodiscard]] bool evaluate(const Value& cell) const {
        switch (op) {
            case ComparisonOp::Equal:          return cell == value;
            case ComparisonOp::NotEqual:       return cell != value;
            case ComparisonOp::LessThan:       return cell < value;
            case ComparisonOp::LessOrEqual:    return cell <= value;
            case ComparisonOp::GreaterThan:    return cell > value;
            case ComparisonOp::GreaterOrEqual: return cell >= value;
        }
        return false;
    }
};

// -----------------------------------------------------------------------------
// Relational Algebra Execution Engine (Volcano Iterator Pattern)
// -----------------------------------------------------------------------------

class ExecutionNode {
public:
    virtual ~ExecutionNode() = default;
    virtual void open() = 0;
    virtual std::optional<Row> next() = 0;
    virtual const Schema& output_schema() const = 0;
};

// 1. Scan Node with Index Acceleration
class PhysicalScanNode : public ExecutionNode {
public:
    PhysicalScanNode(const Table& table, std::optional<Predicate> pred = std::nullopt)
        : m_table(table), m_predicate(std::move(pred)) {}

    void open() override {
        m_cursor = 0;
        m_matching_ids.clear();

        // Check if an index can accelerate this predicate
        if (m_predicate.has_value()) {
            auto col_idx = m_table.schema().index_of(m_predicate->column_name);
            if (col_idx.has_value()) {
                const BSTIndex* index = m_table.get_index(*col_idx);
                if (index && m_predicate->op == ComparisonOp::Equal) {
                    // Logarithmic O(log N) lookup
                    m_matching_ids = index->find_exact(m_predicate->value);
                    m_uses_index = true;
                    return;
                }
            }
        }

        // Fallback: Full Table Scan O(N)
        m_matching_ids = m_table.all_row_ids();
        m_uses_index = false;
    }

    std::optional<Row> next() override {
        while (m_cursor < m_matching_ids.size()) {
            RowID id = m_matching_ids[m_cursor++];
            const Row& candidate = m_table.get_row(id);

            // If we used an exact index, the predicate is guaranteed to match
            if (m_uses_index) {
                return candidate;
            }

            // Otherwise, evaluate the selection predicate manually
            if (!m_predicate.has_value()) {
                return candidate;
            }

            auto col_idx = *m_table.schema().index_of(m_predicate->column_name);
            if (m_predicate->evaluate(candidate[col_idx])) {
                return candidate;
            }
        }
        return std::nullopt;
    }

    const Schema& output_schema() const override {
        return m_table.schema();
    }

    [[nodiscard]] bool used_index() const noexcept { return m_uses_index; }

private:
    const Table& m_table;
    std::optional<Predicate> m_predicate;
    std::vector<RowID> m_matching_ids;
    size_t m_cursor{0};
    bool m_uses_index{false};
};

// 2. Projection Node (π)
class ProjectionNode : public ExecutionNode {
public:
    ProjectionNode(std::unique_ptr<ExecutionNode> child, std::vector<std::string> projected_cols)
        : m_child(std::move(child)), m_requested_names(std::move(projected_cols)) {
        
        const auto& in_schema = m_child->output_schema();
        std::vector<Column> out_cols;

        if (m_requested_names.size() == 1 && m_requested_names[0] == "*") {
            output_m_schema = in_schema;
            for (size_t i = 0; i < in_schema.arity(); ++i) {
                project_m_indices.push_back(i);
            }
        } else {
            for (const auto& name : m_requested_names) {
                auto idx = in_schema.index_of(name);
                if (!idx) throw std::runtime_error("Column '" + name + "' does not exist in child relation.");
                project_m_indices.push_back(*idx);
                out_cols.push_back(in_schema.get_column(*idx));
            }
            output_m_schema = Schema(out_cols);
        }
    }

    void open() override {
        m_child->open();
    }

    std::optional<Row> next() override {
        auto child_row = m_child->next();
        if (!child_row) return std::nullopt;

        Row projected;
        projected.reserve(project_m_indices.size());
        for (size_t idx : project_m_indices) {
            projected.push_back((*child_row)[idx]);
        }
        return projected;
    }

    const Schema& output_schema() const override {
        return output_m_schema;
    }

private:
    std::unique_ptr<ExecutionNode> m_child;
    std::vector<std::string> m_requested_names;
    std::vector<size_t> project_m_indices;
    Schema output_m_schema;
};

// -----------------------------------------------------------------------------
// Database Instance: Catalog and Engine Coordination
// -----------------------------------------------------------------------------

class Database {
public:
    void execute_ddl(std::string_view sql) {
        Lexer lexer{std::string(sql)};
        auto tokens = lexer.tokenise();
        size_t pos = 0;

        if (match(tokens, pos, TokenType::Keyword_CREATE)) {
            consume(tokens, pos, TokenType::Keyword_TABLE, "Expected 'TABLE'");
            Token m_nametok = consume(tokens, pos, TokenType::Identifier, "Expected table name");
            consume(tokens, pos, TokenType::LeftParen, "Expected '('");

            std::vector<Column> cols;
            while (true) {
                Token col_name = consume(tokens, pos, TokenType::Identifier, "Expected column name");
                Token col_type = advance_tok(tokens, pos);
                DataType dtype;
                if (col_type.type == TokenType::Type_INT) dtype = DataType::Integer;
                else if (col_type.type == TokenType::Type_FLOAT) dtype = DataType::Float;
                else if (col_type.type == TokenType::Type_TEXT) dtype = DataType::Text;
                else throw std::runtime_error("Invalid data type for column " + col_name.text);

                bool is_pk = false;
                bool is_not_null = false;

                while (pos < tokens.size() && 
                       (tokens[pos].type == TokenType::Keyword_PRIMARY || 
                        tokens[pos].type == TokenType::Keyword_NOT)) {
                    if (match(tokens, pos, TokenType::Keyword_PRIMARY)) {
                        consume(tokens, pos, TokenType::Keyword_KEY, "Expected 'KEY' after 'PRIMARY'");
                        is_pk = true;
                        is_not_null = true;
                    } else if (match(tokens, pos, TokenType::Keyword_NOT)) {
                        consume(tokens, pos, TokenType::Keyword_NULL, "Expected 'NULL' after 'NOT'");
                        is_not_null = true;
                    }
                }

                cols.push_back({col_name.text, dtype, is_pk, is_not_null});

                if (match(tokens, pos, TokenType::Comma)) continue;
                if (match(tokens, pos, TokenType::RightParen)) break;
                throw std::runtime_error("Expected ',' or ')' in column definitions.");
            }

            m_tables.emplace(m_nametok.text, std::make_unique<Table>(m_nametok.text, Schema(cols)));
        } else {
            throw std::runtime_error("Unsupported DDL command.");
        }
    }

    void execute_insert(std::string_view sql) {
        Lexer lexer{std::string(sql)};
        auto tokens = lexer.tokenise();
        size_t pos = 0;

        consume(tokens, pos, TokenType::Keyword_INSERT, "Expected 'INSERT'");
        consume(tokens, pos, TokenType::Keyword_INTO, "Expected 'INTO'");
        Token tbl_tok = consume(tokens, pos, TokenType::Identifier, "Expected table name");
        consume(tokens, pos, TokenType::Keyword_VALUES, "Expected 'VALUES'");
        consume(tokens, pos, TokenType::LeftParen, "Expected '('");

        auto it = m_tables.find(tbl_tok.text);
        if (it == m_tables.end()) {
            throw std::runtime_error("Table '" + tbl_tok.text + "' does not exist.");
        }

        Row row;
        while (true) {
            Token val_tok = advance_tok(tokens, pos);
            if (val_tok.type == TokenType::Literal_Integer) {
                row.emplace_back(static_cast<int64_t>(std::stoll(val_tok.text)));
            } else if (val_tok.type == TokenType::Literal_Float) {
                row.emplace_back(std::stod(val_tok.text));
            } else if (val_tok.type == TokenType::Literal_String) {
                row.emplace_back(val_tok.text);
            } else if (val_tok.type == TokenType::Keyword_NULL) {
                row.emplace_back(Value());
            } else {
                throw std::runtime_error("Unexpected token in VALUES clause: " + val_tok.text);
            }

            if (match(tokens, pos, TokenType::Comma)) continue;
            if (match(tokens, pos, TokenType::RightParen)) break;
            throw std::runtime_error("Expected ',' or ')' in VALUES clause.");
        }

        it->second->insert_record(std::move(row));
    }

    void print_query(std::string_view sql) {
        Lexer lexer{std::string(sql)};
        auto tokens = lexer.tokenise();
        size_t pos = 0;

        consume(tokens, pos, TokenType::Keyword_SELECT, "Expected 'SELECT'");

        std::vector<std::string> projections;
        if (match(tokens, pos, TokenType::Asterisk)) {
            projections.push_back("*");
        } else {
            while (true) {
                Token col = consume(tokens, pos, TokenType::Identifier, "Expected column name");
                projections.push_back(col.text);
                if (!match(tokens, pos, TokenType::Comma)) break;
            }
        }

        consume(tokens, pos, TokenType::Keyword_FROM, "Expected 'FROM'");
        Token tbl_tok = consume(tokens, pos, TokenType::Identifier, "Expected table name");

        auto it = m_tables.find(tbl_tok.text);
        if (it == m_tables.end()) {
            throw std::runtime_error("Table '" + tbl_tok.text + "' does not exist.");
        }
        Table& table = *(it->second);

        std::optional<Predicate> predicate;
        if (match(tokens, pos, TokenType::Keyword_WHERE)) {
            Token col = consume(tokens, pos, TokenType::Identifier, "Expected column name in WHERE");
            Token op_tok = advance_tok(tokens, pos);
            Token val_tok = advance_tok(tokens, pos);

            ComparisonOp op;
            if (op_tok.type == TokenType::Op_Equal) op = ComparisonOp::Equal;
            else if (op_tok.type == TokenType::Op_NotEqual) op = ComparisonOp::NotEqual;
            else if (op_tok.type == TokenType::Op_Less) op = ComparisonOp::LessThan;
            else if (op_tok.type == TokenType::Op_LessEqual) op = ComparisonOp::LessOrEqual;
            else if (op_tok.type == TokenType::Op_Greater) op = ComparisonOp::GreaterThan;
            else if (op_tok.type == TokenType::Op_GreaterEqual) op = ComparisonOp::GreaterOrEqual;
            else throw std::runtime_error("Unsupported comparison operator: " + op_tok.text);

            Value rhs;
            if (val_tok.type == TokenType::Literal_Integer) rhs = Value(static_cast<int64_t>(std::stoll(val_tok.text)));
            else if (val_tok.type == TokenType::Literal_Float) rhs = Value(std::stod(val_tok.text));
            else if (val_tok.type == TokenType::Literal_String) rhs = Value(val_tok.text);
            else throw std::runtime_error("Invalid literal in WHERE predicate.");

            predicate = Predicate{col.text, op, rhs};
        }

        // Construct Relational Algebra Pipeline
        auto scan_node = std::make_unique<PhysicalScanNode>(table, predicate);
        auto proj_node = std::make_unique<ProjectionNode>(std::move(scan_node), projections);

        // Execute Plan
        proj_node->open();
        const auto& out_schema = proj_node->output_schema();

        // Print Pretty Table Header
        std::cout << "\n+";
        for (const auto& col : out_schema.columns()) {
            std::cout << "----------------------+";
        }
        std::cout << "\n|";
        for (const auto& col : out_schema.columns()) {
            std::cout << " " << std::setw(20) << std::left << col.name << " |";
        }
        std::cout << "\n+";
        for (const auto& col : out_schema.columns()) {
            std::cout << "----------------------+";
        }
        std::cout << "\n";

        size_t row_count = 0;
        while (auto row = proj_node->next()) {
            ++row_count;
            std::cout << "|";
            for (const auto& val : *row) {
                std::ostringstream ss;
                ss << val;
                std::cout << " " << std::setw(20) << std::left << ss.str() << " |";
            }
            std::cout << "\n";
        }

        std::cout << "+";
        for (const auto& col : out_schema.columns()) {
            std::cout << "----------------------+";
        }
        std::cout << "\n(" << row_count << " rows returned)\n\n";
    }

    [[nodiscard]] Table& get_table(const std::string& name) {
        auto it = m_tables.find(name);
        if (it == m_tables.end()) throw std::runtime_error("Table not found: " + name);
        return *(it->second);
    }

private:
    std::map<std::string, std::unique_ptr<Table>> m_tables;

    bool match(const std::vector<Token>& tokens, size_t& pos, TokenType type) {
        if (pos < tokens.size() && tokens[pos].type == type) {
            ++pos;
            return true;
        }
        return false;
    }

    Token advance_tok(const std::vector<Token>& tokens, size_t& pos) {
        if (pos >= tokens.size()) throw std::runtime_error("Unexpected end of tokens.");
        return tokens[pos++];
    }

    Token consume(const std::vector<Token>& tokens, size_t& pos, TokenType expected, std::string_view err) {
        if (pos < tokens.size() && tokens[pos].type == expected) {
            return tokens[pos++];
        }
        throw std::runtime_error(std::string(err) + " at token '" + 
                                 (pos < tokens.size() ? tokens[pos].text : "EOF") + "'");
    }
};

} // namespace CoreDB

#endif // DATABASE_HPP

Listing 16–6: Verification and Demonstration (main.cpp):

In [ ]:
//#include "Database.hpp"
#include <iostream>

int main() {
    using namespace CoreDB;

    std::cout << "====================================================\n";
    std::cout << "      CoreDB Modern C++ Relational Engine Demo      \n";
    std::cout << "====================================================\n\n";

    try {
        Database db;

        // 1. Create Tables
        std::cout << "[Step 1] Creating schema for 'engineers'...\n";
        db.execute_ddl(
            "CREATE TABLE engineers ("
            "  id INT PRIMARY KEY,"
            "  name TEXT NOT NULL,"
            "  salary FLOAT NOT NULL,"
            "  dept TEXT"
            ");"
        );

        // 2. Insert Records
        std::cout << "[Step 2] Inserting records into 'engineers'...\n";
        db.execute_insert("INSERT INTO engineers VALUES (101, 'Ada Lovelace', 8500.50, 'Systems');");
        db.execute_insert("INSERT INTO engineers VALUES (102, 'Alan Turing', 9200.00, 'Cryptography');");
        db.execute_insert("INSERT INTO engineers VALUES (103, 'Grace Hopper', 8900.00, 'Compilers');");
        db.execute_insert("INSERT INTO engineers VALUES (104, 'Claude Shannon', 9100.00, 'Information Theory');");
        db.execute_insert("INSERT INTO engineers VALUES (105, 'John von Neumann', 9950.00, 'Architecture');");

        // 3. Query All Data (π * σ true)
        std::cout << "[Step 3] Query: SELECT * FROM engineers;";
        db.print_query("SELECT * FROM engineers;");

        // 4. Query with Projection and Selection Filter
        std::cout << "[Step 4] Query: SELECT name, salary FROM engineers WHERE salary > 9000.00;";
        db.print_query("SELECT name, salary FROM engineers WHERE salary > 9000.00;");

        // 5. Index-Accelerated Point Query on Primary Key
        std::cout << "[Step 5] Query: SELECT id, name, dept FROM engineers WHERE id = 103;";
        db.print_query("SELECT id, name, dept FROM engineers WHERE id = 103;");

        // 6. Demonstrate Transaction Atomicity and Constraint Rollback
        std::cout << "[Step 6] Testing Transactional ACID Guarantees...\n";
        Table& eng_table = db.get_table("engineers");

        std::cout << " -> Starting Transaction...\n";
        eng_table.begin_transaction();

        eng_table.insert_record({106, "Barbara Liskov", 9400.0, "Distributed"});
        std::cout << " -> Inserted 'Barbara Liskov' successfully.\n";

        try {
            std::cout << " -> Attempting to insert duplicate Primary Key (id: 101)...\n";
            eng_table.insert_record({101, "Imposter Record", 1000.0, "Security"});
        } catch (const std::exception& e) {
            std::cerr << " -> [Caught Expected Constraint Violation]: " << e.what() << "\n";
            std::cout << " -> Executing Atomic ROLLBACK...\n";
            eng_table.rollback();
        }

        std::cout << "\n[Step 7] Verifying table state post-rollback (ID 106 should NOT exist):";
        db.print_query("SELECT id, name FROM engineers;");

    } catch (const std::exception& e) {
        std::cerr << "CRITICAL ERROR: " << e.what() << "\n";
        return 1;
    }

    return 0;
}

In [ ]:
main(); // Invocation of main program for Jupyter notebook running the xeus-cpp kernel

### 16.13 Summary

By implementing a relational database engine from scratch, we have
demystified how structured persistence systems operate beneath the
surface:

- **Relational Algebra** gives databases a rigorous mathematical
  foundation. Its property of closure allows queries to be composed,
  nested, and restructured cleanly.
- **The Iterator Model (Volcano)** processes data one tuple at a time
  across composable operator nodes, keeping memory overhead minimal.
- **Indexing** replaces $O(N)$ sequential scans with logarithmic
  $O(\log N)$ tree searches, turning full-table sweeps into
  lightning-fast lookups.
- **ACID Guarantees** ensure that our persistent data remains resilient,
  consistent, and correct, even when operations fail midway through
  execution.

### 16.14 Programming Challenges

#### Exercise 16.1: The Nested-Loop Equi-Join ($\bowtie$) Operator

**Context and Theory:**

The relational natural join and equi-join combine tuples from two
relations $R$ and $S$ on equality between specified attributes:

$$R \bowtie_{R.A = S.B} S = \sigma_{R.A = S.B}(R \times S)$$

In a Volcano-style iterator model, join operators are demand-driven via
`open()`, `next()`, and `close()`. The fundamental physical algorithm is
the **Nested-Loop Join**. For every outer tuple $r \in R$, the engine
iterates through all inner tuples $s \in S$. The schema of the resulting
relation has an arity equal to:

$$\text{arity}(R \bowtie S) = \text{arity}(R) + \text{arity}(S) - (\text{if natural join then } 1 \text{ else } 0)$$

The naive nested-loop join has a time complexity of $O(|R| \times |S|)$
and an auxiliary memory complexity of $O(1)$ beyond child iterator
state. The outer child is evaluated once, whereas the inner child is
reopened and rescanned $|R|$ times.

<figure id="fig:nested-loop">
<img src="data:image/svg+xml;base64,PHN2ZyB4bWxucz0iaHR0cDovL3d3dy53My5vcmcvMjAwMC9zdmciIHdpZHRoPSI1MzguMiIgaGVp
Z2h0PSI2Ni40IiBjbGFzcz0ic3ZnYm9iIiB2aWV3Qm94PSIwIC00IDUzOC4yIDY2LjQiPgogIDxz
dHlsZT4uc3ZnYm9iIGxpbmUsIC5zdmdib2IgcGF0aCwgLnN2Z2JvYiBjaXJjbGUsIC5zdmdib2Ig
cmVjdCwgLnN2Z2JvYiBwb2x5Z29uIHsKICBzdHJva2U6IGJsYWNrOwogIHN0cm9rZS13aWR0aDog
MS41OwogIHN0cm9rZS1vcGFjaXR5OiAxOwogIGZpbGwtb3BhY2l0eTogMTsKICBzdHJva2UtbGlu
ZWNhcDogcm91bmQ7CiAgc3Ryb2tlLWxpbmVqb2luOiBtaXRlcjsKfQoKLnN2Z2JvYiB0ZXh0IHsK
ICB3aGl0ZS1zcGFjZTogcHJlOwogIGZpbGw6IGJsYWNrOwogIGZvbnQtZmFtaWx5OiBDb3VyaWVy
IE5ldywgRGVqYVZ1IFNhbnMgTW9ubywgbW9ub3NwYWNlOwogIGZvbnQtc2l6ZTogMTNweDsKfQoK
LnN2Z2JvYiByZWN0LmJhY2tkcm9wIHsKICBzdHJva2U6IG5vbmU7CiAgZmlsbDogdHJhbnNwYXJl
bnQ7Cn0KCi5zdmdib2IgLmJyb2tlbiB7CiAgc3Ryb2tlLWRhc2hhcnJheTogODsKfQoKLnN2Z2Jv
YiAuZmlsbGVkIHsKICBmaWxsOiBibGFjazsKfQoKLnN2Z2JvYiAuYmdfZmlsbGVkIHsKICBmaWxs
OiB0cmFuc3BhcmVudDsKICBzdHJva2Utd2lkdGg6IDE7Cn0KCi5zdmdib2IgLm5vZmlsbCB7CiAg
ZmlsbDogdHJhbnNwYXJlbnQ7Cn0KCi5zdmdib2IgLmVuZF9tYXJrZWRfYXJyb3cgewogIG1hcmtl
ci1lbmQ6IHVybCgjYXJyb3cpOwp9Cgouc3ZnYm9iIC5zdGFydF9tYXJrZWRfYXJyb3cgewogIG1h
cmtlci1zdGFydDogdXJsKCNhcnJvdyk7Cn0KCi5zdmdib2IgLmVuZF9tYXJrZWRfZGlhbW9uZCB7
CiAgbWFya2VyLWVuZDogdXJsKCNkaWFtb25kKTsKfQoKLnN2Z2JvYiAuc3RhcnRfbWFya2VkX2Rp
YW1vbmQgewogIG1hcmtlci1zdGFydDogdXJsKCNkaWFtb25kKTsKfQoKLnN2Z2JvYiAuZW5kX21h
cmtlZF9jaXJjbGUgewogIG1hcmtlci1lbmQ6IHVybCgjY2lyY2xlKTsKfQoKLnN2Z2JvYiAuc3Rh
cnRfbWFya2VkX2NpcmNsZSB7CiAgbWFya2VyLXN0YXJ0OiB1cmwoI2NpcmNsZSk7Cn0KCi5zdmdi
b2IgLmVuZF9tYXJrZWRfb3Blbl9jaXJjbGUgewogIG1hcmtlci1lbmQ6IHVybCgjb3Blbl9jaXJj
bGUpOwp9Cgouc3ZnYm9iIC5zdGFydF9tYXJrZWRfb3Blbl9jaXJjbGUgewogIG1hcmtlci1zdGFy
dDogdXJsKCNvcGVuX2NpcmNsZSk7Cn0KCi5zdmdib2IgLmVuZF9tYXJrZWRfYmlnX29wZW5fY2ly
Y2xlIHsKICBtYXJrZXItZW5kOiB1cmwoI2JpZ19vcGVuX2NpcmNsZSk7Cn0KCi5zdmdib2IgLnN0
YXJ0X21hcmtlZF9iaWdfb3Blbl9jaXJjbGUgewogIG1hcmtlci1zdGFydDogdXJsKCNiaWdfb3Bl
bl9jaXJjbGUpOwp9Cgo8L3N0eWxlPgogIDxkZWZzPgogICAgPG1hcmtlciBpZD0iYXJyb3ciIHZp
ZXdCb3g9Ii0yIC0yIDggOCIgcmVmWD0iNCIgcmVmWT0iMiIgbWFya2VyV2lkdGg9IjciIG1hcmtl
ckhlaWdodD0iNyIgb3JpZW50PSJhdXRvLXN0YXJ0LXJldmVyc2UiPgogICAgICA8cG9seWdvbiBw
b2ludHM9IjAsMCAwLDQgNCwyIDAsMCI+PC9wb2x5Z29uPgogICAgPC9tYXJrZXI+CiAgICA8bWFy
a2VyIGlkPSJkaWFtb25kIiB2aWV3Qm94PSItMiAtMiA4IDgiIHJlZlg9IjQiIHJlZlk9IjIiIG1h
cmtlcldpZHRoPSI3IiBtYXJrZXJIZWlnaHQ9IjciIG9yaWVudD0iYXV0by1zdGFydC1yZXZlcnNl
Ij4KICAgICAgPHBvbHlnb24gcG9pbnRzPSIwLDIgMiwwIDQsMiAyLDQgMCwyIj48L3BvbHlnb24+
CiAgICA8L21hcmtlcj4KICAgIDxtYXJrZXIgaWQ9ImNpcmNsZSIgdmlld0JveD0iMCAwIDggOCIg
cmVmWD0iNCIgcmVmWT0iNCIgbWFya2VyV2lkdGg9IjciIG1hcmtlckhlaWdodD0iNyIgb3JpZW50
PSJhdXRvLXN0YXJ0LXJldmVyc2UiPgogICAgICA8Y2lyY2xlIGN4PSI0IiBjeT0iNCIgcj0iMiIg
Y2xhc3M9ImZpbGxlZCI+PC9jaXJjbGU+CiAgICA8L21hcmtlcj4KICAgIDxtYXJrZXIgaWQ9Im9w
ZW5fY2lyY2xlIiB2aWV3Qm94PSIwIDAgOCA4IiByZWZYPSI0IiByZWZZPSI0IiBtYXJrZXJXaWR0
aD0iNyIgbWFya2VySGVpZ2h0PSI3IiBvcmllbnQ9ImF1dG8tc3RhcnQtcmV2ZXJzZSI+CiAgICAg
IDxjaXJjbGUgY3g9IjQiIGN5PSI0IiByPSIyIiBjbGFzcz0iYmdfZmlsbGVkIj48L2NpcmNsZT4K
ICAgIDwvbWFya2VyPgogICAgPG1hcmtlciBpZD0iYmlnX29wZW5fY2lyY2xlIiB2aWV3Qm94PSIw
IDAgOCA4IiByZWZYPSI0IiByZWZZPSI0IiBtYXJrZXJXaWR0aD0iNyIgbWFya2VySGVpZ2h0PSI3
IiBvcmllbnQ9ImF1dG8tc3RhcnQtcmV2ZXJzZSI+CiAgICAgIDxjaXJjbGUgY3g9IjQiIGN5PSI0
IiByPSIzIiBjbGFzcz0iYmdfZmlsbGVkIj48L2NpcmNsZT4KICAgIDwvbWFya2VyPgogIDwvZGVm
cz4KICA8cmVjdCBjbGFzcz0iYmFja2Ryb3AiIHg9IjAiIHk9Ii00IiB3aWR0aD0iNTM4LjIiIGhl
aWdodD0iNjYuNCI+PC9yZWN0PgogIDx0ZXh0IHhtbDpzcGFjZT0icHJlc2VydmUiIHg9IjEuOTUi
IHk9IjI3LjMwMDAwMSIgPls8L3RleHQ+CiAgPHRleHQgeG1sOnNwYWNlPSJwcmVzZXJ2ZSIgeD0i
MS45NSIgeT0iNDIuOSIgPls8L3RleHQ+CiAgPHRleHQgeG1sOnNwYWNlPSJwcmVzZXJ2ZSIgeD0i
MTcuNTUwMDAxIiB5PSIyNy4zMDAwMDEiID5Sb3c8L3RleHQ+CiAgPHRleHQgeG1sOnNwYWNlPSJw
cmVzZXJ2ZSIgeD0iMTcuNTUwMDAxIiB5PSI0Mi45IiA+Um93PC90ZXh0PgogIDx0ZXh0IHhtbDpz
cGFjZT0icHJlc2VydmUiIHg9IjQ4Ljc1IiB5PSIyNy4zMDAwMDEiID4wPC90ZXh0PgogIDx0ZXh0
IHhtbDpzcGFjZT0icHJlc2VydmUiIHg9IjQ4Ljc1IiB5PSI0Mi45IiA+MTwvdGV4dD4KICA8dGV4
dCB4bWw6c3BhY2U9InByZXNlcnZlIiB4PSI2NC4zNSIgeT0iMjcuMzAwMDAxIiA+XTwvdGV4dD4K
ICA8dGV4dCB4bWw6c3BhY2U9InByZXNlcnZlIiB4PSI2NC4zNSIgeT0iNDIuOSIgPl08L3RleHQ+
CiAgPGxpbmUgeDE9Ijc4IiB5MT0iMjMuNDAwMDAyIiB4Mj0iMTcxLjYiIHkyPSIyMy40MDAwMDIi
IGNsYXNzPSJzb2xpZCI+PC9saW5lPgogIDxwb2x5Z29uIHBvaW50cz0iMTcxLjYsMTkuNSAxNzku
NDAwMDEsMjMuNDAwMDAyIDE3MS42LDI3LjMwMDAwMSIgY2xhc3M9ImZpbGxlZCI+PC9wb2x5Z29u
PgogIDxsaW5lIHgxPSI3OCIgeTE9IjM5IiB4Mj0iMTcxLjYiIHkyPSIzOSIgY2xhc3M9InNvbGlk
Ij48L2xpbmU+CiAgPHBvbHlnb24gcG9pbnRzPSIxNzEuNiwzNS4xMDAwMDIgMTc5LjQwMDAxLDM5
IDE3MS42LDQyLjkiIGNsYXNzPSJmaWxsZWQiPjwvcG9seWdvbj4KICA8dGV4dCB4bWw6c3BhY2U9
InByZXNlcnZlIiB4PSIxODkuMTUwMDEiIHk9IjI3LjMwMDAwMSIgPls8L3RleHQ+CiAgPHRleHQg
eG1sOnNwYWNlPSJwcmVzZXJ2ZSIgeD0iMTg5LjE1MDAxIiB5PSI0Mi45IiA+WzwvdGV4dD4KICA8
dGV4dCB4bWw6c3BhY2U9InByZXNlcnZlIiB4PSIzMzcuMzUiIHk9IjI3LjMwMDAwMSIgPl08L3Rl
eHQ+CiAgPHRleHQgeG1sOnNwYWNlPSJwcmVzZXJ2ZSIgeD0iMzM3LjM1IiB5PSI0Mi45IiA+XTwv
dGV4dD4KICA8bGluZSB4MT0iMzUxIiB5MT0iMjMuNDAwMDAyIiB4Mj0iNDEzLjQwMDAyIiB5Mj0i
MjMuNDAwMDAyIiBjbGFzcz0ic29saWQiPjwvbGluZT4KICA8cG9seWdvbiBwb2ludHM9IjQxMy40
MDAwMiwxOS41IDQyMS4yLDIzLjQwMDAwMiA0MTMuNDAwMDIsMjcuMzAwMDAxIiBjbGFzcz0iZmls
bGVkIj48L3BvbHlnb24+CiAgPGxpbmUgeDE9IjM1MSIgeTE9IjM5IiB4Mj0iNDEzLjQwMDAyIiB5
Mj0iMzkiIGNsYXNzPSJzb2xpZCI+PC9saW5lPgogIDxwb2x5Z29uIHBvaW50cz0iNDEzLjQwMDAy
LDM1LjEwMDAwMiA0MjEuMiwzOSA0MTMuNDAwMDIsNDIuOSIgY2xhc3M9ImZpbGxlZCI+PC9wb2x5
Z29uPgogIDx0ZXh0IHhtbDpzcGFjZT0icHJlc2VydmUiIHg9IjIwNC43NSIgeT0iNDIuOSIgPlJl
c2V0PC90ZXh0PgogIDx0ZXh0IHhtbDpzcGFjZT0icHJlc2VydmUiIHg9IjI1MS41NSIgeT0iNDIu
OSIgPiZhbXA7PC90ZXh0PgogIDx0ZXh0IHhtbDpzcGFjZT0icHJlc2VydmUiIHg9IjI2Ny4xNSIg
eT0iNDIuOSIgPlJlc2NhbjwvdGV4dD4KICA8dGV4dCB4bWw6c3BhY2U9InByZXNlcnZlIiB4PSIz
MjEuNzUiIHk9IjQyLjkiID5TPC90ZXh0PgogIDx0ZXh0IHhtbDpzcGFjZT0icHJlc2VydmUiIHg9
IjEuOTUiIHk9IjExLjcwMDAwMSIgPk91dGVyIExvb3AgKFIpICAgICAgICAgSW5uZXIgTG9vcCAo
UykgICAgICAgICAgICAgICAgIE91dHB1dCBNYXRjaDwvdGV4dD4KICA8dGV4dCB4bWw6c3BhY2U9
InByZXNlcnZlIiB4PSIyMDQuNzUiIHk9IjI3LjMwMDAwMSIgPlNjYW4gU18wIC4uIFNfbTwvdGV4
dD4KICA8dGV4dCB4bWw6c3BhY2U9InByZXNlcnZlIiB4PSI0MzAuOTUiIHk9IjI3LjMwMDAwMSIg
PllpZWxkIChyIOKImCBzKTwvdGV4dD4KICA8dGV4dCB4bWw6c3BhY2U9InByZXNlcnZlIiB4PSI0
MzAuOTUiIHk9IjQyLjkiID5ZaWVsZCAociDiiJggcyk8L3RleHQ+Cjwvc3ZnPgo=
" alt="" />
<figcaption>Figure 16–6: Time Complexity of Nested Loop</figcaption>
</figure>

**Specification:**

Implement a `NestedLoopJoinNode` class inheriting from `ExecutionNode`.
The constructor must accept two child execution nodes, the outer join
column name, and the inner join column name. The output schema must
merge all outer columns with all inner columns, renaming or
deduplicating the join key as appropriate.

``` cpp
class NestedLoopJoinNode : public ExecutionNode {
public:
    NestedLoopJoinNode(std::unique_ptr<ExecutionNode> outer,
                       std::unique_ptr<ExecutionNode> inner,
                       std::string outer_col, std::string inner_col);
    void open() override;
    std::optional<Row> next() override;
    const Schema& output_schema() const override { return m_output_schema; }
private:
    std::unique_ptr<ExecutionNode> m_outer, m_inner;
    std::string m_outer_col_name, m_inner_col_name;
    size_t m_outer_col_idx{0}, m_inner_col_idx{0};
    std::optional<Row> m_current_outer_row;
    Schema m_output_schema;
};
```

**Verification Tasks:**

1.  **Disjoint Sets Match:** Populate relation $R$ with keys $\{1, 2\}$
    and relation $S$ with keys $\{3, 4\}$. Verify that `next()` returns
    `std::nullopt` on the first invocation ($0$ rows emitted).
2.  **One-to-Many Join:** Populate relation $R$ (Departments: `id`,
    `dept_name`) with `(1, "Systems")`, and $S$ (Engineers: `id`,
    `dept_id`, `name`) with `(101, 1, "Ada")` and `(102, 1, "Alan")`.
    Join on $R.\text{id} = S.\text{dept\_id}$. Verify that exactly two
    concatenated tuples are emitted: `[1, "Systems", 101, "Ada"]` and
    `[1, "Systems", 102, "Alan"]`.
3.  **Inner Rescan Verification:** Populate $R$ with $3$ rows and $S$
    with $4$ matching rows. Verify that `m_inner->open()` is invoked
    exactly $3$ times and that precisely $12$ joined tuples are emitted.

#### Exercise 16.2: Constant-Time Hash Indexing Engine

**Context and Theory:**

While balanced binary search trees (such as AVL or Red-Black trees)
guarantee $O(\log N)$ worst-case lookups and support efficient range
queries ($[k_{\text{low}}, k_{\text{high}}]$), point queries
(`WHERE id = 42`) can theoretically execute in amortised $O(1)$ time
using a **Hash Index**.

A hash index (see Figure 16–7) evaluates an attribute value through a
hash function $h(k): \text{Domain} \to \mathbb{N}$, mapping the key into
an array of collision buckets. Secondary indexing requires chaining,
because non-unique attributes map a single key to multiple matching
`RowID` instances. The primary operational disadvantage is the complete
loss of ordering: a hash index cannot accelerate range predicates (`<`,
`>`, `<=`, `>=`).

<figure id="fig:hash-lookup">
<img src="data:image/svg+xml;base64,PD94bWwgdmVyc2lvbj0iMS4wIiBlbmNvZGluZz0iVVRGLTgiIHN0YW5kYWxvbmU9Im5vIj8+Cjwh
RE9DVFlQRSBzdmcgUFVCTElDICItLy9XM0MvL0RURCBTVkcgMS4xLy9FTiIKICJodHRwOi8vd3d3
LnczLm9yZy9HcmFwaGljcy9TVkcvMS4xL0RURC9zdmcxMS5kdGQiPgo8IS0tIEdlbmVyYXRlZCBi
eSBncmFwaHZpeiB2ZXJzaW9uIDIuNDMuMCAoMCkKIC0tPgo8IS0tIFRpdGxlOiBHIFBhZ2VzOiAx
IC0tPgo8c3ZnIHdpZHRoPSI1NTlwdCIgaGVpZ2h0PSI0NHB0Igogdmlld0JveD0iMC4wMCAwLjAw
IDU1OS4wMCA0NC4wMCIgeG1sbnM9Imh0dHA6Ly93d3cudzMub3JnLzIwMDAvc3ZnIiB4bWxuczp4
bGluaz0iaHR0cDovL3d3dy53My5vcmcvMTk5OS94bGluayI+CjxnIGlkPSJncmFwaDAiIGNsYXNz
PSJncmFwaCIgdHJhbnNmb3JtPSJzY2FsZSgxIDEpIHJvdGF0ZSgwKSB0cmFuc2xhdGUoNCA0MCki
Pgo8dGl0bGU+RzwvdGl0bGU+CjwhLS0ga2V5IC0tPgo8ZyBpZD0ibm9kZTEiIGNsYXNzPSJub2Rl
Ij4KPHRpdGxlPmtleTwvdGl0bGU+Cjxwb2x5Z29uIGZpbGw9Im5vbmUiIHN0cm9rZT0iYmxhY2si
IHBvaW50cz0iMTAxLC0zNiAwLC0zNiAwLDAgMTAxLDAgMTAxLC0zNiIvPgo8dGV4dCB0ZXh0LWFu
Y2hvcj0ibWlkZGxlIiB4PSI1MC41IiB5PSItMTUuNSIgZm9udC1mYW1pbHk9IkNvdXJpZXIgTmV3
LCBEZWphVnUgU2FucyBNb25vLCBtb25vc3BhY2UiIGZvbnQtc2l6ZT0iMTAuMDAiPktleTogJnF1
b3Q7U3lzdGVtcyZxdW90OzwvdGV4dD4KPC9nPgo8IS0tIG1vZHVsbyAtLT4KPGcgaWQ9Im5vZGUy
IiBjbGFzcz0ibm9kZSI+Cjx0aXRsZT5tb2R1bG88L3RpdGxlPgo8cG9seWdvbiBmaWxsPSJub25l
IiBzdHJva2U9ImJsYWNrIiBwb2ludHM9IjI0MywtMzYgMTMwLC0zNiAxMzAsMCAyNDMsMCAyNDMs
LTM2Ii8+Cjx0ZXh0IHRleHQtYW5jaG9yPSJtaWRkbGUiIHg9IjE4Ni41IiB5PSItMTUuNSIgZm9u
dC1mYW1pbHk9IkNvdXJpZXIgTmV3LCBEZWphVnUgU2FucyBNb25vLCBtb25vc3BhY2UiIGZvbnQt
c2l6ZT0iMTAuMDAiPmgoS2V5KSAlIEJ1Y2tldHM8L3RleHQ+CjwvZz4KPCEtLSBrZXkmIzQ1OyZn
dDttb2R1bG8gLS0+CjxnIGlkPSJlZGdlMSIgY2xhc3M9ImVkZ2UiPgo8dGl0bGU+a2V5JiM0NTsm
Z3Q7bW9kdWxvPC90aXRsZT4KPHBhdGggZmlsbD0ibm9uZSIgc3Ryb2tlPSJibGFjayIgZD0iTTEw
MS4wMiwtMThDMTA3LjAzLC0xOCAxMTMuMjUsLTE4IDExOS40NiwtMTgiLz4KPHBvbHlnb24gZmls
bD0iYmxhY2siIHN0cm9rZT0iYmxhY2siIHBvaW50cz0iMTE5Ljg2LC0yMS41IDEyOS44NiwtMTgg
MTE5Ljg2LC0xNC41IDExOS44NiwtMjEuNSIvPgo8L2c+CjwhLS0gYnVja2V0IC0tPgo8ZyBpZD0i
bm9kZTMiIGNsYXNzPSJub2RlIj4KPHRpdGxlPmJ1Y2tldDwvdGl0bGU+Cjxwb2x5Z29uIGZpbGw9
Im5vbmUiIHN0cm9rZT0iYmxhY2siIHBvaW50cz0iMzQ5LC0zNiAyNzIsLTM2IDI3MiwwIDM0OSww
IDM0OSwtMzYiLz4KPHRleHQgdGV4dC1hbmNob3I9Im1pZGRsZSIgeD0iMzEwLjUiIHk9Ii0xNS41
IiBmb250LWZhbWlseT0iQ291cmllciBOZXcsIERlamFWdSBTYW5zIE1vbm8sIG1vbm9zcGFjZSIg
Zm9udC1zaXplPSIxMC4wMCI+QnVja2V0IFszXTwvdGV4dD4KPC9nPgo8IS0tIG1vZHVsbyYjNDU7
Jmd0O2J1Y2tldCAtLT4KPGcgaWQ9ImVkZ2UyIiBjbGFzcz0iZWRnZSI+Cjx0aXRsZT5tb2R1bG8m
IzQ1OyZndDtidWNrZXQ8L3RpdGxlPgo8cGF0aCBmaWxsPSJub25lIiBzdHJva2U9ImJsYWNrIiBk
PSJNMjQzLjE0LC0xOEMyNDkuMzQsLTE4IDI1NS42MiwtMTggMjYxLjcxLC0xOCIvPgo8cG9seWdv
biBmaWxsPSJibGFjayIgc3Ryb2tlPSJibGFjayIgcG9pbnRzPSIyNjEuNzQsLTIxLjUgMjcxLjc0
LC0xOCAyNjEuNzQsLTE0LjUgMjYxLjc0LC0yMS41Ii8+CjwvZz4KPCEtLSByZXN1bHRzIC0tPgo8
ZyBpZD0ibm9kZTQiIGNsYXNzPSJub2RlIj4KPHRpdGxlPnJlc3VsdHM8L3RpdGxlPgo8cG9seWdv
biBmaWxsPSJub25lIiBzdHJva2U9ImJsYWNrIiBwb2ludHM9IjU1MSwtMzYgMzc4LC0zNiAzNzgs
MCA1NTEsMCA1NTEsLTM2Ii8+Cjx0ZXh0IHRleHQtYW5jaG9yPSJtaWRkbGUiIHg9IjQ2NC41IiB5
PSItMTUuNSIgZm9udC1mYW1pbHk9IkNvdXJpZXIgTmV3LCBEZWphVnUgU2FucyBNb25vLCBtb25v
c3BhY2UiIGZvbnQtc2l6ZT0iMTAuMDAiPlsgUm93SUQ6IDEwMSwgUm93SUQ6IDEwNCBdPC90ZXh0
Pgo8L2c+CjwhLS0gYnVja2V0JiM0NTsmZ3Q7cmVzdWx0cyAtLT4KPGcgaWQ9ImVkZ2UzIiBjbGFz
cz0iZWRnZSI+Cjx0aXRsZT5idWNrZXQmIzQ1OyZndDtyZXN1bHRzPC90aXRsZT4KPHBhdGggZmls
bD0ibm9uZSIgc3Ryb2tlPSJibGFjayIgZD0iTTM0OS4wNywtMThDMzU0LjksLTE4IDM2MS4xNCwt
MTggMzY3LjU4LC0xOCIvPgo8cG9seWdvbiBmaWxsPSJibGFjayIgc3Ryb2tlPSJibGFjayIgcG9p
bnRzPSIzNjcuODQsLTIxLjUgMzc3Ljg0LC0xOCAzNjcuODQsLTE0LjUgMzY3Ljg0LC0yMS41Ii8+
CjwvZz4KPC9nPgo8L3N2Zz4K
" alt="" />
<figcaption>Figure 16–7: Process of Hash Index Lookup</figcaption>
</figure>

**Specification:**

Create a `HashIndex` class that implements point lookups using
`std::unordered_map` with a custom hash functor capable of hashing
`CoreDB::Value`. Integrate `HashIndex` as an alternative indexing
backend in `Table`.

``` cpp
struct ValueHasher {
    std::size_t operator()(const Value& v) const noexcept {
        return std::visit([](const auto& val) -> std::size_t {
            using T = std::decay_t<decltype(val)>;
            if constexpr (std::is_same_v<T, std::monostate>) return 0;
            else if constexpr (std::is_same_v<T, std::string>) 
                return std::hash<std::string>{}(val);
            else return std::hash<double>{}(static_cast<double>(val));
        }, v.raw_data());
    }
};

class HashIndex {
public:
    void insert(const Value& key, RowID id);
    [[nodiscard]] std::vector<RowID> find_exact(const Value& key) const;
    void remove(const Value& key, RowID id);
private:
    std::unordered_map<Value, std::vector<RowID>, ValueHasher> m_buckets;
};
```

**Verification Tasks:**

1.  **Multi-Value Point Lookup:** Insert $10,000$ rows where $500$ rows
    share the department key `"Compilers"`. Verify that
    `find_exact("Compilers")` returns a vector of exactly $500$ distinct
    `RowID` references.
2.  **Key Absence Invariant:** Query a non-existent key `"Distributed"`.
    Verify that an empty `std::vector<RowID>` is returned in
    sub-microsecond time without generating exceptions or dangling
    bucket entries.
3.  **Collision Chain Integrity:** Insert two distinct string values
    engineered or selected to produce identical hash codes modulo bucket
    count. Verify that both distinct keys return their exact,
    non-overlapping `RowID` sets.

#### Exercise 16.3: Relational Mutation with Index Synchronisation (`DELETE` and `UPDATE`)

**Context and Theory:**

In a production database, mutating stored records requires strict
synchronisation between the primary row storage (the heap) and all
auxiliary search indices.

When a row is removed via `DELETE`:

1.  The row slot in the table heap must be deactivated or tombstoned
    ($O(1)$).
2.  For *every* index declared on the table, the old value must be
    deleted from the tree or hash structure ($O(K \log N)$ where $K$ is
    the number of indices).

When a row is modified via `UPDATE`:

If an updated column is indexed, the old key must be excised from the
index and the new key inserted. If a constraint (such as `PRIMARY KEY`
uniqueness) is violated during an `UPDATE`, the engine must abort and
revert all partially applied mutations.

**Specification:**

Extend `Table` to support `delete_records` and `update_records`. Both
functions must accept a `Predicate` and operate transactionally,
updating all registered BST and Hash indices while logging changes to
the undo log.

``` cpp
class Table {
    // ... existing definitions ...
public:
    size_t delete_records(const Predicate& predicate);
    size_t update_records(const Predicate& predicate, 
                          const std::string& column_name, 
                          const Value& new_value);
private:
    void remove_from_indices(RowID id, const Row& row);
    void insert_into_indices(RowID id, const Row& row);
};
```

**Verification Tasks:**

1.  **Index Eviction on Delete:** Insert $1,000$ records. Delete rows
    where `dept = 'Systems'`. Verify that scanning the table returns no
    rows matching the predicate, and that `index->find_exact("Systems")`
    returns an empty array.
2.  **Index Key Mutation on Update:** Update row with ID $103$ to set
    `dept = 'Cryptography'`. Verify that
    `index->find_exact("Cryptography")` yields row ID $103$, while a
    lookup for its previous department yields nothing.
3.  **Rollback on Constraint Violation:** Attempt an update that sets
    the primary key of an existing row to collide with another active
    row. Verify that the operation throws a constraint violation and
    that neither the table nor the indices retain the mutated key.

#### Exercise 16.4: Volcano Aggregate Pipeline (`SUM`, `COUNT`, `AVG`, `MIN`, `MAX`)

**Context and Theory:**

Relational algebra includes the extended generalised projection operator
$\mathcal{G}_{f_1(A_1), \dots, f_k(A_k)}(R)$, which computes aggregate
statistics over relations.

In contrast to selection ($\sigma$) and projection ($\pi$), aggregate
operators are **pipeline breakers**. A selection operator streams tuples
on-the-fly ($O(1)$ transient memory); an aggregate operator cannot yield
its first output tuple until it has consumed its child iterator to
exhaustion (`EOF`), as shown in Figure 16–8. The algorithm maintains an
accumulator state $(c, s, m_{\min}, m_{\max})$:

$$\text{Count: } c_{i} = c_{i-1} + 1 \qquad \text{Sum: } s_{i} = s_{i-1} + v_i \qquad \text{Avg: } \mu_i = \frac{s_i}{c_i}$$

<figure id="fig:aggregate-operators">
<img src="data:image/svg+xml;base64,PHN2ZyB4bWxucz0iaHR0cDovL3d3dy53My5vcmcvMjAwMC9zdmciIHdpZHRoPSI2MzEuOCIgaGVp
Z2h0PSI2Ni40IiBjbGFzcz0ic3ZnYm9iIiB2aWV3Qm94PSIwIC00IDYzMS44IDY2LjQiPgogIDxz
dHlsZT4uc3ZnYm9iIGxpbmUsIC5zdmdib2IgcGF0aCwgLnN2Z2JvYiBjaXJjbGUsIC5zdmdib2Ig
cmVjdCwgLnN2Z2JvYiBwb2x5Z29uIHsKICBzdHJva2U6IGJsYWNrOwogIHN0cm9rZS13aWR0aDog
MS41OwogIHN0cm9rZS1vcGFjaXR5OiAxOwogIGZpbGwtb3BhY2l0eTogMTsKICBzdHJva2UtbGlu
ZWNhcDogcm91bmQ7CiAgc3Ryb2tlLWxpbmVqb2luOiBtaXRlcjsKfQoKLnN2Z2JvYiB0ZXh0IHsK
ICB3aGl0ZS1zcGFjZTogcHJlOwogIGZpbGw6IGJsYWNrOwogIGZvbnQtZmFtaWx5OiBDb3VyaWVy
IE5ldywgRGVqYVZ1IFNhbnMgTW9ubywgbW9ub3NwYWNlOwogIGZvbnQtc2l6ZTogMTNweDsKfQoK
LnN2Z2JvYiByZWN0LmJhY2tkcm9wIHsKICBzdHJva2U6IG5vbmU7CiAgZmlsbDogdHJhbnNwYXJl
bnQ7Cn0KCi5zdmdib2IgLmJyb2tlbiB7CiAgc3Ryb2tlLWRhc2hhcnJheTogODsKfQoKLnN2Z2Jv
YiAuZmlsbGVkIHsKICBmaWxsOiBibGFjazsKfQoKLnN2Z2JvYiAuYmdfZmlsbGVkIHsKICBmaWxs
OiB0cmFuc3BhcmVudDsKICBzdHJva2Utd2lkdGg6IDE7Cn0KCi5zdmdib2IgLm5vZmlsbCB7CiAg
ZmlsbDogdHJhbnNwYXJlbnQ7Cn0KCi5zdmdib2IgLmVuZF9tYXJrZWRfYXJyb3cgewogIG1hcmtl
ci1lbmQ6IHVybCgjYXJyb3cpOwp9Cgouc3ZnYm9iIC5zdGFydF9tYXJrZWRfYXJyb3cgewogIG1h
cmtlci1zdGFydDogdXJsKCNhcnJvdyk7Cn0KCi5zdmdib2IgLmVuZF9tYXJrZWRfZGlhbW9uZCB7
CiAgbWFya2VyLWVuZDogdXJsKCNkaWFtb25kKTsKfQoKLnN2Z2JvYiAuc3RhcnRfbWFya2VkX2Rp
YW1vbmQgewogIG1hcmtlci1zdGFydDogdXJsKCNkaWFtb25kKTsKfQoKLnN2Z2JvYiAuZW5kX21h
cmtlZF9jaXJjbGUgewogIG1hcmtlci1lbmQ6IHVybCgjY2lyY2xlKTsKfQoKLnN2Z2JvYiAuc3Rh
cnRfbWFya2VkX2NpcmNsZSB7CiAgbWFya2VyLXN0YXJ0OiB1cmwoI2NpcmNsZSk7Cn0KCi5zdmdi
b2IgLmVuZF9tYXJrZWRfb3Blbl9jaXJjbGUgewogIG1hcmtlci1lbmQ6IHVybCgjb3Blbl9jaXJj
bGUpOwp9Cgouc3ZnYm9iIC5zdGFydF9tYXJrZWRfb3Blbl9jaXJjbGUgewogIG1hcmtlci1zdGFy
dDogdXJsKCNvcGVuX2NpcmNsZSk7Cn0KCi5zdmdib2IgLmVuZF9tYXJrZWRfYmlnX29wZW5fY2ly
Y2xlIHsKICBtYXJrZXItZW5kOiB1cmwoI2JpZ19vcGVuX2NpcmNsZSk7Cn0KCi5zdmdib2IgLnN0
YXJ0X21hcmtlZF9iaWdfb3Blbl9jaXJjbGUgewogIG1hcmtlci1zdGFydDogdXJsKCNiaWdfb3Bl
bl9jaXJjbGUpOwp9Cgo8L3N0eWxlPgogIDxkZWZzPgogICAgPG1hcmtlciBpZD0iYXJyb3ciIHZp
ZXdCb3g9Ii0yIC0yIDggOCIgcmVmWD0iNCIgcmVmWT0iMiIgbWFya2VyV2lkdGg9IjciIG1hcmtl
ckhlaWdodD0iNyIgb3JpZW50PSJhdXRvLXN0YXJ0LXJldmVyc2UiPgogICAgICA8cG9seWdvbiBw
b2ludHM9IjAsMCAwLDQgNCwyIDAsMCI+PC9wb2x5Z29uPgogICAgPC9tYXJrZXI+CiAgICA8bWFy
a2VyIGlkPSJkaWFtb25kIiB2aWV3Qm94PSItMiAtMiA4IDgiIHJlZlg9IjQiIHJlZlk9IjIiIG1h
cmtlcldpZHRoPSI3IiBtYXJrZXJIZWlnaHQ9IjciIG9yaWVudD0iYXV0by1zdGFydC1yZXZlcnNl
Ij4KICAgICAgPHBvbHlnb24gcG9pbnRzPSIwLDIgMiwwIDQsMiAyLDQgMCwyIj48L3BvbHlnb24+
CiAgICA8L21hcmtlcj4KICAgIDxtYXJrZXIgaWQ9ImNpcmNsZSIgdmlld0JveD0iMCAwIDggOCIg
cmVmWD0iNCIgcmVmWT0iNCIgbWFya2VyV2lkdGg9IjciIG1hcmtlckhlaWdodD0iNyIgb3JpZW50
PSJhdXRvLXN0YXJ0LXJldmVyc2UiPgogICAgICA8Y2lyY2xlIGN4PSI0IiBjeT0iNCIgcj0iMiIg
Y2xhc3M9ImZpbGxlZCI+PC9jaXJjbGU+CiAgICA8L21hcmtlcj4KICAgIDxtYXJrZXIgaWQ9Im9w
ZW5fY2lyY2xlIiB2aWV3Qm94PSIwIDAgOCA4IiByZWZYPSI0IiByZWZZPSI0IiBtYXJrZXJXaWR0
aD0iNyIgbWFya2VySGVpZ2h0PSI3IiBvcmllbnQ9ImF1dG8tc3RhcnQtcmV2ZXJzZSI+CiAgICAg
IDxjaXJjbGUgY3g9IjQiIGN5PSI0IiByPSIyIiBjbGFzcz0iYmdfZmlsbGVkIj48L2NpcmNsZT4K
ICAgIDwvbWFya2VyPgogICAgPG1hcmtlciBpZD0iYmlnX29wZW5fY2lyY2xlIiB2aWV3Qm94PSIw
IDAgOCA4IiByZWZYPSI0IiByZWZZPSI0IiBtYXJrZXJXaWR0aD0iNyIgbWFya2VySGVpZ2h0PSI3
IiBvcmllbnQ9ImF1dG8tc3RhcnQtcmV2ZXJzZSI+CiAgICAgIDxjaXJjbGUgY3g9IjQiIGN5PSI0
IiByPSIzIiBjbGFzcz0iYmdfZmlsbGVkIj48L2NpcmNsZT4KICAgIDwvbWFya2VyPgogIDwvZGVm
cz4KICA8cmVjdCBjbGFzcz0iYmFja2Ryb3AiIHg9IjAiIHk9Ii00IiB3aWR0aD0iNjMxLjgiIGhl
aWdodD0iNjYuNCI+PC9yZWN0PgogIDx0ZXh0IHhtbDpzcGFjZT0icHJlc2VydmUiIHg9IjkuNzUi
IHk9IjExLjcwMDAwMSIgPkNoaWxkPC90ZXh0PgogIDx0ZXh0IHhtbDpzcGFjZT0icHJlc2VydmUi
IHg9IjU2LjU1MDAwMyIgeT0iMTEuNzAwMDAxIiA+U3RyZWFtOjwvdGV4dD4KICA8dGV4dCB4bWw6
c3BhY2U9InByZXNlcnZlIiB4PSIxMTguOTUwMDA1IiB5PSIxMS43MDAwMDEiID5bPC90ZXh0Pgog
IDx0ZXh0IHhtbDpzcGFjZT0icHJlc2VydmUiIHg9IjEzNC41NSIgeT0iMTEuNzAwMDAxIiA+Um93
PC90ZXh0PgogIDx0ZXh0IHhtbDpzcGFjZT0icHJlc2VydmUiIHg9IjE2NS43NSIgeT0iMTEuNzAw
MDAxIiA+XTwvdGV4dD4KICA8bGluZSB4MT0iMTc5LjQwMDAxIiB5MT0iNy44IiB4Mj0iMTk1IiB5
Mj0iNy44IiBjbGFzcz0ic29saWQiPjwvbGluZT4KICA8cG9seWdvbiBwb2ludHM9IjE5NSwzLjkg
MjAyLjgsNy44IDE5NSwxMS43MDAwMDEiIGNsYXNzPSJmaWxsZWQiPjwvcG9seWdvbj4KICA8dGV4
dCB4bWw6c3BhY2U9InByZXNlcnZlIiB4PSIyMTIuNTUiIHk9IjExLjcwMDAwMSIgPls8L3RleHQ+
CiAgPHRleHQgeG1sOnNwYWNlPSJwcmVzZXJ2ZSIgeD0iMjI4LjE1MDAxIiB5PSIxMS43MDAwMDEi
ID5Sb3c8L3RleHQ+CiAgPHRleHQgeG1sOnNwYWNlPSJwcmVzZXJ2ZSIgeD0iMjU5LjM1IiB5PSIx
MS43MDAwMDEiID5dPC90ZXh0PgogIDxsaW5lIHgxPSIyNzMiIHkxPSI3LjgiIHgyPSIyODguNiIg
eTI9IjcuOCIgY2xhc3M9InNvbGlkIj48L2xpbmU+CiAgPHBvbHlnb24gcG9pbnRzPSIyODguNiwz
LjkgMjk2LjQsNy44IDI4OC42LDExLjcwMDAwMSIgY2xhc3M9ImZpbGxlZCI+PC9wb2x5Z29uPgog
IDx0ZXh0IHhtbDpzcGFjZT0icHJlc2VydmUiIHg9IjMwNi4xNSIgeT0iMTEuNzAwMDAxIiA+Wzwv
dGV4dD4KICA8dGV4dCB4bWw6c3BhY2U9InByZXNlcnZlIiB4PSIzMjEuNzUiIHk9IjExLjcwMDAw
MSIgPlJvdzwvdGV4dD4KICA8dGV4dCB4bWw6c3BhY2U9InByZXNlcnZlIiB4PSIzNTIuOTUiIHk9
IjExLjcwMDAwMSIgPl08L3RleHQ+CiAgPGxpbmUgeDE9IjM2Ni42IiB5MT0iNy44IiB4Mj0iMzgy
LjIiIHkyPSI3LjgiIGNsYXNzPSJzb2xpZCI+PC9saW5lPgogIDxwb2x5Z29uIHBvaW50cz0iMzgy
LjIsMy45IDM5MCw3LjggMzgyLjIsMTEuNzAwMDAxIiBjbGFzcz0iZmlsbGVkIj48L3BvbHlnb24+
CiAgPHBvbHlnb24gcG9pbnRzPSI0NTIuNDAwMDIsMzUuMTAwMDAyIDQ2MC4yLDM5IDQ1Mi40MDAw
Miw0Mi45IiBjbGFzcz0iZmlsbGVkIj48L3BvbHlnb24+CiAgPHRleHQgeG1sOnNwYWNlPSJwcmVz
ZXJ2ZSIgeD0iNDY5Ljk1IiB5PSI0Mi45IiA+WWllbGQ8L3RleHQ+CiAgPHRleHQgeG1sOnNwYWNl
PSJwcmVzZXJ2ZSIgeD0iNTE2Ljc1IiB5PSI0Mi45IiA+QWdncmVnYXRlZDwvdGV4dD4KICA8dGV4
dCB4bWw6c3BhY2U9InByZXNlcnZlIiB4PSI2MDIuNTUiIHk9IjQyLjkiID5Sb3c8L3RleHQ+CiAg
PHRleHQgeG1sOnNwYWNlPSJwcmVzZXJ2ZSIgeD0iMzk5Ljc1IiB5PSIxMS43MDAwMDEiID5FT0Y8
L3RleHQ+CiAgPHRleHQgeG1sOnNwYWNlPSJwcmVzZXJ2ZSIgeD0iOS43NSIgeT0iNDIuOSIgPkFj
Y3VtdWxhdG9yOiAoY291bnQ9Mywgc3VtPTI0MC4wLCBhdmc9ODAuMCk8L3RleHQ+CiAgPGc+CiAg
ICA8bGluZSB4MT0iNDA5LjUiIHkxPSIxNS42IiB4Mj0iNDA5LjUiIHkyPSIzOSIgY2xhc3M9InNv
bGlkIj48L2xpbmU+CiAgICA8bGluZSB4MT0iMzU4LjgwMDAyIiB5MT0iMzkiIHgyPSI0NTIuNDAw
MDIiIHkyPSIzOSIgY2xhc3M9InNvbGlkIj48L2xpbmU+CiAgPC9nPgo8L3N2Zz4K
" alt="" />
<figcaption>Figure 16–8: Operation of Aggregate Pipeline</figcaption>
</figure>

**Specification:**

Implement an `AggregateNode` conforming to the `ExecutionNode`
interface. The constructor must take a child execution node, an
aggregation function enumeration, and the target column name. When
evaluated, it must emit exactly one row containing the computed
aggregate scalar.

``` cpp
enum class AggregateType { Count, Sum, Avg, Min, Max };

class AggregateNode : public ExecutionNode {
public:
    AggregateNode(std::unique_ptr<ExecutionNode> child, 
                  AggregateType agg_type, 
                  std::string target_col);
    void open() override;
    std::optional<Row> next() override;
    const Schema& output_schema() const override { return m_schema; }
private:
    std::unique_ptr<ExecutionNode> m_child;
    AggregateType m_agg_type;
    std::string m_target_col;
    Schema m_schema;
    bool m_has_emitted{false};
};
```

**Verification Tasks:**

1.  **Zero-Row Aggregation Invariant:** Pass an empty child iterator
    (filtered by an unsatisfiable predicate) to an `AggregateNode`
    configured for `COUNT`. Verify that it emits a single row containing
    `Value(int64_t(0))`.
2.  **Floating-Point Precision:** Feed $10,000$ values drawn uniformly
    from $[0.0, 100.0]$ with an analytical mean $\mu = 50.0$. Verify
    that `SUM`, `AVG`, `MIN`, and `MAX` match calculations within an
    absolute floating-point tolerance $\epsilon = 10^{-5}$.
3.  **Composition:** Chain `PhysicalScanNode` $\to$ `ProjectionNode`
    $\to$ `AggregateNode`. Verify that
    `SELECT AVG(salary) FROM engineers WHERE dept = 'Compilers'` returns
    the expected scalar.

#### Exercise 16.5: Recursive Boolean Expression Trees (DNF and CNF)

**Context and Theory:**

Basic database engines limit selection to a single column predicate
($A \text{ op } c$). Production engines evaluate compound propositional
formulas composed of conjunctions ($\land$), disjunctions ($\lor$), and
negations ($\neg$).

Under Codd's relational model, queries operate under **Three-Valued
Logic (3VL)** when `NULL` values are present, evaluating propositions to
`TRUE`, `FALSE`, or `UNKNOWN`. For selection filters, tuples are
retained if and only if the expression evaluates to `TRUE`. A diagram of
a Boolean Expression Tree suitable to be used with 3VL is shown in
Figure 16–9.

<figure id="fig:three-value-logic">
<img src="data:image/svg+xml;base64,PD94bWwgdmVyc2lvbj0iMS4wIiBlbmNvZGluZz0iVVRGLTgiIHN0YW5kYWxvbmU9Im5vIj8+Cjwh
RE9DVFlQRSBzdmcgUFVCTElDICItLy9XM0MvL0RURCBTVkcgMS4xLy9FTiIKICJodHRwOi8vd3d3
LnczLm9yZy9HcmFwaGljcy9TVkcvMS4xL0RURC9zdmcxMS5kdGQiPgo8IS0tIEdlbmVyYXRlZCBi
eSBncmFwaHZpeiB2ZXJzaW9uIDIuNDMuMCAoMCkKIC0tPgo8IS0tIFRpdGxlOiBHIFBhZ2VzOiAx
IC0tPgo8c3ZnIHdpZHRoPSI0NjhwdCIgaGVpZ2h0PSIyMzlwdCIKIHZpZXdCb3g9IjAuMDAgMC4w
MCA0NjguMTIgMjM5LjAwIiB4bWxucz0iaHR0cDovL3d3dy53My5vcmcvMjAwMC9zdmciIHhtbG5z
OnhsaW5rPSJodHRwOi8vd3d3LnczLm9yZy8xOTk5L3hsaW5rIj4KPGcgaWQ9ImdyYXBoMCIgY2xh
c3M9ImdyYXBoIiB0cmFuc2Zvcm09InNjYWxlKDEgMSkgcm90YXRlKDApIHRyYW5zbGF0ZSg0IDIz
NSkiPgo8dGl0bGU+RzwvdGl0bGU+CjwhLS0gb3IgLS0+CjxnIGlkPSJub2RlMSIgY2xhc3M9Im5v
ZGUiPgo8dGl0bGU+b3I8L3RpdGxlPgo8ZWxsaXBzZSBmaWxsPSJub25lIiBzdHJva2U9ImJsYWNr
IiBjeD0iMjY3Ljg2IiBjeT0iLTIxMyIgcng9IjM0Ljk2IiByeT0iMTgiLz4KPHRleHQgdGV4dC1h
bmNob3I9Im1pZGRsZSIgeD0iMjY3Ljg2IiB5PSItMjEwLjUiIGZvbnQtZmFtaWx5PSJDb3VyaWVy
IE5ldywgRGVqYVZ1IFNhbnMgTW9ubywgbW9ub3NwYWNlIiBmb250LXNpemU9IjEwLjAwIj5PUiBO
b2RlPC90ZXh0Pgo8L2c+CjwhLS0gYW5kIC0tPgo8ZyBpZD0ibm9kZTIiIGNsYXNzPSJub2RlIj4K
PHRpdGxlPmFuZDwvdGl0bGU+CjxlbGxpcHNlIGZpbGw9Im5vbmUiIHN0cm9rZT0iYmxhY2siIGN4
PSIyMTYuODYiIGN5PSItMTQ4IiByeD0iMzguMDMiIHJ5PSIxOCIvPgo8dGV4dCB0ZXh0LWFuY2hv
cj0ibWlkZGxlIiB4PSIyMTYuODYiIHk9Ii0xNDUuNSIgZm9udC1mYW1pbHk9IkNvdXJpZXIgTmV3
LCBEZWphVnUgU2FucyBNb25vLCBtb25vc3BhY2UiIGZvbnQtc2l6ZT0iMTAuMDAiPkFORCBOb2Rl
PC90ZXh0Pgo8L2c+CjwhLS0gb3ImIzQ1OyZndDthbmQgLS0+CjxnIGlkPSJlZGdlMSIgY2xhc3M9
ImVkZ2UiPgo8dGl0bGU+b3ImIzQ1OyZndDthbmQ8L3RpdGxlPgo8cGF0aCBmaWxsPSJub25lIiBz
dHJva2U9ImJsYWNrIiBkPSJNMjU0Ljk5LC0xOTYuMUMyNDkuMzYsLTE4OS4xNCAyNDIuNjIsLTE4
MC44MiAyMzYuMzksLTE3My4xMyIvPgo8cG9seWdvbiBmaWxsPSJibGFjayIgc3Ryb2tlPSJibGFj
ayIgcG9pbnRzPSIyMzguOTEsLTE3MC42OCAyMjkuOSwtMTY1LjExIDIzMy40NywtMTc1LjA5IDIz
OC45MSwtMTcwLjY4Ii8+CjwvZz4KPCEtLSBlcXVhbDEgLS0+CjxnIGlkPSJub2RlMyIgY2xhc3M9
Im5vZGUiPgo8dGl0bGU+ZXF1YWwxPC90aXRsZT4KPGVsbGlwc2UgZmlsbD0ibm9uZSIgc3Ryb2tl
PSJibGFjayIgY3g9IjMxOC44NiIgY3k9Ii0xNDgiIHJ4PSI0NS4xNSIgcnk9IjE4Ii8+Cjx0ZXh0
IHRleHQtYW5jaG9yPSJtaWRkbGUiIHg9IjMxOC44NiIgeT0iLTE0NS41IiBmb250LWZhbWlseT0i
Q291cmllciBOZXcsIERlamFWdSBTYW5zIE1vbm8sIG1vbm9zcGFjZSIgZm9udC1zaXplPSIxMC4w
MCI+RXF1YWwgTm9kZTwvdGV4dD4KPC9nPgo8IS0tIG9yJiM0NTsmZ3Q7ZXF1YWwxIC0tPgo8ZyBp
ZD0iZWRnZTIiIGNsYXNzPSJlZGdlIj4KPHRpdGxlPm9yJiM0NTsmZ3Q7ZXF1YWwxPC90aXRsZT4K
PHBhdGggZmlsbD0ibm9uZSIgc3Ryb2tlPSJibGFjayIgZD0iTTI4MC43MywtMTk2LjFDMjg2LjI4
LC0xODkuMjUgMjkyLjg5LC0xODEuMDggMjk5LjA0LC0xNzMuNDgiLz4KPHBvbHlnb24gZmlsbD0i
YmxhY2siIHN0cm9rZT0iYmxhY2siIHBvaW50cz0iMzAxLjg5LC0xNzUuNTMgMzA1LjQ2LC0xNjUu
NTUgMjk2LjQ1LC0xNzEuMTIgMzAxLjg5LC0xNzUuNTMiLz4KPC9nPgo8IS0tIGdyZWF0ZXIgLS0+
CjxnIGlkPSJub2RlNCIgY2xhc3M9Im5vZGUiPgo8dGl0bGU+Z3JlYXRlcjwvdGl0bGU+CjxlbGxp
cHNlIGZpbGw9Im5vbmUiIHN0cm9rZT0iYmxhY2siIGN4PSIxNDIuODYiIGN5PSItODMiIHJ4PSIz
MS40IiByeT0iMTgiLz4KPHRleHQgdGV4dC1hbmNob3I9Im1pZGRsZSIgeD0iMTQyLjg2IiB5PSIt
ODAuNSIgZm9udC1mYW1pbHk9IkNvdXJpZXIgTmV3LCBEZWphVnUgU2FucyBNb25vLCBtb25vc3Bh
Y2UiIGZvbnQtc2l6ZT0iMTAuMDAiPiZndDsgTm9kZTwvdGV4dD4KPC9nPgo8IS0tIGFuZCYjNDU7
Jmd0O2dyZWF0ZXIgLS0+CjxnIGlkPSJlZGdlMyIgY2xhc3M9ImVkZ2UiPgo8dGl0bGU+YW5kJiM0
NTsmZ3Q7Z3JlYXRlcjwvdGl0bGU+CjxwYXRoIGZpbGw9Im5vbmUiIHN0cm9rZT0iYmxhY2siIGQ9
Ik0xOTguOTUsLTEzMS43NUMxODkuNSwtMTIzLjcgMTc3Ljc2LC0xMTMuNzEgMTY3LjQ3LC0xMDQu
OTUiLz4KPHBvbHlnb24gZmlsbD0iYmxhY2siIHN0cm9rZT0iYmxhY2siIHBvaW50cz0iMTY5LjQ4
LC0xMDIuMDYgMTU5LjU5LC05OC4yNSAxNjQuOTQsLTEwNy4zOSAxNjkuNDgsLTEwMi4wNiIvPgo8
L2c+CjwhLS0gZXF1YWwyIC0tPgo8ZyBpZD0ibm9kZTUiIGNsYXNzPSJub2RlIj4KPHRpdGxlPmVx
dWFsMjwvdGl0bGU+CjxlbGxpcHNlIGZpbGw9Im5vbmUiIHN0cm9rZT0iYmxhY2siIGN4PSIyMTgu
ODYiIGN5PSItODMiIHJ4PSIyNyIgcnk9IjE4Ii8+Cjx0ZXh0IHRleHQtYW5jaG9yPSJtaWRkbGUi
IHg9IjIxOC44NiIgeT0iLTgwLjUiIGZvbnQtZmFtaWx5PSJDb3VyaWVyIE5ldywgRGVqYVZ1IFNh
bnMgTW9ubywgbW9ub3NwYWNlIiBmb250LXNpemU9IjEwLjAwIj49PTwvdGV4dD4KPC9nPgo8IS0t
IGFuZCYjNDU7Jmd0O2VxdWFsMiAtLT4KPGcgaWQ9ImVkZ2U0IiBjbGFzcz0iZWRnZSI+Cjx0aXRs
ZT5hbmQmIzQ1OyZndDtlcXVhbDI8L3RpdGxlPgo8cGF0aCBmaWxsPSJub25lIiBzdHJva2U9ImJs
YWNrIiBkPSJNMjE3LjQxLC0xMjkuNzhDMjE3LjU5LC0xMjQuMDkgMjE3Ljc5LC0xMTcuNjEgMjE3
Ljk5LC0xMTEuMzQiLz4KPHBvbHlnb24gZmlsbD0iYmxhY2siIHN0cm9rZT0iYmxhY2siIHBvaW50
cz0iMjIxLjQ5LC0xMTEuMyAyMTguMzEsLTEwMS4xOSAyMTQuNSwtMTExLjA4IDIyMS40OSwtMTEx
LjMiLz4KPC9nPgo8IS0tIGRlcHQgLS0+CjxnIGlkPSJub2RlMTAiIGNsYXNzPSJub2RlIj4KPHRp
dGxlPmRlcHQ8L3RpdGxlPgo8ZWxsaXBzZSBmaWxsPSJub25lIiBzdHJva2U9ImJsYWNrIiBjeD0i
MzE2Ljg2IiBjeT0iLTgzIiByeD0iNDguNzIiIHJ5PSIxOCIvPgo8dGV4dCB0ZXh0LWFuY2hvcj0i
bWlkZGxlIiB4PSIzMTYuODYiIHk9Ii04MC41IiBmb250LWZhbWlseT0iQ291cmllciBOZXcsIERl
amFWdSBTYW5zIE1vbm8sIG1vbm9zcGFjZSIgZm9udC1zaXplPSIxMC4wMCI+Q29sKCZxdW90O2Rl
cHQmcXVvdDspPC90ZXh0Pgo8L2c+CjwhLS0gZXF1YWwxJiM0NTsmZ3Q7ZGVwdCAtLT4KPGcgaWQ9
ImVkZ2U5IiBjbGFzcz0iZWRnZSI+Cjx0aXRsZT5lcXVhbDEmIzQ1OyZndDtkZXB0PC90aXRsZT4K
PHBhdGggZmlsbD0ibm9uZSIgc3Ryb2tlPSJibGFjayIgZD0iTTMxOC4zMSwtMTI5Ljc4QzMxOC4x
MywtMTI0LjA5IDMxNy45MywtMTE3LjYxIDMxNy43MywtMTExLjM0Ii8+Cjxwb2x5Z29uIGZpbGw9
ImJsYWNrIiBzdHJva2U9ImJsYWNrIiBwb2ludHM9IjMyMS4yMiwtMTExLjA4IDMxNy40LC0xMDEu
MTkgMzE0LjIyLC0xMTEuMyAzMjEuMjIsLTExMS4wOCIvPgo8L2c+CjwhLS0gc2VjdXJpdHkgLS0+
CjxnIGlkPSJub2RlMTEiIGNsYXNzPSJub2RlIj4KPHRpdGxlPnNlY3VyaXR5PC90aXRsZT4KPGVs
bGlwc2UgZmlsbD0ibm9uZSIgc3Ryb2tlPSJibGFjayIgY3g9IjQyMS44NiIgY3k9Ii04MyIgcng9
IjM4LjAzIiByeT0iMTgiLz4KPHRleHQgdGV4dC1hbmNob3I9Im1pZGRsZSIgeD0iNDIxLjg2IiB5
PSItODAuNSIgZm9udC1mYW1pbHk9IkNvdXJpZXIgTmV3LCBEZWphVnUgU2FucyBNb25vLCBtb25v
c3BhY2UiIGZvbnQtc2l6ZT0iMTAuMDAiPlNlY3VyaXR5PC90ZXh0Pgo8L2c+CjwhLS0gZXF1YWwx
JiM0NTsmZ3Q7c2VjdXJpdHkgLS0+CjxnIGlkPSJlZGdlMTAiIGNsYXNzPSJlZGdlIj4KPHRpdGxl
PmVxdWFsMSYjNDU7Jmd0O3NlY3VyaXR5PC90aXRsZT4KPHBhdGggZmlsbD0ibm9uZSIgc3Ryb2tl
PSJibGFjayIgZD0iTTM0Mi40OSwtMTMyLjU1QzM1Ni43OSwtMTIzLjggMzc1LjE5LC0xMTIuNTUg
MzkwLjY3LC0xMDMuMDgiLz4KPHBvbHlnb24gZmlsbD0iYmxhY2siIHN0cm9rZT0iYmxhY2siIHBv
aW50cz0iMzkyLjY5LC0xMDUuOTQgMzk5LjQsLTk3Ljc0IDM4OS4wNCwtOTkuOTcgMzkyLjY5LC0x
MDUuOTQiLz4KPC9nPgo8IS0tIGFnZSAtLT4KPGcgaWQ9Im5vZGU2IiBjbGFzcz0ibm9kZSI+Cjx0
aXRsZT5hZ2U8L3RpdGxlPgo8ZWxsaXBzZSBmaWxsPSJub25lIiBzdHJva2U9ImJsYWNrIiBjeD0i
NDguODYiIGN5PSItMTgiIHJ4PSI0OC43MiIgcnk9IjE4Ii8+Cjx0ZXh0IHRleHQtYW5jaG9yPSJt
aWRkbGUiIHg9IjQ4Ljg2IiB5PSItMTUuNSIgZm9udC1mYW1pbHk9IkNvdXJpZXIgTmV3LCBEZWph
VnUgU2FucyBNb25vLCBtb25vc3BhY2UiIGZvbnQtc2l6ZT0iMTAuMDAiPkNvbCAoJnF1b3Q7YWdl
JnF1b3Q7KTwvdGV4dD4KPC9nPgo8IS0tIGdyZWF0ZXImIzQ1OyZndDthZ2UgLS0+CjxnIGlkPSJl
ZGdlNSIgY2xhc3M9ImVkZ2UiPgo8dGl0bGU+Z3JlYXRlciYjNDU7Jmd0O2FnZTwvdGl0bGU+Cjxw
YXRoIGZpbGw9Im5vbmUiIHN0cm9rZT0iYmxhY2siIGQ9Ik0xMjIuOTIsLTY4LjY0QzExMC40Mywt
NjAuMjYgOTQuMDYsLTQ5LjI5IDc5Ljg5LC0zOS44Ii8+Cjxwb2x5Z29uIGZpbGw9ImJsYWNrIiBz
dHJva2U9ImJsYWNrIiBwb2ludHM9IjgxLjY2LC0zNi43NyA3MS40LC0zNC4xMSA3Ny43NiwtNDIu
NTggODEuNjYsLTM2Ljc3Ii8+CjwvZz4KPCEtLSAyNSAtLT4KPGcgaWQ9Im5vZGU3IiBjbGFzcz0i
bm9kZSI+Cjx0aXRsZT4yNTwvdGl0bGU+CjxlbGxpcHNlIGZpbGw9Im5vbmUiIHN0cm9rZT0iYmxh
Y2siIGN4PSIxNDIuODYiIGN5PSItMTgiIHJ4PSIyNyIgcnk9IjE4Ii8+Cjx0ZXh0IHRleHQtYW5j
aG9yPSJtaWRkbGUiIHg9IjE0Mi44NiIgeT0iLTE1LjUiIGZvbnQtZmFtaWx5PSJDb3VyaWVyIE5l
dywgRGVqYVZ1IFNhbnMgTW9ubywgbW9ub3NwYWNlIiBmb250LXNpemU9IjEwLjAwIj4yNTwvdGV4
dD4KPC9nPgo8IS0tIGdyZWF0ZXImIzQ1OyZndDsyNSAtLT4KPGcgaWQ9ImVkZ2U2IiBjbGFzcz0i
ZWRnZSI+Cjx0aXRsZT5ncmVhdGVyJiM0NTsmZ3Q7MjU8L3RpdGxlPgo8cGF0aCBmaWxsPSJub25l
IiBzdHJva2U9ImJsYWNrIiBkPSJNMTQyLjg2LC02NC43OEMxNDIuODYsLTU5LjA5IDE0Mi44Niwt
NTIuNjEgMTQyLjg2LC00Ni4zNCIvPgo8cG9seWdvbiBmaWxsPSJibGFjayIgc3Ryb2tlPSJibGFj
ayIgcG9pbnRzPSIxNDYuMzYsLTQ2LjE5IDE0Mi44NiwtMzYuMTkgMTM5LjM2LC00Ni4xOSAxNDYu
MzYsLTQ2LjE5Ii8+CjwvZz4KPCEtLSBjb2wgLS0+CjxnIGlkPSJub2RlOCIgY2xhc3M9Im5vZGUi
Pgo8dGl0bGU+Y29sPC90aXRsZT4KPGVsbGlwc2UgZmlsbD0ibm9uZSIgc3Ryb2tlPSJibGFjayIg
Y3g9IjIxOC44NiIgY3k9Ii0xOCIgcng9IjMxLjQiIHJ5PSIxOCIvPgo8dGV4dCB0ZXh0LWFuY2hv
cj0ibWlkZGxlIiB4PSIyMTguODYiIHk9Ii0xNS41IiBmb250LWZhbWlseT0iQ291cmllciBOZXcs
IERlamFWdSBTYW5zIE1vbm8sIG1vbm9zcGFjZSIgZm9udC1zaXplPSIxMC4wMCI+Q29sLi4uPC90
ZXh0Pgo8L2c+CjwhLS0gZXF1YWwyJiM0NTsmZ3Q7Y29sIC0tPgo8ZyBpZD0iZWRnZTciIGNsYXNz
PSJlZGdlIj4KPHRpdGxlPmVxdWFsMiYjNDU7Jmd0O2NvbDwvdGl0bGU+CjxwYXRoIGZpbGw9Im5v
bmUiIHN0cm9rZT0iYmxhY2siIGQ9Ik0yMTguODYsLTY0Ljc4QzIxOC44NiwtNTkuMDkgMjE4Ljg2
LC01Mi42MSAyMTguODYsLTQ2LjM0Ii8+Cjxwb2x5Z29uIGZpbGw9ImJsYWNrIiBzdHJva2U9ImJs
YWNrIiBwb2ludHM9IjIyMi4zNiwtNDYuMTkgMjE4Ljg2LC0zNi4xOSAyMTUuMzYsLTQ2LjE5IDIy
Mi4zNiwtNDYuMTkiLz4KPC9nPgo8IS0tIHN5c3RlbXMgLS0+CjxnIGlkPSJub2RlOSIgY2xhc3M9
Im5vZGUiPgo8dGl0bGU+c3lzdGVtczwvdGl0bGU+CjxlbGxpcHNlIGZpbGw9Im5vbmUiIHN0cm9r
ZT0iYmxhY2siIGN4PSIzMDkuODYiIGN5PSItMTgiIHJ4PSI0MS41OSIgcnk9IjE4Ii8+Cjx0ZXh0
IHRleHQtYW5jaG9yPSJtaWRkbGUiIHg9IjMwOS44NiIgeT0iLTE1LjUiIGZvbnQtZmFtaWx5PSJD
b3VyaWVyIE5ldywgRGVqYVZ1IFNhbnMgTW9ubywgbW9ub3NwYWNlIiBmb250LXNpemU9IjEwLjAw
Ij4mcXVvdDtTeXN0ZW1zJnF1b3Q7PC90ZXh0Pgo8L2c+CjwhLS0gZXF1YWwyJiM0NTsmZ3Q7c3lz
dGVtcyAtLT4KPGcgaWQ9ImVkZ2U4IiBjbGFzcz0iZWRnZSI+Cjx0aXRsZT5lcXVhbDImIzQ1OyZn
dDtzeXN0ZW1zPC90aXRsZT4KPHBhdGggZmlsbD0ibm9uZSIgc3Ryb2tlPSJibGFjayIgZD0iTTIz
Ni44NCwtNjkuNTVDMjQ5LjE5LC02MSAyNjUuODgsLTQ5LjQ1IDI4MC4xOSwtMzkuNTQiLz4KPHBv
bHlnb24gZmlsbD0iYmxhY2siIHN0cm9rZT0iYmxhY2siIHBvaW50cz0iMjgyLjQ5LC00Mi4yIDI4
OC43MiwtMzMuNjMgMjc4LjUxLC0zNi40NSAyODIuNDksLTQyLjIiLz4KPC9nPgo8L2c+Cjwvc3Zn
Pgo=
" alt="" />
<figcaption>Figure 16–9: Boolean Expression Tree</figcaption>
</figure>

**Specification:**

Design an Abstract Syntax Tree (AST) hierarchy for boolean expressions
based on a `BooleanExpr` base class. Implement terminal nodes
(`ComparisonPredicate`) and non-terminal combinators (`AndExpr`,
`OrExpr`, `NotExpr`). Integrate this tree into `PhysicalScanNode`.

``` cpp
class BooleanExpr {
public:
    virtual ~BooleanExpr() = default;
    [[nodiscard]] virtual bool evaluate(const Row& row, const Schema& schema) const = 0;
};

class BinaryLogicExpr : public BooleanExpr {
public:
    enum class LogicOp { And, Or };
    BinaryLogicExpr(std::unique_ptr<BooleanExpr> left, LogicOp op, std::unique_ptr<BooleanExpr> right);
    bool evaluate(const Row& row, const Schema& schema) const override;
private:
    std::unique_ptr<BooleanExpr> m_left, m_right;
    LogicOp m_op;
};
```

**Verification Tasks:**

1.  **De Morgan's Laws Equivalence:** Construct two filters over a
    table: $P_1 = \neg (A \land B)$ and $P_2 = (\neg A) \lor (\neg B)$.
    Verify across $1,000$ diverse rows that $P_1$ and $P_2$ emit
    identical sets of `RowID` instances.
2.  **Short-Circuit Evaluation:** Construct an expression
    $(A = 1) \lor (B / 0 = 1)$. Configure the left branch to evaluate to
    `TRUE`. Verify that the right branch is never evaluated, avoiding a
    division-by-zero exception.
3.  **Three-Valued Logic Null Check:** Evaluate `(NULL = NULL)` and
    `(age > 20 AND NULL)`. Verify that both evaluate to `UNKNOWN`
    (treated as `FALSE` by the selection filter) and that no rows are
    returned.

#### Exercise 16.6: Paged B+ Tree Leaf Traversal

**Context and Theory:**

Binary search trees (such as the AVL tree used in Section 16.8) suffer
from high tree heights ($h = \log_2 N$) and poor cache locality.
Production database storage engines rely on **B+ Trees**.

A B+ Tree differs from a standard B-Tree in two critical structural
ways:

1.  Internal nodes store only routing keys and child pointers; they
    never store row references.
2.  Leaf nodes store all key-payload pairs and are linked contiguously
    via `next` and `prev` pointers into a doubly-linked list.

This architecture enables $O(\log_B N)$ point lookups, followed by
$O(K)$ sequential memory traversals for range queries
$[k_{\text{start}}, k_{\text{end}}]$, avoiding parent node backtracking.

<figure id="fig:b-plus-tree">
<img src="data:image/svg+xml;base64,PHN2ZyB4bWxucz0iaHR0cDovL3d3dy53My5vcmcvMjAwMC9zdmciIHdpZHRoPSI0NDQuNiIgaGVp
Z2h0PSI2Ni40IiBjbGFzcz0ic3ZnYm9iIiB2aWV3Qm94PSIwIC00IDQ0NC42IDY2LjQiPgogIDxz
dHlsZT4uc3ZnYm9iIGxpbmUsIC5zdmdib2IgcGF0aCwgLnN2Z2JvYiBjaXJjbGUsIC5zdmdib2Ig
cmVjdCwgLnN2Z2JvYiBwb2x5Z29uIHsKICBzdHJva2U6IGJsYWNrOwogIHN0cm9rZS13aWR0aDog
MS41OwogIHN0cm9rZS1vcGFjaXR5OiAxOwogIGZpbGwtb3BhY2l0eTogMTsKICBzdHJva2UtbGlu
ZWNhcDogcm91bmQ7CiAgc3Ryb2tlLWxpbmVqb2luOiBtaXRlcjsKfQoKLnN2Z2JvYiB0ZXh0IHsK
ICB3aGl0ZS1zcGFjZTogcHJlOwogIGZpbGw6IGJsYWNrOwogIGZvbnQtZmFtaWx5OiBDb3VyaWVy
IE5ldywgRGVqYVZ1IFNhbnMgTW9ubywgbW9ub3NwYWNlOwogIGZvbnQtc2l6ZTogMTNweDsKfQoK
LnN2Z2JvYiByZWN0LmJhY2tkcm9wIHsKICBzdHJva2U6IG5vbmU7CiAgZmlsbDogdHJhbnNwYXJl
bnQ7Cn0KCi5zdmdib2IgLmJyb2tlbiB7CiAgc3Ryb2tlLWRhc2hhcnJheTogODsKfQoKLnN2Z2Jv
YiAuZmlsbGVkIHsKICBmaWxsOiBibGFjazsKfQoKLnN2Z2JvYiAuYmdfZmlsbGVkIHsKICBmaWxs
OiB0cmFuc3BhcmVudDsKICBzdHJva2Utd2lkdGg6IDE7Cn0KCi5zdmdib2IgLm5vZmlsbCB7CiAg
ZmlsbDogdHJhbnNwYXJlbnQ7Cn0KCi5zdmdib2IgLmVuZF9tYXJrZWRfYXJyb3cgewogIG1hcmtl
ci1lbmQ6IHVybCgjYXJyb3cpOwp9Cgouc3ZnYm9iIC5zdGFydF9tYXJrZWRfYXJyb3cgewogIG1h
cmtlci1zdGFydDogdXJsKCNhcnJvdyk7Cn0KCi5zdmdib2IgLmVuZF9tYXJrZWRfZGlhbW9uZCB7
CiAgbWFya2VyLWVuZDogdXJsKCNkaWFtb25kKTsKfQoKLnN2Z2JvYiAuc3RhcnRfbWFya2VkX2Rp
YW1vbmQgewogIG1hcmtlci1zdGFydDogdXJsKCNkaWFtb25kKTsKfQoKLnN2Z2JvYiAuZW5kX21h
cmtlZF9jaXJjbGUgewogIG1hcmtlci1lbmQ6IHVybCgjY2lyY2xlKTsKfQoKLnN2Z2JvYiAuc3Rh
cnRfbWFya2VkX2NpcmNsZSB7CiAgbWFya2VyLXN0YXJ0OiB1cmwoI2NpcmNsZSk7Cn0KCi5zdmdi
b2IgLmVuZF9tYXJrZWRfb3Blbl9jaXJjbGUgewogIG1hcmtlci1lbmQ6IHVybCgjb3Blbl9jaXJj
bGUpOwp9Cgouc3ZnYm9iIC5zdGFydF9tYXJrZWRfb3Blbl9jaXJjbGUgewogIG1hcmtlci1zdGFy
dDogdXJsKCNvcGVuX2NpcmNsZSk7Cn0KCi5zdmdib2IgLmVuZF9tYXJrZWRfYmlnX29wZW5fY2ly
Y2xlIHsKICBtYXJrZXItZW5kOiB1cmwoI2JpZ19vcGVuX2NpcmNsZSk7Cn0KCi5zdmdib2IgLnN0
YXJ0X21hcmtlZF9iaWdfb3Blbl9jaXJjbGUgewogIG1hcmtlci1zdGFydDogdXJsKCNiaWdfb3Bl
bl9jaXJjbGUpOwp9Cgo8L3N0eWxlPgogIDxkZWZzPgogICAgPG1hcmtlciBpZD0iYXJyb3ciIHZp
ZXdCb3g9Ii0yIC0yIDggOCIgcmVmWD0iNCIgcmVmWT0iMiIgbWFya2VyV2lkdGg9IjciIG1hcmtl
ckhlaWdodD0iNyIgb3JpZW50PSJhdXRvLXN0YXJ0LXJldmVyc2UiPgogICAgICA8cG9seWdvbiBw
b2ludHM9IjAsMCAwLDQgNCwyIDAsMCI+PC9wb2x5Z29uPgogICAgPC9tYXJrZXI+CiAgICA8bWFy
a2VyIGlkPSJkaWFtb25kIiB2aWV3Qm94PSItMiAtMiA4IDgiIHJlZlg9IjQiIHJlZlk9IjIiIG1h
cmtlcldpZHRoPSI3IiBtYXJrZXJIZWlnaHQ9IjciIG9yaWVudD0iYXV0by1zdGFydC1yZXZlcnNl
Ij4KICAgICAgPHBvbHlnb24gcG9pbnRzPSIwLDIgMiwwIDQsMiAyLDQgMCwyIj48L3BvbHlnb24+
CiAgICA8L21hcmtlcj4KICAgIDxtYXJrZXIgaWQ9ImNpcmNsZSIgdmlld0JveD0iMCAwIDggOCIg
cmVmWD0iNCIgcmVmWT0iNCIgbWFya2VyV2lkdGg9IjciIG1hcmtlckhlaWdodD0iNyIgb3JpZW50
PSJhdXRvLXN0YXJ0LXJldmVyc2UiPgogICAgICA8Y2lyY2xlIGN4PSI0IiBjeT0iNCIgcj0iMiIg
Y2xhc3M9ImZpbGxlZCI+PC9jaXJjbGU+CiAgICA8L21hcmtlcj4KICAgIDxtYXJrZXIgaWQ9Im9w
ZW5fY2lyY2xlIiB2aWV3Qm94PSIwIDAgOCA4IiByZWZYPSI0IiByZWZZPSI0IiBtYXJrZXJXaWR0
aD0iNyIgbWFya2VySGVpZ2h0PSI3IiBvcmllbnQ9ImF1dG8tc3RhcnQtcmV2ZXJzZSI+CiAgICAg
IDxjaXJjbGUgY3g9IjQiIGN5PSI0IiByPSIyIiBjbGFzcz0iYmdfZmlsbGVkIj48L2NpcmNsZT4K
ICAgIDwvbWFya2VyPgogICAgPG1hcmtlciBpZD0iYmlnX29wZW5fY2lyY2xlIiB2aWV3Qm94PSIw
IDAgOCA4IiByZWZYPSI0IiByZWZZPSI0IiBtYXJrZXJXaWR0aD0iNyIgbWFya2VySGVpZ2h0PSI3
IiBvcmllbnQ9ImF1dG8tc3RhcnQtcmV2ZXJzZSI+CiAgICAgIDxjaXJjbGUgY3g9IjQiIGN5PSI0
IiByPSIzIiBjbGFzcz0iYmdfZmlsbGVkIj48L2NpcmNsZT4KICAgIDwvbWFya2VyPgogIDwvZGVm
cz4KICA8cmVjdCBjbGFzcz0iYmFja2Ryb3AiIHg9IjAiIHk9Ii00IiB3aWR0aD0iNDQ0LjYiIGhl
aWdodD0iNjYuNCI+PC9yZWN0PgogIDxwb2x5Z29uIHBvaW50cz0iMTk1LDMuOSAxODcuMjAwMDEs
Ny44IDE5NSwxMS43MDAwMDEiIGNsYXNzPSJmaWxsZWQiPjwvcG9seWdvbj4KICA8bGluZSB4MT0i
MTk1IiB5MT0iNy44IiB4Mj0iMjU3LjQiIHkyPSI3LjgiIGNsYXNzPSJzb2xpZCI+PC9saW5lPgog
IDxwb2x5Z29uIHBvaW50cz0iMjU3LjQsMy45IDI2NS4yLDcuOCAyNTcuNCwxMS43MDAwMDEiIGNs
YXNzPSJmaWxsZWQiPjwvcG9seWdvbj4KICA8bGluZSB4MT0iNzQuMSIgeTE9IjE1LjYiIHgyPSI3
NC4xIiB5Mj0iMzEuMiIgY2xhc3M9InNvbGlkIj48L2xpbmU+CiAgPHRleHQgeG1sOnNwYWNlPSJw
cmVzZXJ2ZSIgeD0iNTYuNTUwMDAzIiB5PSI0Mi45IiA+Um93MDwvdGV4dD4KICA8bGluZSB4MT0i
MTEzLjEwMDAwNiIgeTE9IjE1LjYiIHgyPSIxMTMuMTAwMDA2IiB5Mj0iMzEuMiIgY2xhc3M9InNv
bGlkIj48L2xpbmU+CiAgPHRleHQgeG1sOnNwYWNlPSJwcmVzZXJ2ZSIgeD0iOTUuNTUiIHk9IjQy
LjkiID5Sb3cxPC90ZXh0PgogIDxsaW5lIHgxPSIxNTIuMSIgeTE9IjE1LjYiIHgyPSIxNTIuMSIg
eTI9IjMxLjIiIGNsYXNzPSJzb2xpZCI+PC9saW5lPgogIDx0ZXh0IHhtbDpzcGFjZT0icHJlc2Vy
dmUiIHg9IjEzNC41NSIgeT0iNDIuOSIgPlJvdzI8L3RleHQ+CiAgPGxpbmUgeDE9IjMzOS4zMDAw
MiIgeTE9IjE1LjYiIHgyPSIzMzkuMzAwMDIiIHkyPSIzMS4yIiBjbGFzcz0ic29saWQiPjwvbGlu
ZT4KICA8dGV4dCB4bWw6c3BhY2U9InByZXNlcnZlIiB4PSIzMjEuNzUiIHk9IjQyLjkiID5Sb3cz
PC90ZXh0PgogIDxsaW5lIHgxPSIzNzguMzAwMDIiIHkxPSIxNS42IiB4Mj0iMzc4LjMwMDAyIiB5
Mj0iMzEuMiIgY2xhc3M9InNvbGlkIj48L2xpbmU+CiAgPHRleHQgeG1sOnNwYWNlPSJwcmVzZXJ2
ZSIgeD0iMzYwLjc1IiB5PSI0Mi45IiA+Um93NDwvdGV4dD4KICA8bGluZSB4MT0iNDE3LjMwMDAy
IiB5MT0iMTUuNiIgeDI9IjQxNy4zMDAwMiIgeTI9IjMxLjIiIGNsYXNzPSJzb2xpZCI+PC9saW5l
PgogIDx0ZXh0IHhtbDpzcGFjZT0icHJlc2VydmUiIHg9IjM5OS43NSIgeT0iNDIuOSIgPlJvdzU8
L3RleHQ+CiAgPHRleHQgeG1sOnNwYWNlPSJwcmVzZXJ2ZSIgeD0iOS43NSIgeT0iMTEuNzAwMDAx
IiA+TGVhZiBbIDEwIHwgMjAgfCAzMCBdPC90ZXh0PgogIDx0ZXh0IHhtbDpzcGFjZT0icHJlc2Vy
dmUiIHg9IjI3NC45NSIgeT0iMTEuNzAwMDAxIiA+TGVhZiBbIDQwIHwgNTAgfCA2MCBdPC90ZXh0
Pgo8L3N2Zz4K
" alt="" />
<figcaption>Figure 16–10: B+ Tree Layout</figcaption>
</figure>

**Specification:**

Implement an in-memory B+ Tree leaf node structure, `BPlusTreeLeafNode`,
with fixed capacity $B$. Support key insertion with node splitting, key
lookups, and range scanning using linked leaf sibling pointers.

``` cpp
template <typename Key, typename Value, size_t Capacity = 4>
struct BPlusTreeLeafNode {
    size_t size{0};
    std::array<Key, Capacity> keys;
    std::array<std::vector<Value>, Capacity> payloads;
    BPlusTreeLeafNode* next{nullptr};
    BPlusTreeLeafNode* prev{nullptr};

    [[nodiscard]] bool is_full() const noexcept { return size == Capacity; }
    size_t find_key_index(const Key& key) const;
    void insert_non_full(const Key& key, const Value& val);
    std::unique_ptr<BPlusTreeLeafNode> split();
};
```

**Verification Tasks:**

1.  **Node Split and Link Invariant:** Insert $5$ sorted keys into a
    leaf node with capacity $B = 4$. Verify that splitting creates a new
    leaf, distributes keys evenly, and sets `left->next == right` and
    `right->prev == left`.
2.  **Range Query via Sibling Links:** Populate a multi-leaf structure
    with integers $1 \dots 100$. Execute a range search for
    $25 \le k \le 75$. Verify that the scan finds key $25$ via index
    lookup and traverses sibling pointers directly to key $75$ without
    referencing parent nodes.
3.  **Reverse Range Traversal:** Utilise `prev` pointers to scan
    backwards from key $50$ down to $10$. Verify exact descending
    sequence retrieval.

#### Exercise 16.7: Cost-Based Optimiser and Selectivity Estimation

**Context and Theory:**

A query optimiser translates a logical relational algebra expression
into an efficient physical execution plan. When evaluating a selection
predicate $\sigma_{A = v}(R)$, the optimiser must decide between:

1.  **Full Sequential Scan:** Cost
    $C_{\text{seq}} = |R| \cdot c_{\text{cpu}}$
2.  **Index Scan:** Cost
    $C_{\text{idx}} = \log_2 N \cdot c_{\text{tree}} + (\text{Selectivity} \cdot |R|) \cdot (c_{\text{fetch}} + c_{\text{cpu}})$

**Selectivity ($S \in [0.0, 1.0]$)** represents the fraction of tuples
expected to satisfy the predicate. Assuming a uniform distribution over
domain cardinality $V(A, R)$:

$$S(A = v) = \frac{1}{|V(A, R)|} \qquad S(A > v) = \frac{\max(A) - v}{\max(A) - \min(A)}$$

If selectivity is high (e.g., $S > 0.3$), sequential scanning is faster
than index lookups due to cache prefetching and eliminating secondary
index pointer indirection, see also Figure 16–11.

<figure id="fig:selectivity-estimator">
<img src="data:image/svg+xml;base64,PD94bWwgdmVyc2lvbj0iMS4wIiBlbmNvZGluZz0iVVRGLTgiIHN0YW5kYWxvbmU9Im5vIj8+Cjwh
RE9DVFlQRSBzdmcgUFVCTElDICItLy9XM0MvL0RURCBTVkcgMS4xLy9FTiIKICJodHRwOi8vd3d3
LnczLm9yZy9HcmFwaGljcy9TVkcvMS4xL0RURC9zdmcxMS5kdGQiPgo8IS0tIEdlbmVyYXRlZCBi
eSBncmFwaHZpeiB2ZXJzaW9uIDIuNDMuMCAoMCkKIC0tPgo8IS0tIFRpdGxlOiBHIFBhZ2VzOiAx
IC0tPgo8c3ZnIHdpZHRoPSIyNjRwdCIgaGVpZ2h0PSIyMThwdCIKIHZpZXdCb3g9IjAuMDAgMC4w
MCAyNjMuOTEgMjE4LjAwIiB4bWxucz0iaHR0cDovL3d3dy53My5vcmcvMjAwMC9zdmciIHhtbG5z
OnhsaW5rPSJodHRwOi8vd3d3LnczLm9yZy8xOTk5L3hsaW5rIj4KPGcgaWQ9ImdyYXBoMCIgY2xh
c3M9ImdyYXBoIiB0cmFuc2Zvcm09InNjYWxlKDEgMSkgcm90YXRlKDApIHRyYW5zbGF0ZSg0IDIx
NCkiPgo8dGl0bGU+RzwvdGl0bGU+CjwhLS0gcXVlcnkgLS0+CjxnIGlkPSJub2RlMSIgY2xhc3M9
Im5vZGUiPgo8dGl0bGU+cXVlcnk8L3RpdGxlPgo8cG9seWdvbiBmaWxsPSJub25lIiBzdHJva2U9
ImJsYWNrIiBwb2ludHM9IjE4My45NSwtMjEwIDcwLjk1LC0yMTAgNzAuOTUsLTE3NCAxODMuOTUs
LTE3NCAxODMuOTUsLTIxMCIvPgo8dGV4dCB0ZXh0LWFuY2hvcj0ibWlkZGxlIiB4PSIxMjcuNDUi
IHk9Ii0xODkuNSIgZm9udC1mYW1pbHk9IkNvdXJpZXIgTmV3LCBEZWphVnUgU2FucyBNb25vLCBt
b25vc3BhY2UiIGZvbnQtc2l6ZT0iMTAuMDAiPlF1ZXJ5OiBTRUxFQ1QuLi48L3RleHQ+CjwvZz4K
PCEtLSBlc3RpbWF0b3IgLS0+CjxnIGlkPSJub2RlMiIgY2xhc3M9Im5vZGUiPgo8dGl0bGU+ZXN0
aW1hdG9yPC90aXRsZT4KPGVsbGlwc2UgZmlsbD0ibm9uZSIgc3Ryb2tlPSJibGFjayIgY3g9IjEy
Ny40NSIgY3k9Ii0xMzQiIHJ4PSI4NC4zNiIgcnk9IjE4Ii8+Cjx0ZXh0IHRleHQtYW5jaG9yPSJt
aWRkbGUiIHg9IjEyNy40NSIgeT0iLTEzMS41IiBmb250LWZhbWlseT0iQ291cmllciBOZXcsIERl
amFWdSBTYW5zIE1vbm8sIG1vbm9zcGFjZSIgZm9udC1zaXplPSIxMC4wMCI+U2VsZWN0aXZpdHkg
RXN0aW1hdG9yPC90ZXh0Pgo8L2c+CjwhLS0gcXVlcnkmIzQ1OyZndDtlc3RpbWF0b3IgLS0+Cjxn
IGlkPSJlZGdlMSIgY2xhc3M9ImVkZ2UiPgo8dGl0bGU+cXVlcnkmIzQ1OyZndDtlc3RpbWF0b3I8
L3RpdGxlPgo8cGF0aCBmaWxsPSJub25lIiBzdHJva2U9ImJsYWNrIiBkPSJNMTI3LjQ1LC0xNzMu
ODlDMTI3LjQ1LC0xNzAuMiAxMjcuNDUsLTE2Ni4yMSAxMjcuNDUsLTE2Mi4yNCIvPgo8cG9seWdv
biBmaWxsPSJibGFjayIgc3Ryb2tlPSJibGFjayIgcG9pbnRzPSIxMzAuOTYsLTE2MiAxMjcuNDUs
LTE1MiAxMjMuOTYsLTE2MiAxMzAuOTYsLTE2MiIvPgo8L2c+CjwhLS0gdGhyZXNob2xkMSAtLT4K
PGcgaWQ9Im5vZGUzIiBjbGFzcz0ibm9kZSI+Cjx0aXRsZT50aHJlc2hvbGQxPC90aXRsZT4KPGVs
bGlwc2UgZmlsbD0ibm9uZSIgc3Ryb2tlPSJibGFjayIgY3g9IjYxLjQ1IiBjeT0iLTc2IiByeD0i
NTUuODUiIHJ5PSIxOCIvPgo8dGV4dCB0ZXh0LWFuY2hvcj0ibWlkZGxlIiB4PSI2MS40NSIgeT0i
LTczLjUiIGZvbnQtZmFtaWx5PSJDb3VyaWVyIE5ldywgRGVqYVZ1IFNhbnMgTW9ubywgbW9ub3Nw
YWNlIiBmb250LXNpemU9IjEwLjAwIj5TICZsdDsgVGhyZXNob2xkPC90ZXh0Pgo8L2c+CjwhLS0g
ZXN0aW1hdG9yJiM0NTsmZ3Q7dGhyZXNob2xkMSAtLT4KPGcgaWQ9ImVkZ2UyIiBjbGFzcz0iZWRn
ZSI+Cjx0aXRsZT5lc3RpbWF0b3ImIzQ1OyZndDt0aHJlc2hvbGQxPC90aXRsZT4KPHBhdGggZmls
bD0ibm9uZSIgc3Ryb2tlPSJibGFjayIgZD0iTTEwNy42NSwtMTE2LjJDMTAxLjU3LC0xMTEuMDMg
OTQuNzYsLTEwNS4yNiA4OC4yOSwtOTkuNzciLz4KPHBvbHlnb24gZmlsbD0iYmxhY2siIHN0cm9r
ZT0iYmxhY2siIHBvaW50cz0iOTAuMjMsLTk2LjgyIDgwLjM0LC05My4wMiA4NS43LC0xMDIuMTYg
OTAuMjMsLTk2LjgyIi8+CjwvZz4KPCEtLSB0aHJlc2hvbGQyIC0tPgo8ZyBpZD0ibm9kZTQiIGNs
YXNzPSJub2RlIj4KPHRpdGxlPnRocmVzaG9sZDI8L3RpdGxlPgo8ZWxsaXBzZSBmaWxsPSJub25l
IiBzdHJva2U9ImJsYWNrIiBjeD0iMTk1LjQ1IiBjeT0iLTc2IiByeD0iNTkuNDEiIHJ5PSIxOCIv
Pgo8dGV4dCB0ZXh0LWFuY2hvcj0ibWlkZGxlIiB4PSIxOTUuNDUiIHk9Ii03My41IiBmb250LWZh
bWlseT0iQ291cmllciBOZXcsIERlamFWdSBTYW5zIE1vbm8sIG1vbm9zcGFjZSIgZm9udC1zaXpl
PSIxMC4wMCI+UyAmZ3Q7PSBUaHJlc2hvbGQ8L3RleHQ+CjwvZz4KPCEtLSBlc3RpbWF0b3ImIzQ1
OyZndDt0aHJlc2hvbGQyIC0tPgo8ZyBpZD0iZWRnZTMiIGNsYXNzPSJlZGdlIj4KPHRpdGxlPmVz
dGltYXRvciYjNDU7Jmd0O3RocmVzaG9sZDI8L3RpdGxlPgo8cGF0aCBmaWxsPSJub25lIiBzdHJv
a2U9ImJsYWNrIiBkPSJNMTQ3LjQ5LC0xMTYuNUMxNTMuOTEsLTExMS4yMSAxNjEuMTMsLTEwNS4y
NiAxNjcuOTksLTk5LjYyIi8+Cjxwb2x5Z29uIGZpbGw9ImJsYWNrIiBzdHJva2U9ImJsYWNrIiBw
b2ludHM9IjE3MC4zLC0xMDIuMjUgMTc1LjgsLTkzLjE5IDE2NS44NSwtOTYuODQgMTcwLjMsLTEw
Mi4yNSIvPgo8L2c+CjwhLS0gc2NhbjEgLS0+CjxnIGlkPSJub2RlNSIgY2xhc3M9Im5vZGUiPgo8
dGl0bGU+c2NhbjE8L3RpdGxlPgo8ZWxsaXBzZSBmaWxsPSJub25lIiBzdHJva2U9ImJsYWNrIiBj
eD0iNTkuNDUiIGN5PSItMTgiIHJ4PSI1OS40MSIgcnk9IjE4Ii8+Cjx0ZXh0IHRleHQtYW5jaG9y
PSJtaWRkbGUiIHg9IjU5LjQ1IiB5PSItMTUuNSIgZm9udC1mYW1pbHk9IkNvdXJpZXIgTmV3LCBE
ZWphVnUgU2FucyBNb25vLCBtb25vc3BhY2UiIGZvbnQtc2l6ZT0iMTAuMDAiPlVzZSBJbmRleCBT
Y2FuPC90ZXh0Pgo8L2c+CjwhLS0gdGhyZXNob2xkMSYjNDU7Jmd0O3NjYW4xIC0tPgo8ZyBpZD0i
ZWRnZTQiIGNsYXNzPSJlZGdlIj4KPHRpdGxlPnRocmVzaG9sZDEmIzQ1OyZndDtzY2FuMTwvdGl0
bGU+CjxwYXRoIGZpbGw9Im5vbmUiIHN0cm9rZT0iYmxhY2siIGQ9Ik02MC44NCwtNTcuODlDNjAu
NzEsLTU0LjIgNjAuNTcsLTUwLjIxIDYwLjQzLC00Ni4yNCIvPgo8cG9seWdvbiBmaWxsPSJibGFj
ayIgc3Ryb2tlPSJibGFjayIgcG9pbnRzPSI2My45MiwtNDUuODcgNjAuMDYsLTM2IDU2LjkyLC00
Ni4xMiA2My45MiwtNDUuODciLz4KPC9nPgo8IS0tIHNjYW4yIC0tPgo8ZyBpZD0ibm9kZTYiIGNs
YXNzPSJub2RlIj4KPHRpdGxlPnNjYW4yPC90aXRsZT4KPGVsbGlwc2UgZmlsbD0ibm9uZSIgc3Ry
b2tlPSJibGFjayIgY3g9IjE5Ni40NSIgY3k9Ii0xOCIgcng9IjU5LjQxIiByeT0iMTgiLz4KPHRl
eHQgdGV4dC1hbmNob3I9Im1pZGRsZSIgeD0iMTk2LjQ1IiB5PSItMTUuNSIgZm9udC1mYW1pbHk9
IkNvdXJpZXIgTmV3LCBEZWphVnUgU2FucyBNb25vLCBtb25vc3BhY2UiIGZvbnQtc2l6ZT0iMTAu
MDAiPlVzZSBUYWJsZSBTY2FuPC90ZXh0Pgo8L2c+CjwhLS0gdGhyZXNob2xkMiYjNDU7Jmd0O3Nj
YW4yIC0tPgo8ZyBpZD0iZWRnZTUiIGNsYXNzPSJlZGdlIj4KPHRpdGxlPnRocmVzaG9sZDImIzQ1
OyZndDtzY2FuMjwvdGl0bGU+CjxwYXRoIGZpbGw9Im5vbmUiIHN0cm9rZT0iYmxhY2siIGQ9Ik0x
OTUuNzYsLTU3Ljg5QzE5NS44MywtNTQuMiAxOTUuOSwtNTAuMjEgMTk1Ljk3LC00Ni4yNCIvPgo8
cG9seWdvbiBmaWxsPSJibGFjayIgc3Ryb2tlPSJibGFjayIgcG9pbnRzPSIxOTkuNDcsLTQ2LjA2
IDE5Ni4xNSwtMzYgMTkyLjQ3LC00NS45NCAxOTkuNDcsLTQ2LjA2Ii8+CjwvZz4KPC9nPgo8L3N2
Zz4K
" alt="" />
<figcaption>Figure 16–11: Selectivity Estimator Schematic</figcaption>
</figure>

**Specification:**

Create a `CostBasedOptimizer` class that inspects `Table` catalog
statistics (row count, min/max values, distinct value counts). Given a
query predicate, estimate its selectivity, compute operational costs,
and select between `IndexScan` and `SequentialScan`.

``` cpp
struct TableStatistics {
    size_t total_rows{0};
    std::unordered_map<std::string, size_t> distinct_counts;
    std::unordered_map<std::string, std::pair<double, double>> numeric_bounds;
};

class CostBasedOptimizer {
public:
    explicit CostBasedOptimizer(TableStatistics stats);
    double estimate_selectivity(const Predicate& pred) const;
    bool should_use_index(const Predicate& pred, bool index_exists) const;
private:
    TableStatistics m_stats;
    static constexpr double CPU_TUPLE_COST = 0.01;
    static constexpr double RANDOM_FETCH_COST = 0.1;
};
```

**Verification Tasks:**

1.  **Unique Primary Key Selectivity:** Given $|R| = 1,000,000$ and
    distinct count $V(\text{id}) = 1,000,000$, estimate selectivity for
    $\text{id} = 42$. Verify that $S = 10^{-6}$ and that the optimiser
    selects an Index Scan.
2.  **Low-Cardinality Scan Selection:** Given $V(\text{gender}) = 2$ on
    $|R| = 500,000$, evaluate $\text{gender} = \text{'F'}$. Verify that
    $S \approx 0.5$ and that the optimiser falls back to a sequential
    table scan even if an index is present.
3.  **Numeric Range Estimation:** On an attribute with bounds
    $[0.0, 1000.0]$, estimate selectivity for $A > 800.0$. Verify that
    calculated selectivity is $0.20 \pm 0.01$.

#### Exercise 16.8: Composite Multi-Column Indexing

**Context and Theory:**

Real-world queries frequently filter across multiple attributes
simultaneously:

``` sql
SELECT * FROM Engineers WHERE dept = 'Systems' AND salary >= 9000.00;
```

Single-column indices force the engine to scan one index and evaluate
the remaining predicates sequentially in memory. A **Composite Index**
indexes a tuple of attributes $(A_1, A_2, \dots, A_m)$ using
**lexicographical ordering**:

$$(u_1, u_2) < (v_1, v_2) \iff (u_1 < v_1) \lor (u_1 = v_1 \land u_2 < v_2)$$

Composite indices obey the **Left-Prefix Rule**: an index over
$(A, B, C)$ accelerates queries filtering on $(A)$, $(A, B)$, and
$(A, B, C)$, but cannot accelerate queries filtering solely on $(B)$ or
$(C)$ without scanning the entire index (see Figure 16–12).

<figure id="fig:left-prefix">
<img src="data:image/svg+xml;base64,PHN2ZyB4bWxucz0iaHR0cDovL3d3dy53My5vcmcvMjAwMC9zdmciIHdpZHRoPSI2MzkuNjAwMDQi
IGhlaWdodD0iNjYuNCIgY2xhc3M9InN2Z2JvYiIgdmlld0JveD0iMCAtNCA2MzkuNjAwMDQgNjYu
NCI+CiAgPHN0eWxlPi5zdmdib2IgbGluZSwgLnN2Z2JvYiBwYXRoLCAuc3ZnYm9iIGNpcmNsZSwg
LnN2Z2JvYiByZWN0LCAuc3ZnYm9iIHBvbHlnb24gewogIHN0cm9rZTogYmxhY2s7CiAgc3Ryb2tl
LXdpZHRoOiAxLjU7CiAgc3Ryb2tlLW9wYWNpdHk6IDE7CiAgZmlsbC1vcGFjaXR5OiAxOwogIHN0
cm9rZS1saW5lY2FwOiByb3VuZDsKICBzdHJva2UtbGluZWpvaW46IG1pdGVyOwp9Cgouc3ZnYm9i
IHRleHQgewogIHdoaXRlLXNwYWNlOiBwcmU7CiAgZmlsbDogYmxhY2s7CiAgZm9udC1mYW1pbHk6
IENvdXJpZXIgTmV3LCBEZWphVnUgU2FucyBNb25vLCBtb25vc3BhY2U7CiAgZm9udC1zaXplOiAx
M3B4Owp9Cgouc3ZnYm9iIHJlY3QuYmFja2Ryb3AgewogIHN0cm9rZTogbm9uZTsKICBmaWxsOiB0
cmFuc3BhcmVudDsKfQoKLnN2Z2JvYiAuYnJva2VuIHsKICBzdHJva2UtZGFzaGFycmF5OiA4Owp9
Cgouc3ZnYm9iIC5maWxsZWQgewogIGZpbGw6IGJsYWNrOwp9Cgouc3ZnYm9iIC5iZ19maWxsZWQg
ewogIGZpbGw6IHRyYW5zcGFyZW50OwogIHN0cm9rZS13aWR0aDogMTsKfQoKLnN2Z2JvYiAubm9m
aWxsIHsKICBmaWxsOiB0cmFuc3BhcmVudDsKfQoKLnN2Z2JvYiAuZW5kX21hcmtlZF9hcnJvdyB7
CiAgbWFya2VyLWVuZDogdXJsKCNhcnJvdyk7Cn0KCi5zdmdib2IgLnN0YXJ0X21hcmtlZF9hcnJv
dyB7CiAgbWFya2VyLXN0YXJ0OiB1cmwoI2Fycm93KTsKfQoKLnN2Z2JvYiAuZW5kX21hcmtlZF9k
aWFtb25kIHsKICBtYXJrZXItZW5kOiB1cmwoI2RpYW1vbmQpOwp9Cgouc3ZnYm9iIC5zdGFydF9t
YXJrZWRfZGlhbW9uZCB7CiAgbWFya2VyLXN0YXJ0OiB1cmwoI2RpYW1vbmQpOwp9Cgouc3ZnYm9i
IC5lbmRfbWFya2VkX2NpcmNsZSB7CiAgbWFya2VyLWVuZDogdXJsKCNjaXJjbGUpOwp9Cgouc3Zn
Ym9iIC5zdGFydF9tYXJrZWRfY2lyY2xlIHsKICBtYXJrZXItc3RhcnQ6IHVybCgjY2lyY2xlKTsK
fQoKLnN2Z2JvYiAuZW5kX21hcmtlZF9vcGVuX2NpcmNsZSB7CiAgbWFya2VyLWVuZDogdXJsKCNv
cGVuX2NpcmNsZSk7Cn0KCi5zdmdib2IgLnN0YXJ0X21hcmtlZF9vcGVuX2NpcmNsZSB7CiAgbWFy
a2VyLXN0YXJ0OiB1cmwoI29wZW5fY2lyY2xlKTsKfQoKLnN2Z2JvYiAuZW5kX21hcmtlZF9iaWdf
b3Blbl9jaXJjbGUgewogIG1hcmtlci1lbmQ6IHVybCgjYmlnX29wZW5fY2lyY2xlKTsKfQoKLnN2
Z2JvYiAuc3RhcnRfbWFya2VkX2JpZ19vcGVuX2NpcmNsZSB7CiAgbWFya2VyLXN0YXJ0OiB1cmwo
I2JpZ19vcGVuX2NpcmNsZSk7Cn0KCjwvc3R5bGU+CiAgPGRlZnM+CiAgICA8bWFya2VyIGlkPSJh
cnJvdyIgdmlld0JveD0iLTIgLTIgOCA4IiByZWZYPSI0IiByZWZZPSIyIiBtYXJrZXJXaWR0aD0i
NyIgbWFya2VySGVpZ2h0PSI3IiBvcmllbnQ9ImF1dG8tc3RhcnQtcmV2ZXJzZSI+CiAgICAgIDxw
b2x5Z29uIHBvaW50cz0iMCwwIDAsNCA0LDIgMCwwIj48L3BvbHlnb24+CiAgICA8L21hcmtlcj4K
ICAgIDxtYXJrZXIgaWQ9ImRpYW1vbmQiIHZpZXdCb3g9Ii0yIC0yIDggOCIgcmVmWD0iNCIgcmVm
WT0iMiIgbWFya2VyV2lkdGg9IjciIG1hcmtlckhlaWdodD0iNyIgb3JpZW50PSJhdXRvLXN0YXJ0
LXJldmVyc2UiPgogICAgICA8cG9seWdvbiBwb2ludHM9IjAsMiAyLDAgNCwyIDIsNCAwLDIiPjwv
cG9seWdvbj4KICAgIDwvbWFya2VyPgogICAgPG1hcmtlciBpZD0iY2lyY2xlIiB2aWV3Qm94PSIw
IDAgOCA4IiByZWZYPSI0IiByZWZZPSI0IiBtYXJrZXJXaWR0aD0iNyIgbWFya2VySGVpZ2h0PSI3
IiBvcmllbnQ9ImF1dG8tc3RhcnQtcmV2ZXJzZSI+CiAgICAgIDxjaXJjbGUgY3g9IjQiIGN5PSI0
IiByPSIyIiBjbGFzcz0iZmlsbGVkIj48L2NpcmNsZT4KICAgIDwvbWFya2VyPgogICAgPG1hcmtl
ciBpZD0ib3Blbl9jaXJjbGUiIHZpZXdCb3g9IjAgMCA4IDgiIHJlZlg9IjQiIHJlZlk9IjQiIG1h
cmtlcldpZHRoPSI3IiBtYXJrZXJIZWlnaHQ9IjciIG9yaWVudD0iYXV0by1zdGFydC1yZXZlcnNl
Ij4KICAgICAgPGNpcmNsZSBjeD0iNCIgY3k9IjQiIHI9IjIiIGNsYXNzPSJiZ19maWxsZWQiPjwv
Y2lyY2xlPgogICAgPC9tYXJrZXI+CiAgICA8bWFya2VyIGlkPSJiaWdfb3Blbl9jaXJjbGUiIHZp
ZXdCb3g9IjAgMCA4IDgiIHJlZlg9IjQiIHJlZlk9IjQiIG1hcmtlcldpZHRoPSI3IiBtYXJrZXJI
ZWlnaHQ9IjciIG9yaWVudD0iYXV0by1zdGFydC1yZXZlcnNlIj4KICAgICAgPGNpcmNsZSBjeD0i
NCIgY3k9IjQiIHI9IjMiIGNsYXNzPSJiZ19maWxsZWQiPjwvY2lyY2xlPgogICAgPC9tYXJrZXI+
CiAgPC9kZWZzPgogIDxyZWN0IGNsYXNzPSJiYWNrZHJvcCIgeD0iMCIgeT0iLTQiIHdpZHRoPSI2
MzkuNjAwMDQiIGhlaWdodD0iNjYuNCI+PC9yZWN0PgogIDx0ZXh0IHhtbDpzcGFjZT0icHJlc2Vy
dmUiIHg9IjE3My41NSIgeT0iNDIuOSIgPlByZWZpeDwvdGV4dD4KICA8dGV4dCB4bWw6c3BhY2U9
InByZXNlcnZlIiB4PSIyMjguMTUwMDEiIHk9IjQyLjkiID5tYXRjaDwvdGV4dD4KICA8dGV4dCB4
bWw6c3BhY2U9InByZXNlcnZlIiB4PSIzNDUuMTUiIHk9IjQyLjkiID5QcmVmaXg8L3RleHQ+CiAg
PHRleHQgeG1sOnNwYWNlPSJwcmVzZXJ2ZSIgeD0iMzk5Ljc1IiB5PSI0Mi45IiA+bWF0Y2g8L3Rl
eHQ+CiAgPHRleHQgeG1sOnNwYWNlPSJwcmVzZXJ2ZSIgeD0iNTE2Ljc1IiB5PSI0Mi45IiA+TWlz
bWF0Y2g8L3RleHQ+CiAgPHRleHQgeG1sOnNwYWNlPSJwcmVzZXJ2ZSIgeD0iOS43NSIgeT0iMTEu
NzAwMDAxIiA+Q29tcG9zaXRlIEtleSBUdXBsZTogKCYjMzk7U3lzdGVtcyYjMzk7LCA4NTAwLjAp
ICZsdDsgKCYjMzk7U3lzdGVtcyYjMzk7LCA5MjAwLjApICZsdDsgKCYjMzk7V2ViJiMzOTssIDQw
MDAuMCk8L3RleHQ+CiAgPGc+CiAgICA8bGluZSB4MT0iMTg3LjIwMDAxIiB5MT0iMTUuNiIgeDI9
IjE5NSIgeTI9IjMxLjIiIGNsYXNzPSJzb2xpZCI+PC9saW5lPgogICAgPGxpbmUgeDE9IjE5NSIg
eTE9IjMxLjIiIHgyPSIyNDEuOCIgeTI9IjMxLjIiIGNsYXNzPSJzb2xpZCI+PC9saW5lPgogICAg
PGxpbmUgeDE9IjI0OS42IiB5MT0iMTUuNiIgeDI9IjI0MS44IiB5Mj0iMzEuMiIgY2xhc3M9InNv
bGlkIj48L2xpbmU+CiAgPC9nPgogIDxnPgogICAgPGxpbmUgeDE9IjM1OC44MDAwMiIgeTE9IjE1
LjYiIHgyPSIzNjYuNiIgeTI9IjMxLjIiIGNsYXNzPSJzb2xpZCI+PC9saW5lPgogICAgPGxpbmUg
eDE9IjM2Ni42IiB5MT0iMzEuMiIgeDI9IjQxMy40MDAwMiIgeTI9IjMxLjIiIGNsYXNzPSJzb2xp
ZCI+PC9saW5lPgogICAgPGxpbmUgeDE9IjQyMS4yIiB5MT0iMTUuNiIgeDI9IjQxMy40MDAwMiIg
eTI9IjMxLjIiIGNsYXNzPSJzb2xpZCI+PC9saW5lPgogIDwvZz4KICA8Zz4KICAgIDxsaW5lIHgx
PSI1MzAuNCIgeTE9IjE1LjYiIHgyPSI1MzguMiIgeTI9IjMxLjIiIGNsYXNzPSJzb2xpZCI+PC9s
aW5lPgogICAgPGxpbmUgeDE9IjUzOC4yIiB5MT0iMzEuMiIgeDI9IjU1My44IiB5Mj0iMzEuMiIg
Y2xhc3M9InNvbGlkIj48L2xpbmU+CiAgICA8bGluZSB4MT0iNTYxLjYwMDA0IiB5MT0iMTUuNiIg
eDI9IjU1My44IiB5Mj0iMzEuMiIgY2xhc3M9InNvbGlkIj48L2xpbmU+CiAgPC9nPgo8L3N2Zz4K
" alt="" />
<figcaption>Figure 16–12: Left-Prefix Matching</figcaption>
</figure>

**Specification:**

Implement a `CompositeKey` type containing `std::vector<Value>` with
lexicographical comparison operators. Construct a `CompositeBSTIndex`
class capable of mapping multi-attribute keys to `RowID` sets.

``` cpp
class CompositeKey {
public:
    CompositeKey(std::vector<Value> values) : m_values(std::move(values)) {}
    bool operator<(const CompositeKey& other) const {
        return std::lexicographical_compare(
            m_values.begin(), m_values.end(),
            other.m_values.begin(), other.m_values.end()
        );
    }
    bool operator==(const CompositeKey& other) const { return m_values == other.m_values; }
private:
    std::vector<Value> m_values;
};

class CompositeBSTIndex {
public:
    void insert(const CompositeKey& key, RowID id);
    std::vector<RowID> find_exact(const CompositeKey& key) const;
    std::vector<RowID> find_prefix_range(const CompositeKey& prefix_low, 
                                         const CompositeKey& prefix_high) const;
};
```

**Verification Tasks:**

1.  **Lexicographical Ordering Invariant:** Insert composite keys
    `("Beta", 2)`, `("Alpha", 10)`, and `("Alpha", 5)`. Verify that an
    in-order traversal yields `("Alpha", 5)`, `("Alpha", 10)`, then
    `("Beta", 2)`.
2.  **Point Lookup on Multi-Key:** Insert $5,000$ rows into
    `(dept, salary)`. Query for `("Systems", 8500.50)`. Verify that only
    matching row IDs are returned.
3.  **Left-Prefix Range Scan:** Query for all engineers in
    `dept = "Systems"` across any salary range by scanning with bounds
    `["Systems", -\infty]` to `["Systems", +\infty]`. Verify that all
    department employees are retrieved without scanning non-matching
    departments.

#### Exercise 16.9: Write-Ahead Logging (WAL) and Crash Recovery

**Context and Theory:**

To satisfy the **Durability ($D$)** invariant of ACID without writing
entire memory pages to disk synchronously on every transaction commit,
databases use **Write-Ahead Logging (WAL)** governed by two invariants:

1.  **The WAL Invariant:** Dirty data pages in memory must never be
    flushed to non-volatile storage until the corresponding log records
    detailing the mutation have been flushed to disk.
2.  **Commit Durability:** A transaction is not committed until its
    `COMMIT` log entry is guaranteed to be persistent on non-volatile
    media.

Every log record is assigned a strictly increasing **Log Sequence Number
(LSN)** (as shown in Figure 16–13). During recovery following a
catastrophic power loss, the engine executes an ARIES-style recovery
pass:

- **REDO Pass:** Replays all logged modifications forward from the last
  checkpoint to restore the buffer state.
- **UNDO Pass:** Scans backward to reverse the modifications of active
  transactions that never committed before the crash.

<figure id="fig:wal-strategy">
<img src="data:image/svg+xml;base64,PHN2ZyB4bWxucz0iaHR0cDovL3d3dy53My5vcmcvMjAwMC9zdmciIHdpZHRoPSI1MzguMiIgaGVp
Z2h0PSI2Ni40IiBjbGFzcz0ic3ZnYm9iIiB2aWV3Qm94PSIwIC00IDUzOC4yIDY2LjQiPgogIDxz
dHlsZT4uc3ZnYm9iIGxpbmUsIC5zdmdib2IgcGF0aCwgLnN2Z2JvYiBjaXJjbGUsIC5zdmdib2Ig
cmVjdCwgLnN2Z2JvYiBwb2x5Z29uIHsKICBzdHJva2U6IGJsYWNrOwogIHN0cm9rZS13aWR0aDog
MS41OwogIHN0cm9rZS1vcGFjaXR5OiAxOwogIGZpbGwtb3BhY2l0eTogMTsKICBzdHJva2UtbGlu
ZWNhcDogcm91bmQ7CiAgc3Ryb2tlLWxpbmVqb2luOiBtaXRlcjsKfQoKLnN2Z2JvYiB0ZXh0IHsK
ICB3aGl0ZS1zcGFjZTogcHJlOwogIGZpbGw6IGJsYWNrOwogIGZvbnQtZmFtaWx5OiBDb3VyaWVy
IE5ldywgRGVqYVZ1IFNhbnMgTW9ubywgbW9ub3NwYWNlOwogIGZvbnQtc2l6ZTogMTNweDsKfQoK
LnN2Z2JvYiByZWN0LmJhY2tkcm9wIHsKICBzdHJva2U6IG5vbmU7CiAgZmlsbDogdHJhbnNwYXJl
bnQ7Cn0KCi5zdmdib2IgLmJyb2tlbiB7CiAgc3Ryb2tlLWRhc2hhcnJheTogODsKfQoKLnN2Z2Jv
YiAuZmlsbGVkIHsKICBmaWxsOiBibGFjazsKfQoKLnN2Z2JvYiAuYmdfZmlsbGVkIHsKICBmaWxs
OiB0cmFuc3BhcmVudDsKICBzdHJva2Utd2lkdGg6IDE7Cn0KCi5zdmdib2IgLm5vZmlsbCB7CiAg
ZmlsbDogdHJhbnNwYXJlbnQ7Cn0KCi5zdmdib2IgLmVuZF9tYXJrZWRfYXJyb3cgewogIG1hcmtl
ci1lbmQ6IHVybCgjYXJyb3cpOwp9Cgouc3ZnYm9iIC5zdGFydF9tYXJrZWRfYXJyb3cgewogIG1h
cmtlci1zdGFydDogdXJsKCNhcnJvdyk7Cn0KCi5zdmdib2IgLmVuZF9tYXJrZWRfZGlhbW9uZCB7
CiAgbWFya2VyLWVuZDogdXJsKCNkaWFtb25kKTsKfQoKLnN2Z2JvYiAuc3RhcnRfbWFya2VkX2Rp
YW1vbmQgewogIG1hcmtlci1zdGFydDogdXJsKCNkaWFtb25kKTsKfQoKLnN2Z2JvYiAuZW5kX21h
cmtlZF9jaXJjbGUgewogIG1hcmtlci1lbmQ6IHVybCgjY2lyY2xlKTsKfQoKLnN2Z2JvYiAuc3Rh
cnRfbWFya2VkX2NpcmNsZSB7CiAgbWFya2VyLXN0YXJ0OiB1cmwoI2NpcmNsZSk7Cn0KCi5zdmdi
b2IgLmVuZF9tYXJrZWRfb3Blbl9jaXJjbGUgewogIG1hcmtlci1lbmQ6IHVybCgjb3Blbl9jaXJj
bGUpOwp9Cgouc3ZnYm9iIC5zdGFydF9tYXJrZWRfb3Blbl9jaXJjbGUgewogIG1hcmtlci1zdGFy
dDogdXJsKCNvcGVuX2NpcmNsZSk7Cn0KCi5zdmdib2IgLmVuZF9tYXJrZWRfYmlnX29wZW5fY2ly
Y2xlIHsKICBtYXJrZXItZW5kOiB1cmwoI2JpZ19vcGVuX2NpcmNsZSk7Cn0KCi5zdmdib2IgLnN0
YXJ0X21hcmtlZF9iaWdfb3Blbl9jaXJjbGUgewogIG1hcmtlci1zdGFydDogdXJsKCNiaWdfb3Bl
bl9jaXJjbGUpOwp9Cgo8L3N0eWxlPgogIDxkZWZzPgogICAgPG1hcmtlciBpZD0iYXJyb3ciIHZp
ZXdCb3g9Ii0yIC0yIDggOCIgcmVmWD0iNCIgcmVmWT0iMiIgbWFya2VyV2lkdGg9IjciIG1hcmtl
ckhlaWdodD0iNyIgb3JpZW50PSJhdXRvLXN0YXJ0LXJldmVyc2UiPgogICAgICA8cG9seWdvbiBw
b2ludHM9IjAsMCAwLDQgNCwyIDAsMCI+PC9wb2x5Z29uPgogICAgPC9tYXJrZXI+CiAgICA8bWFy
a2VyIGlkPSJkaWFtb25kIiB2aWV3Qm94PSItMiAtMiA4IDgiIHJlZlg9IjQiIHJlZlk9IjIiIG1h
cmtlcldpZHRoPSI3IiBtYXJrZXJIZWlnaHQ9IjciIG9yaWVudD0iYXV0by1zdGFydC1yZXZlcnNl
Ij4KICAgICAgPHBvbHlnb24gcG9pbnRzPSIwLDIgMiwwIDQsMiAyLDQgMCwyIj48L3BvbHlnb24+
CiAgICA8L21hcmtlcj4KICAgIDxtYXJrZXIgaWQ9ImNpcmNsZSIgdmlld0JveD0iMCAwIDggOCIg
cmVmWD0iNCIgcmVmWT0iNCIgbWFya2VyV2lkdGg9IjciIG1hcmtlckhlaWdodD0iNyIgb3JpZW50
PSJhdXRvLXN0YXJ0LXJldmVyc2UiPgogICAgICA8Y2lyY2xlIGN4PSI0IiBjeT0iNCIgcj0iMiIg
Y2xhc3M9ImZpbGxlZCI+PC9jaXJjbGU+CiAgICA8L21hcmtlcj4KICAgIDxtYXJrZXIgaWQ9Im9w
ZW5fY2lyY2xlIiB2aWV3Qm94PSIwIDAgOCA4IiByZWZYPSI0IiByZWZZPSI0IiBtYXJrZXJXaWR0
aD0iNyIgbWFya2VySGVpZ2h0PSI3IiBvcmllbnQ9ImF1dG8tc3RhcnQtcmV2ZXJzZSI+CiAgICAg
IDxjaXJjbGUgY3g9IjQiIGN5PSI0IiByPSIyIiBjbGFzcz0iYmdfZmlsbGVkIj48L2NpcmNsZT4K
ICAgIDwvbWFya2VyPgogICAgPG1hcmtlciBpZD0iYmlnX29wZW5fY2lyY2xlIiB2aWV3Qm94PSIw
IDAgOCA4IiByZWZYPSI0IiByZWZZPSI0IiBtYXJrZXJXaWR0aD0iNyIgbWFya2VySGVpZ2h0PSI3
IiBvcmllbnQ9ImF1dG8tc3RhcnQtcmV2ZXJzZSI+CiAgICAgIDxjaXJjbGUgY3g9IjQiIGN5PSI0
IiByPSIzIiBjbGFzcz0iYmdfZmlsbGVkIj48L2NpcmNsZT4KICAgIDwvbWFya2VyPgogIDwvZGVm
cz4KICA8cmVjdCBjbGFzcz0iYmFja2Ryb3AiIHg9IjAiIHk9Ii00IiB3aWR0aD0iNTM4LjIiIGhl
aWdodD0iNjYuNCI+PC9yZWN0PgogIDx0ZXh0IHhtbDpzcGFjZT0icHJlc2VydmUiIHg9IjEuOTUi
IHk9IjExLjcwMDAwMSIgPltMU048L3RleHQ+CiAgPHRleHQgeG1sOnNwYWNlPSJwcmVzZXJ2ZSIg
eD0iNDAuOTUiIHk9IjExLjcwMDAwMSIgPjE6PC90ZXh0PgogIDx0ZXh0IHhtbDpzcGFjZT0icHJl
c2VydmUiIHg9IjY0LjM1IiB5PSIxMS43MDAwMDEiID5UWDE8L3RleHQ+CiAgPHRleHQgeG1sOnNw
YWNlPSJwcmVzZXJ2ZSIgeD0iOTUuNTUiIHk9IjExLjcwMDAwMSIgPlNUQVJUXTwvdGV4dD4KICA8
bGluZSB4MT0iMTQ4LjIiIHkxPSI3LjgiIHgyPSIxNjMuOCIgeTI9IjcuOCIgY2xhc3M9InNvbGlk
Ij48L2xpbmU+CiAgPHBvbHlnb24gcG9pbnRzPSIxNjMuOCwzLjkgMTcxLjYsNy44IDE2My44LDEx
LjcwMDAwMSIgY2xhc3M9ImZpbGxlZCI+PC9wb2x5Z29uPgogIDx0ZXh0IHhtbDpzcGFjZT0icHJl
c2VydmUiIHg9IjE4MS4zNSIgeT0iMTEuNzAwMDAxIiA+W0xTTjwvdGV4dD4KICA8dGV4dCB4bWw6
c3BhY2U9InByZXNlcnZlIiB4PSIyMjAuMzUiIHk9IjExLjcwMDAwMSIgPjI6PC90ZXh0PgogIDx0
ZXh0IHhtbDpzcGFjZT0icHJlc2VydmUiIHg9IjI0My43NSIgeT0iMTEuNzAwMDAxIiA+VFgxPC90
ZXh0PgogIDx0ZXh0IHhtbDpzcGFjZT0icHJlc2VydmUiIHg9IjI3NC45NSIgeT0iMTEuNzAwMDAx
IiA+SU5TRVJUPC90ZXh0PgogIDx0ZXh0IHhtbDpzcGFjZT0icHJlc2VydmUiIHg9IjMyOS41NTAw
MiIgeT0iMTEuNzAwMDAxIiA+Um93IzFdPC90ZXh0PgogIDxsaW5lIHgxPSIzODIuMiIgeTE9Ijcu
OCIgeDI9IjM5Ny44MDAwMiIgeTI9IjcuOCIgY2xhc3M9InNvbGlkIj48L2xpbmU+CiAgPHBvbHln
b24gcG9pbnRzPSIzOTcuODAwMDIsMy45IDQwNS42LDcuOCAzOTcuODAwMDIsMTEuNzAwMDAxIiBj
bGFzcz0iZmlsbGVkIj48L3BvbHlnb24+CiAgPHRleHQgeG1sOnNwYWNlPSJwcmVzZXJ2ZSIgeD0i
NDE1LjM1IiB5PSIxMS43MDAwMDEiID5bTFNOPC90ZXh0PgogIDx0ZXh0IHhtbDpzcGFjZT0icHJl
c2VydmUiIHg9IjQ1NC4zNSIgeT0iMTEuNzAwMDAxIiA+Mzo8L3RleHQ+CiAgPHRleHQgeG1sOnNw
YWNlPSJwcmVzZXJ2ZSIgeD0iNDc3Ljc1IiB5PSIxMS43MDAwMDEiID5DUkFTSCFdPC90ZXh0Pgog
IDxwb2x5Z29uIHBvaW50cz0iNDI5LDM1LjEwMDAwMiA0MjEuMiwzOSA0MjksNDIuOSIgY2xhc3M9
ImZpbGxlZCI+PC9wb2x5Z29uPgogIDx0ZXh0IHhtbDpzcGFjZT0icHJlc2VydmUiIHg9IjEuOTUi
IHk9IjQyLjkiID5SZWNvdmVyeTwvdGV4dD4KICA8dGV4dCB4bWw6c3BhY2U9InByZXNlcnZlIiB4
PSI3Mi4xNSIgeT0iNDIuOSIgPlN0cmF0ZWd5OjwvdGV4dD4KICA8dGV4dCB4bWw6c3BhY2U9InBy
ZXNlcnZlIiB4PSIxNTAuMTUwMDEiIHk9IjQyLjkiID5VbmRvPC90ZXh0PgogIDx0ZXh0IHhtbDpz
cGFjZT0icHJlc2VydmUiIHg9IjE4OS4xNTAwMSIgeT0iNDIuOSIgPlRYMTwvdGV4dD4KICA8dGV4
dCB4bWw6c3BhY2U9InByZXNlcnZlIiB4PSIyMjAuMzUiIHk9IjQyLjkiID5tdXRhdGlvbjwvdGV4
dD4KICA8dGV4dCB4bWw6c3BhY2U9InByZXNlcnZlIiB4PSIyODIuNzUiIHk9IjQyLjkiID4oUm9s
bGJhY2sgUm93IzEpPC90ZXh0PgogIDxnPgogICAgPGxpbmUgeDE9IjQ3MS45MDAwMiIgeTE9IjE1
LjYiIHgyPSI0NzEuOTAwMDIiIHkyPSIzOSIgY2xhc3M9InNvbGlkIj48L2xpbmU+CiAgICA8bGlu
ZSB4MT0iNDI5IiB5MT0iMzkiIHgyPSI0NzEuOTAwMDIiIHkyPSIzOSIgY2xhc3M9InNvbGlkIj48
L2xpbmU+CiAgPC9nPgo8L3N2Zz4K
" alt="" />
<figcaption>Figure 16–13: Write-Ahead Log Stream</figcaption>
</figure>

**Specification:**

Construct a `WALManager` class that writes binary mutation records to an
append-only stream. Implement serialization and deserialization routines
alongside a `recover()` function that inspects the log, commits
completed transactions, and rolls back uncommitted dirty rows.

``` cpp
enum class LogType : uint8_t { Begin, Insert, Commit, Rollback };

struct LogRecord {
    uint64_t lsn;
    uint32_t tx_id;
    LogType type;
    RowID target_row;
    Row payload;
};

class WALManager {
public:
    explicit WALManager(std::iostream& log_stream);
    uint64_t append_record(uint32_t tx_id, LogType type, RowID row_id, Row data);
    void flush();
    void recover(Table& target_table);
private:
    std::iostream& m_stream;
    uint64_t m_current_lsn{1};
};
```

**Verification Tasks:**

1.  **Redo Committed Transaction:** Write an `Insert` record followed by
    a `Commit` record to the WAL stream. Initialise an empty `Table` and
    execute `recover()`. Verify that the inserted row exists in the
    restored table.
2.  **Undo Uncommitted Transaction:** Write an `Insert` record for a
    transaction that ends abruptly without a matching `Commit` record
    (simulating power failure). Run `recover()`. Verify that the
    uncommitted row is pruned from active storage.
3.  **LSN Monotonicity:** Generate $1,000$ concurrent WAL writes. Verify
    that every assigned LSN is strictly monotonically increasing
    ($LSN_i < LSN_{i+1}$) and that deserialised binary frames pass
    payload verification.

#### Exercise 16.10: Multi-Version Concurrency Control (MVCC) Engine

**Context and Theory:**

Lock-based concurrency control (such as Two-Phase Locking, 2PL) forces
readers to block writers and writers to block readers, degrading
throughput. **Multi-Version Concurrency Control (MVCC)** guarantees that
**readers never block writers, and writers never block readers**.

Under MVCC, mutating a row does not overwrite its physical memory.
Instead, the storage layer maintains multiple versions of each tuple
decorated with visibility metadata (as in Figure 16–14):

- **$x_{\min}$:** The Transaction ID that created/inserted this version.
- **$x_{\max}$:** The Transaction ID that deleted or superseded this
  version (or $\infty$ / active sentinel if valid).

A transaction with ID $T_{\text{read}}$ can view a tuple version if and
only if:

$$x_{\min} \le T_{\text{read}} \quad \land \quad (x_{\max} > T_{\text{read}} \lor x_{\max} = \text{unassigned})$$

<figure id="fig:tuple-chain">
<img src="data:image/svg+xml;base64,PHN2ZyB4bWxucz0iaHR0cDovL3d3dy53My5vcmcvMjAwMC9zdmciIHdpZHRoPSI1NDYiIGhlaWdo
dD0iNTAuODAwMDAzIiBjbGFzcz0ic3ZnYm9iIiB2aWV3Qm94PSIwIC00IDU0NiA1MC44MDAwMDMi
PgogIDxzdHlsZT4uc3ZnYm9iIGxpbmUsIC5zdmdib2IgcGF0aCwgLnN2Z2JvYiBjaXJjbGUsIC5z
dmdib2IgcmVjdCwgLnN2Z2JvYiBwb2x5Z29uIHsKICBzdHJva2U6IGJsYWNrOwogIHN0cm9rZS13
aWR0aDogMS41OwogIHN0cm9rZS1vcGFjaXR5OiAxOwogIGZpbGwtb3BhY2l0eTogMTsKICBzdHJv
a2UtbGluZWNhcDogcm91bmQ7CiAgc3Ryb2tlLWxpbmVqb2luOiBtaXRlcjsKfQoKLnN2Z2JvYiB0
ZXh0IHsKICB3aGl0ZS1zcGFjZTogcHJlOwogIGZpbGw6IGJsYWNrOwogIGZvbnQtZmFtaWx5OiBD
b3VyaWVyIE5ldywgRGVqYVZ1IFNhbnMgTW9ubywgbW9ub3NwYWNlOwogIGZvbnQtc2l6ZTogMTNw
eDsKfQoKLnN2Z2JvYiByZWN0LmJhY2tkcm9wIHsKICBzdHJva2U6IG5vbmU7CiAgZmlsbDogdHJh
bnNwYXJlbnQ7Cn0KCi5zdmdib2IgLmJyb2tlbiB7CiAgc3Ryb2tlLWRhc2hhcnJheTogODsKfQoK
LnN2Z2JvYiAuZmlsbGVkIHsKICBmaWxsOiBibGFjazsKfQoKLnN2Z2JvYiAuYmdfZmlsbGVkIHsK
ICBmaWxsOiB0cmFuc3BhcmVudDsKICBzdHJva2Utd2lkdGg6IDE7Cn0KCi5zdmdib2IgLm5vZmls
bCB7CiAgZmlsbDogdHJhbnNwYXJlbnQ7Cn0KCi5zdmdib2IgLmVuZF9tYXJrZWRfYXJyb3cgewog
IG1hcmtlci1lbmQ6IHVybCgjYXJyb3cpOwp9Cgouc3ZnYm9iIC5zdGFydF9tYXJrZWRfYXJyb3cg
ewogIG1hcmtlci1zdGFydDogdXJsKCNhcnJvdyk7Cn0KCi5zdmdib2IgLmVuZF9tYXJrZWRfZGlh
bW9uZCB7CiAgbWFya2VyLWVuZDogdXJsKCNkaWFtb25kKTsKfQoKLnN2Z2JvYiAuc3RhcnRfbWFy
a2VkX2RpYW1vbmQgewogIG1hcmtlci1zdGFydDogdXJsKCNkaWFtb25kKTsKfQoKLnN2Z2JvYiAu
ZW5kX21hcmtlZF9jaXJjbGUgewogIG1hcmtlci1lbmQ6IHVybCgjY2lyY2xlKTsKfQoKLnN2Z2Jv
YiAuc3RhcnRfbWFya2VkX2NpcmNsZSB7CiAgbWFya2VyLXN0YXJ0OiB1cmwoI2NpcmNsZSk7Cn0K
Ci5zdmdib2IgLmVuZF9tYXJrZWRfb3Blbl9jaXJjbGUgewogIG1hcmtlci1lbmQ6IHVybCgjb3Bl
bl9jaXJjbGUpOwp9Cgouc3ZnYm9iIC5zdGFydF9tYXJrZWRfb3Blbl9jaXJjbGUgewogIG1hcmtl
ci1zdGFydDogdXJsKCNvcGVuX2NpcmNsZSk7Cn0KCi5zdmdib2IgLmVuZF9tYXJrZWRfYmlnX29w
ZW5fY2lyY2xlIHsKICBtYXJrZXItZW5kOiB1cmwoI2JpZ19vcGVuX2NpcmNsZSk7Cn0KCi5zdmdi
b2IgLnN0YXJ0X21hcmtlZF9iaWdfb3Blbl9jaXJjbGUgewogIG1hcmtlci1zdGFydDogdXJsKCNi
aWdfb3Blbl9jaXJjbGUpOwp9Cgo8L3N0eWxlPgogIDxkZWZzPgogICAgPG1hcmtlciBpZD0iYXJy
b3ciIHZpZXdCb3g9Ii0yIC0yIDggOCIgcmVmWD0iNCIgcmVmWT0iMiIgbWFya2VyV2lkdGg9Ijci
IG1hcmtlckhlaWdodD0iNyIgb3JpZW50PSJhdXRvLXN0YXJ0LXJldmVyc2UiPgogICAgICA8cG9s
eWdvbiBwb2ludHM9IjAsMCAwLDQgNCwyIDAsMCI+PC9wb2x5Z29uPgogICAgPC9tYXJrZXI+CiAg
ICA8bWFya2VyIGlkPSJkaWFtb25kIiB2aWV3Qm94PSItMiAtMiA4IDgiIHJlZlg9IjQiIHJlZlk9
IjIiIG1hcmtlcldpZHRoPSI3IiBtYXJrZXJIZWlnaHQ9IjciIG9yaWVudD0iYXV0by1zdGFydC1y
ZXZlcnNlIj4KICAgICAgPHBvbHlnb24gcG9pbnRzPSIwLDIgMiwwIDQsMiAyLDQgMCwyIj48L3Bv
bHlnb24+CiAgICA8L21hcmtlcj4KICAgIDxtYXJrZXIgaWQ9ImNpcmNsZSIgdmlld0JveD0iMCAw
IDggOCIgcmVmWD0iNCIgcmVmWT0iNCIgbWFya2VyV2lkdGg9IjciIG1hcmtlckhlaWdodD0iNyIg
b3JpZW50PSJhdXRvLXN0YXJ0LXJldmVyc2UiPgogICAgICA8Y2lyY2xlIGN4PSI0IiBjeT0iNCIg
cj0iMiIgY2xhc3M9ImZpbGxlZCI+PC9jaXJjbGU+CiAgICA8L21hcmtlcj4KICAgIDxtYXJrZXIg
aWQ9Im9wZW5fY2lyY2xlIiB2aWV3Qm94PSIwIDAgOCA4IiByZWZYPSI0IiByZWZZPSI0IiBtYXJr
ZXJXaWR0aD0iNyIgbWFya2VySGVpZ2h0PSI3IiBvcmllbnQ9ImF1dG8tc3RhcnQtcmV2ZXJzZSI+
CiAgICAgIDxjaXJjbGUgY3g9IjQiIGN5PSI0IiByPSIyIiBjbGFzcz0iYmdfZmlsbGVkIj48L2Np
cmNsZT4KICAgIDwvbWFya2VyPgogICAgPG1hcmtlciBpZD0iYmlnX29wZW5fY2lyY2xlIiB2aWV3
Qm94PSIwIDAgOCA4IiByZWZYPSI0IiByZWZZPSI0IiBtYXJrZXJXaWR0aD0iNyIgbWFya2VySGVp
Z2h0PSI3IiBvcmllbnQ9ImF1dG8tc3RhcnQtcmV2ZXJzZSI+CiAgICAgIDxjaXJjbGUgY3g9IjQi
IGN5PSI0IiByPSIzIiBjbGFzcz0iYmdfZmlsbGVkIj48L2NpcmNsZT4KICAgIDwvbWFya2VyPgog
IDwvZGVmcz4KICA8cmVjdCBjbGFzcz0iYmFja2Ryb3AiIHg9IjAiIHk9Ii00IiB3aWR0aD0iNTQ2
IiBoZWlnaHQ9IjUwLjgwMDAwMyI+PC9yZWN0PgogIDxsaW5lIHgxPSIyODguNiIgeTE9IjcuOCIg
eDI9IjMwNC4yIiB5Mj0iNy44IiBjbGFzcz0ic29saWQiPjwvbGluZT4KICA8cG9seWdvbiBwb2lu
dHM9IjMwNC4yLDMuOSAzMTIsNy44IDMwNC4yLDExLjcwMDAwMSIgY2xhc3M9ImZpbGxlZCI+PC9w
b2x5Z29uPgogIDxsaW5lIHgxPSIyODguNiIgeTE9IjIzLjQwMDAwMiIgeDI9IjMwNC4yIiB5Mj0i
MjMuNDAwMDAyIiBjbGFzcz0ic29saWQiPjwvbGluZT4KICA8cG9seWdvbiBwb2ludHM9IjMwNC4y
LDE5LjUgMzEyLDIzLjQwMDAwMiAzMDQuMiwyNy4zMDAwMDEiIGNsYXNzPSJmaWxsZWQiPjwvcG9s
eWdvbj4KICA8dGV4dCB4bWw6c3BhY2U9InByZXNlcnZlIiB4PSIxLjk1IiB5PSIxMS43MDAwMDEi
ID5WZXJzaW9uIEE6IFsgeG1pbjogMTAwLCB4bWF4OiAxMDUgXTwvdGV4dD4KICA8dGV4dCB4bWw6
c3BhY2U9InByZXNlcnZlIiB4PSIzMjEuNzUiIHk9IjExLjcwMDAwMSIgPkRhdGE6ICYjMzk7QWRh
IExvdmVsYWNlJiMzOTs8L3RleHQ+CiAgPHRleHQgeG1sOnNwYWNlPSJwcmVzZXJ2ZSIgeD0iMS45
NSIgeT0iMjcuMzAwMDAxIiA+VmVyc2lvbiBCOiBbIHhtaW46IDEwNSwgeG1heDogaW5mIF08L3Rl
eHQ+CiAgPHRleHQgeG1sOnNwYWNlPSJwcmVzZXJ2ZSIgeD0iMzIxLjc1IiB5PSIyNy4zMDAwMDEi
ID5EYXRhOiAmIzM5O0NvdW50ZXNzIG9mIExvdmVsYWNlJiMzOTs8L3RleHQ+Cjwvc3ZnPgo=
" alt="" />
<figcaption>Figure 16–14: Tuple Version Chain for RowID #1</figcaption>
</figure>

**Specification:**

Modify `Row` to contain MVCC version headers ($x_{\min}$, $x_{\max}$).
Implement an `MVCCTable` that performs snapshot-isolated scans,
non-destructive updates (marking $x_{\max}$ on the old record and
appending a new record with $x_{\min} = T$), and row-level conflict
checks.

``` cpp
struct MVCCRowHeader {
    uint64_t xmin{0};
    uint64_t xmax{std::numeric_limits<uint64_t>::max()};
};

class MVCCTable {
public:
    RowID insert(Row row, uint64_t tx_id);
    bool update(RowID id, Row new_row, uint64_t tx_id);
    bool remove(RowID id, uint64_t tx_id);
    std::vector<Row> snapshot_scan(uint64_t tx_id) const;
private:
    std::vector<Row> m_data;
    std::vector<MVCCRowHeader> m_headers;
};
```

**Verification Tasks:**

1.  **Snapshot Isolation Visibility:** Transaction $T_{100}$ inserts a
    row. Transaction $T_{105}$ modifies that row. Verify that an active
    reader transaction operating at snapshot $T_{102}$ reads the
    original value, while a reader at $T_{106}$ reads the modified
    value.
2.  **Non-Repeatable Read Prevention:** Open transaction $T_1$.
    Concurrently insert $50$ rows via transaction $T_2$ and commit.
    Verify that rescanning inside $T_1$ yields the identical row count
    as its opening scan.
3.  **Write-Write Conflict Detection:** Attempt to modify the same
    active tuple concurrently from two transactions $T_a$ and $T_b$.
    Verify that the engine detects an active uncommitted writer
    ($x_{\max} \neq \infty$) and aborts the second transaction.

#### Exercise 16.11: Paged Buffer Pool Manager with CLOCK Eviction

**Context and Theory:**

Databases manipulate datasets that exceed available physical RAM. Memory
management is orchestrated by the **Buffer Pool Manager**, which caches
fixed-size frames (typically $4\text{ KB}$ or $8\text{ KB}$) mapped from
secondary storage.

When a page is requested:

1.  If present in the pool, its **Pin Count** is incremented, and an
    in-memory pointer is returned.
2.  If absent, an existing unpinned frame ($\text{Pin Count} = 0$) must
    be evicted. If that victim frame is **dirty** (modified), it must be
    flushed to disk before replacement.

Eviction is governed by the **CLOCK (Second-Chance)** algorithm: a
circular iterator checks frame reference bits (see Figure 16–15). If the
bit is $1$, it is cleared to $0$; if the bit is $0$ and unpinned, the
frame is selected as the eviction victim.

<figure id="fig:clock-algorithm">
<img src="data:image/svg+xml;base64,PHN2ZyB4bWxucz0iaHR0cDovL3d3dy53My5vcmcvMjAwMC9zdmciIHdpZHRoPSI1MjIuNjAwMDQi
IGhlaWdodD0iNTAuODAwMDAzIiBjbGFzcz0ic3ZnYm9iIiB2aWV3Qm94PSIwIC00IDUyMi42MDAw
NCA1MC44MDAwMDMiPgogIDxzdHlsZT4uc3ZnYm9iIGxpbmUsIC5zdmdib2IgcGF0aCwgLnN2Z2Jv
YiBjaXJjbGUsIC5zdmdib2IgcmVjdCwgLnN2Z2JvYiBwb2x5Z29uIHsKICBzdHJva2U6IGJsYWNr
OwogIHN0cm9rZS13aWR0aDogMS41OwogIHN0cm9rZS1vcGFjaXR5OiAxOwogIGZpbGwtb3BhY2l0
eTogMTsKICBzdHJva2UtbGluZWNhcDogcm91bmQ7CiAgc3Ryb2tlLWxpbmVqb2luOiBtaXRlcjsK
fQoKLnN2Z2JvYiB0ZXh0IHsKICB3aGl0ZS1zcGFjZTogcHJlOwogIGZpbGw6IGJsYWNrOwogIGZv
bnQtZmFtaWx5OiBDb3VyaWVyIE5ldywgRGVqYVZ1IFNhbnMgTW9ubywgbW9ub3NwYWNlOwogIGZv
bnQtc2l6ZTogMTNweDsKfQoKLnN2Z2JvYiByZWN0LmJhY2tkcm9wIHsKICBzdHJva2U6IG5vbmU7
CiAgZmlsbDogdHJhbnNwYXJlbnQ7Cn0KCi5zdmdib2IgLmJyb2tlbiB7CiAgc3Ryb2tlLWRhc2hh
cnJheTogODsKfQoKLnN2Z2JvYiAuZmlsbGVkIHsKICBmaWxsOiBibGFjazsKfQoKLnN2Z2JvYiAu
YmdfZmlsbGVkIHsKICBmaWxsOiB0cmFuc3BhcmVudDsKICBzdHJva2Utd2lkdGg6IDE7Cn0KCi5z
dmdib2IgLm5vZmlsbCB7CiAgZmlsbDogdHJhbnNwYXJlbnQ7Cn0KCi5zdmdib2IgLmVuZF9tYXJr
ZWRfYXJyb3cgewogIG1hcmtlci1lbmQ6IHVybCgjYXJyb3cpOwp9Cgouc3ZnYm9iIC5zdGFydF9t
YXJrZWRfYXJyb3cgewogIG1hcmtlci1zdGFydDogdXJsKCNhcnJvdyk7Cn0KCi5zdmdib2IgLmVu
ZF9tYXJrZWRfZGlhbW9uZCB7CiAgbWFya2VyLWVuZDogdXJsKCNkaWFtb25kKTsKfQoKLnN2Z2Jv
YiAuc3RhcnRfbWFya2VkX2RpYW1vbmQgewogIG1hcmtlci1zdGFydDogdXJsKCNkaWFtb25kKTsK
fQoKLnN2Z2JvYiAuZW5kX21hcmtlZF9jaXJjbGUgewogIG1hcmtlci1lbmQ6IHVybCgjY2lyY2xl
KTsKfQoKLnN2Z2JvYiAuc3RhcnRfbWFya2VkX2NpcmNsZSB7CiAgbWFya2VyLXN0YXJ0OiB1cmwo
I2NpcmNsZSk7Cn0KCi5zdmdib2IgLmVuZF9tYXJrZWRfb3Blbl9jaXJjbGUgewogIG1hcmtlci1l
bmQ6IHVybCgjb3Blbl9jaXJjbGUpOwp9Cgouc3ZnYm9iIC5zdGFydF9tYXJrZWRfb3Blbl9jaXJj
bGUgewogIG1hcmtlci1zdGFydDogdXJsKCNvcGVuX2NpcmNsZSk7Cn0KCi5zdmdib2IgLmVuZF9t
YXJrZWRfYmlnX29wZW5fY2lyY2xlIHsKICBtYXJrZXItZW5kOiB1cmwoI2JpZ19vcGVuX2NpcmNs
ZSk7Cn0KCi5zdmdib2IgLnN0YXJ0X21hcmtlZF9iaWdfb3Blbl9jaXJjbGUgewogIG1hcmtlci1z
dGFydDogdXJsKCNiaWdfb3Blbl9jaXJjbGUpOwp9Cgo8L3N0eWxlPgogIDxkZWZzPgogICAgPG1h
cmtlciBpZD0iYXJyb3ciIHZpZXdCb3g9Ii0yIC0yIDggOCIgcmVmWD0iNCIgcmVmWT0iMiIgbWFy
a2VyV2lkdGg9IjciIG1hcmtlckhlaWdodD0iNyIgb3JpZW50PSJhdXRvLXN0YXJ0LXJldmVyc2Ui
PgogICAgICA8cG9seWdvbiBwb2ludHM9IjAsMCAwLDQgNCwyIDAsMCI+PC9wb2x5Z29uPgogICAg
PC9tYXJrZXI+CiAgICA8bWFya2VyIGlkPSJkaWFtb25kIiB2aWV3Qm94PSItMiAtMiA4IDgiIHJl
Zlg9IjQiIHJlZlk9IjIiIG1hcmtlcldpZHRoPSI3IiBtYXJrZXJIZWlnaHQ9IjciIG9yaWVudD0i
YXV0by1zdGFydC1yZXZlcnNlIj4KICAgICAgPHBvbHlnb24gcG9pbnRzPSIwLDIgMiwwIDQsMiAy
LDQgMCwyIj48L3BvbHlnb24+CiAgICA8L21hcmtlcj4KICAgIDxtYXJrZXIgaWQ9ImNpcmNsZSIg
dmlld0JveD0iMCAwIDggOCIgcmVmWD0iNCIgcmVmWT0iNCIgbWFya2VyV2lkdGg9IjciIG1hcmtl
ckhlaWdodD0iNyIgb3JpZW50PSJhdXRvLXN0YXJ0LXJldmVyc2UiPgogICAgICA8Y2lyY2xlIGN4
PSI0IiBjeT0iNCIgcj0iMiIgY2xhc3M9ImZpbGxlZCI+PC9jaXJjbGU+CiAgICA8L21hcmtlcj4K
ICAgIDxtYXJrZXIgaWQ9Im9wZW5fY2lyY2xlIiB2aWV3Qm94PSIwIDAgOCA4IiByZWZYPSI0IiBy
ZWZZPSI0IiBtYXJrZXJXaWR0aD0iNyIgbWFya2VySGVpZ2h0PSI3IiBvcmllbnQ9ImF1dG8tc3Rh
cnQtcmV2ZXJzZSI+CiAgICAgIDxjaXJjbGUgY3g9IjQiIGN5PSI0IiByPSIyIiBjbGFzcz0iYmdf
ZmlsbGVkIj48L2NpcmNsZT4KICAgIDwvbWFya2VyPgogICAgPG1hcmtlciBpZD0iYmlnX29wZW5f
Y2lyY2xlIiB2aWV3Qm94PSIwIDAgOCA4IiByZWZYPSI0IiByZWZZPSI0IiBtYXJrZXJXaWR0aD0i
NyIgbWFya2VySGVpZ2h0PSI3IiBvcmllbnQ9ImF1dG8tc3RhcnQtcmV2ZXJzZSI+CiAgICAgIDxj
aXJjbGUgY3g9IjQiIGN5PSI0IiByPSIzIiBjbGFzcz0iYmdfZmlsbGVkIj48L2NpcmNsZT4KICAg
IDwvbWFya2VyPgogIDwvZGVmcz4KICA8cmVjdCBjbGFzcz0iYmFja2Ryb3AiIHg9IjAiIHk9Ii00
IiB3aWR0aD0iNTIyLjYwMDA0IiBoZWlnaHQ9IjUwLjgwMDAwMyI+PC9yZWN0PgogIDx0ZXh0IHht
bDpzcGFjZT0icHJlc2VydmUiIHg9IjEuOTUiIHk9IjExLjcwMDAwMSIgPkNMT0NLPC90ZXh0Pgog
IDx0ZXh0IHhtbDpzcGFjZT0icHJlc2VydmUiIHg9IjQ4Ljc1IiB5PSIxMS43MDAwMDEiID5IYW5k
PC90ZXh0PgogIDxsaW5lIHgxPSI4NS44IiB5MT0iNy44IiB4Mj0iMTAxLjQiIHkyPSI3LjgiIGNs
YXNzPSJzb2xpZCI+PC9saW5lPgogIDxwb2x5Z29uIHBvaW50cz0iMTAxLjQsMy45IDEwOS4yMDAw
MDUsNy44IDEwMS40LDExLjcwMDAwMSIgY2xhc3M9ImZpbGxlZCI+PC9wb2x5Z29uPgogIDxsaW5l
IHgxPSIzMjcuNiIgeTE9IjcuOCIgeDI9IjM0My4yIiB5Mj0iNy44IiBjbGFzcz0ic29saWQiPjwv
bGluZT4KICA8cG9seWdvbiBwb2ludHM9IjM0My4yLDMuOSAzNTEsNy44IDM0My4yLDExLjcwMDAw
MSIgY2xhc3M9ImZpbGxlZCI+PC9wb2x5Z29uPgogIDxsaW5lIHgxPSIzMjcuNiIgeTE9IjIzLjQw
MDAwMiIgeDI9IjM0My4yIiB5Mj0iMjMuNDAwMDAyIiBjbGFzcz0ic29saWQiPjwvbGluZT4KICA8
cG9seWdvbiBwb2ludHM9IjM0My4yLDE5LjUgMzUxLDIzLjQwMDAwMiAzNDMuMiwyNy4zMDAwMDEi
IGNsYXNzPSJmaWxsZWQiPjwvcG9seWdvbj4KICA8dGV4dCB4bWw6c3BhY2U9InByZXNlcnZlIiB4
PSIzNjAuNzUiIHk9IjI3LjMwMDAwMSIgPkV2aWN0PC90ZXh0PgogIDx0ZXh0IHhtbDpzcGFjZT0i
cHJlc2VydmUiIHg9IjQwNy41NTAwMiIgeT0iMjcuMzAwMDAxIiA+VmljdGltPC90ZXh0PgogIDx0
ZXh0IHhtbDpzcGFjZT0icHJlc2VydmUiIHg9IjQ2Mi4xNTAwMiIgeT0iMjcuMzAwMDAxIiA+Rm91
bmQhPC90ZXh0PgogIDx0ZXh0IHhtbDpzcGFjZT0icHJlc2VydmUiIHg9IjExOC45NTAwMDUiIHk9
IjExLjcwMDAwMSIgPlsgRnJhbWUgMDogUmVmPTEsIFBpbj0wIF08L3RleHQ+CiAgPHRleHQgeG1s
OnNwYWNlPSJwcmVzZXJ2ZSIgeD0iMzYwLjc1IiB5PSIxMS43MDAwMDEiID5DbGVhciBSZWY9MCwg
QWR2YW5jZTwvdGV4dD4KICA8dGV4dCB4bWw6c3BhY2U9InByZXNlcnZlIiB4PSIxMTguOTUwMDA1
IiB5PSIyNy4zMDAwMDEiID5bIEZyYW1lIDE6IFJlZj0wLCBQaW49MCBdPC90ZXh0Pgo8L3N2Zz4K
" alt="" />
<figcaption>Figure 16–15: CLOCK Algorithm</figcaption>
</figure>

**Specification:**

Implement a `BufferPoolManager` maintaining an array of in-memory page
slots. Implement `fetch_page`, `unpin_page`, and the CLOCK replacement
policy, handling dirty page identification.

``` cpp
class BufferPoolManager {
public:
    static constexpr size_t PAGE_SIZE = 4096;
    explicit BufferPoolManager(size_t pool_size);
    char* fetch_page(size_t page_id);
    void unpin_page(size_t page_id, bool is_dirty);
    void flush_all();
private:
    struct Frame {
        size_t page_id{0};
        uint32_t pin_count{0};
        bool is_dirty{false};
        bool reference_bit{false};
        std::array<char, PAGE_SIZE> data;
    };
    std::vector<Frame> m_frames;
    size_t m_clock_hand{0};
    size_t find_victim_frame();
};
```

**Verification Tasks:**

1.  **Cache Hit Verification:** Request page $5$ repeatedly across $10$
    iterations. Verify through an internal tracking counter that
    secondary storage read is invoked exactly once.
2.  **Pinned Page Immortality:** Pin every frame in the pool
    ($\text{Pin Count} > 0$). Request an un-cached page ID. Verify that
    the eviction algorithm refuses to evict active frames and throws a
    `std::runtime_error("Buffer pool exhausted")`.
3.  **Dirty Writeback Invariant:** Fetch page $2$, mutate its memory,
    and unpin with `is_dirty = true`. Force eviction by requesting new
    pages. Verify that the system executes a physical disk write before
    overwriting the slot.

#### Exercise 16.12: Foreign Key Constraint Enforcement and Referential Cascades

**Context and Theory:**

The relational model enforces **Referential Integrity** through Foreign
Key constraints. Let child relation $R_c$ declare foreign key $A_c$
referencing primary key $A_p$ of parent relation $R_p$:

$$\forall t_c \in R_c : t_c[A_c] \in \pi_{A_p}(R_p) \lor t_c[A_c] = \text{NULL}$$

Mutations must safeguard this invariant (see Figure 16–16):

1.  **Child Insertion/Update:** An inserted child row must be rejected
    if its foreign key value does not exist within the parent primary
    key index.
2.  **Parent Deletion/Update Actions:**
    - `RESTRICT`: The deletion of a parent record is rejected if
      referenced by any child rows.
    - `CASCADE`: Deleting a parent row triggers automatic deletion of
      all dependent child records across the foreign key link.

<figure id="fig:referential-cascades">
<img src="data:image/svg+xml;base64,PHN2ZyB4bWxucz0iaHR0cDovL3d3dy53My5vcmcvMjAwMC9zdmciIHdpZHRoPSI1MTQuOCIgaGVp
Z2h0PSI5Ny42MDAwMDYiIGNsYXNzPSJzdmdib2IiIHZpZXdCb3g9IjAgLTQgNTE0LjggOTcuNjAw
MDA2Ij4KICA8c3R5bGU+LnN2Z2JvYiBsaW5lLCAuc3ZnYm9iIHBhdGgsIC5zdmdib2IgY2lyY2xl
LCAuc3ZnYm9iIHJlY3QsIC5zdmdib2IgcG9seWdvbiB7CiAgc3Ryb2tlOiBibGFjazsKICBzdHJv
a2Utd2lkdGg6IDEuNTsKICBzdHJva2Utb3BhY2l0eTogMTsKICBmaWxsLW9wYWNpdHk6IDE7CiAg
c3Ryb2tlLWxpbmVjYXA6IHJvdW5kOwogIHN0cm9rZS1saW5lam9pbjogbWl0ZXI7Cn0KCi5zdmdi
b2IgdGV4dCB7CiAgd2hpdGUtc3BhY2U6IHByZTsKICBmaWxsOiBibGFjazsKICBmb250LWZhbWls
eTogQ291cmllciBOZXcsIERlamFWdSBTYW5zIE1vbm8sIG1vbm9zcGFjZTsKICBmb250LXNpemU6
IDEzcHg7Cn0KCi5zdmdib2IgcmVjdC5iYWNrZHJvcCB7CiAgc3Ryb2tlOiBub25lOwogIGZpbGw6
IHRyYW5zcGFyZW50Owp9Cgouc3ZnYm9iIC5icm9rZW4gewogIHN0cm9rZS1kYXNoYXJyYXk6IDg7
Cn0KCi5zdmdib2IgLmZpbGxlZCB7CiAgZmlsbDogYmxhY2s7Cn0KCi5zdmdib2IgLmJnX2ZpbGxl
ZCB7CiAgZmlsbDogdHJhbnNwYXJlbnQ7CiAgc3Ryb2tlLXdpZHRoOiAxOwp9Cgouc3ZnYm9iIC5u
b2ZpbGwgewogIGZpbGw6IHRyYW5zcGFyZW50Owp9Cgouc3ZnYm9iIC5lbmRfbWFya2VkX2Fycm93
IHsKICBtYXJrZXItZW5kOiB1cmwoI2Fycm93KTsKfQoKLnN2Z2JvYiAuc3RhcnRfbWFya2VkX2Fy
cm93IHsKICBtYXJrZXItc3RhcnQ6IHVybCgjYXJyb3cpOwp9Cgouc3ZnYm9iIC5lbmRfbWFya2Vk
X2RpYW1vbmQgewogIG1hcmtlci1lbmQ6IHVybCgjZGlhbW9uZCk7Cn0KCi5zdmdib2IgLnN0YXJ0
X21hcmtlZF9kaWFtb25kIHsKICBtYXJrZXItc3RhcnQ6IHVybCgjZGlhbW9uZCk7Cn0KCi5zdmdi
b2IgLmVuZF9tYXJrZWRfY2lyY2xlIHsKICBtYXJrZXItZW5kOiB1cmwoI2NpcmNsZSk7Cn0KCi5z
dmdib2IgLnN0YXJ0X21hcmtlZF9jaXJjbGUgewogIG1hcmtlci1zdGFydDogdXJsKCNjaXJjbGUp
Owp9Cgouc3ZnYm9iIC5lbmRfbWFya2VkX29wZW5fY2lyY2xlIHsKICBtYXJrZXItZW5kOiB1cmwo
I29wZW5fY2lyY2xlKTsKfQoKLnN2Z2JvYiAuc3RhcnRfbWFya2VkX29wZW5fY2lyY2xlIHsKICBt
YXJrZXItc3RhcnQ6IHVybCgjb3Blbl9jaXJjbGUpOwp9Cgouc3ZnYm9iIC5lbmRfbWFya2VkX2Jp
Z19vcGVuX2NpcmNsZSB7CiAgbWFya2VyLWVuZDogdXJsKCNiaWdfb3Blbl9jaXJjbGUpOwp9Cgou
c3ZnYm9iIC5zdGFydF9tYXJrZWRfYmlnX29wZW5fY2lyY2xlIHsKICBtYXJrZXItc3RhcnQ6IHVy
bCgjYmlnX29wZW5fY2lyY2xlKTsKfQoKPC9zdHlsZT4KICA8ZGVmcz4KICAgIDxtYXJrZXIgaWQ9
ImFycm93IiB2aWV3Qm94PSItMiAtMiA4IDgiIHJlZlg9IjQiIHJlZlk9IjIiIG1hcmtlcldpZHRo
PSI3IiBtYXJrZXJIZWlnaHQ9IjciIG9yaWVudD0iYXV0by1zdGFydC1yZXZlcnNlIj4KICAgICAg
PHBvbHlnb24gcG9pbnRzPSIwLDAgMCw0IDQsMiAwLDAiPjwvcG9seWdvbj4KICAgIDwvbWFya2Vy
PgogICAgPG1hcmtlciBpZD0iZGlhbW9uZCIgdmlld0JveD0iLTIgLTIgOCA4IiByZWZYPSI0IiBy
ZWZZPSIyIiBtYXJrZXJXaWR0aD0iNyIgbWFya2VySGVpZ2h0PSI3IiBvcmllbnQ9ImF1dG8tc3Rh
cnQtcmV2ZXJzZSI+CiAgICAgIDxwb2x5Z29uIHBvaW50cz0iMCwyIDIsMCA0LDIgMiw0IDAsMiI+
PC9wb2x5Z29uPgogICAgPC9tYXJrZXI+CiAgICA8bWFya2VyIGlkPSJjaXJjbGUiIHZpZXdCb3g9
IjAgMCA4IDgiIHJlZlg9IjQiIHJlZlk9IjQiIG1hcmtlcldpZHRoPSI3IiBtYXJrZXJIZWlnaHQ9
IjciIG9yaWVudD0iYXV0by1zdGFydC1yZXZlcnNlIj4KICAgICAgPGNpcmNsZSBjeD0iNCIgY3k9
IjQiIHI9IjIiIGNsYXNzPSJmaWxsZWQiPjwvY2lyY2xlPgogICAgPC9tYXJrZXI+CiAgICA8bWFy
a2VyIGlkPSJvcGVuX2NpcmNsZSIgdmlld0JveD0iMCAwIDggOCIgcmVmWD0iNCIgcmVmWT0iNCIg
bWFya2VyV2lkdGg9IjciIG1hcmtlckhlaWdodD0iNyIgb3JpZW50PSJhdXRvLXN0YXJ0LXJldmVy
c2UiPgogICAgICA8Y2lyY2xlIGN4PSI0IiBjeT0iNCIgcj0iMiIgY2xhc3M9ImJnX2ZpbGxlZCI+
PC9jaXJjbGU+CiAgICA8L21hcmtlcj4KICAgIDxtYXJrZXIgaWQ9ImJpZ19vcGVuX2NpcmNsZSIg
dmlld0JveD0iMCAwIDggOCIgcmVmWD0iNCIgcmVmWT0iNCIgbWFya2VyV2lkdGg9IjciIG1hcmtl
ckhlaWdodD0iNyIgb3JpZW50PSJhdXRvLXN0YXJ0LXJldmVyc2UiPgogICAgICA8Y2lyY2xlIGN4
PSI0IiBjeT0iNCIgcj0iMyIgY2xhc3M9ImJnX2ZpbGxlZCI+PC9jaXJjbGU+CiAgICA8L21hcmtl
cj4KICA8L2RlZnM+CiAgPHJlY3QgY2xhc3M9ImJhY2tkcm9wIiB4PSIwIiB5PSItNCIgd2lkdGg9
IjUxNC44IiBoZWlnaHQ9Ijk3LjYwMDAwNiI+PC9yZWN0PgogIDxwb2x5Z29uIHBvaW50cz0iMjI2
LjIwMDAxLDMuOSAyMTguNDAwMDEsNy44IDIyNi4yMDAwMSwxMS43MDAwMDEiIGNsYXNzPSJmaWxs
ZWQiPjwvcG9seWdvbj4KICA8dGV4dCB4bWw6c3BhY2U9InByZXNlcnZlIiB4PSIyOTAuNTUwMDIi
IHk9IjI3LjMwMDAwMSIgPkZvcmVpZ248L3RleHQ+CiAgPHRleHQgeG1sOnNwYWNlPSJwcmVzZXJ2
ZSIgeD0iMzUyLjk1IiB5PSIyNy4zMDAwMDEiID5LZXk8L3RleHQ+CiAgPHRleHQgeG1sOnNwYWNl
PSJwcmVzZXJ2ZSIgeD0iMzg0LjE1MDAyIiB5PSIyNy4zMDAwMDEiID5SZWZlcmVuY2U8L3RleHQ+
CiAgPHRleHQgeG1sOnNwYWNlPSJwcmVzZXJ2ZSIgeD0iMS45NSIgeT0iNzQuMSIgPls8L3RleHQ+
CiAgPHRleHQgeG1sOnNwYWNlPSJwcmVzZXJ2ZSIgeD0iMTcuNTUwMDAxIiB5PSI3NC4xIiA+QWN0
aW9uOjwvdGV4dD4KICA8dGV4dCB4bWw6c3BhY2U9InByZXNlcnZlIiB4PSI3OS45NTAwMDUiIHk9
Ijc0LjEiID5DQVNDQURFPC90ZXh0PgogIDx0ZXh0IHhtbDpzcGFjZT0icHJlc2VydmUiIHg9IjE0
Mi4zNSIgeT0iNzQuMSIgPkRlbGV0ZTwvdGV4dD4KICA8dGV4dCB4bWw6c3BhY2U9InByZXNlcnZl
IiB4PSIxOTYuOTUwMDEiIHk9Ijc0LjEiID5QYXJlbnQ8L3RleHQ+CiAgPGxpbmUgeDE9IjI0OS42
IiB5MT0iNzAuMjAwMDA1IiB4Mj0iMjczIiB5Mj0iNzAuMjAwMDA1IiBjbGFzcz0ic29saWQiPjwv
bGluZT4KICA8cG9seWdvbiBwb2ludHM9IjI3Myw2Ni4zIDI4MC44MDAwMiw3MC4yMDAwMDUgMjcz
LDc0LjEiIGNsYXNzPSJmaWxsZWQiPjwvcG9seWdvbj4KICA8dGV4dCB4bWw6c3BhY2U9InByZXNl
cnZlIiB4PSIyOTAuNTUwMDIiIHk9Ijc0LjEiID5BdXRvbWF0aWNhbGx5PC90ZXh0PgogIDx0ZXh0
IHhtbDpzcGFjZT0icHJlc2VydmUiIHg9IjM5OS43NSIgeT0iNzQuMSIgPlBydW5lczwvdGV4dD4K
ICA8dGV4dCB4bWw6c3BhY2U9InByZXNlcnZlIiB4PSI0NTQuMzUiIHk9Ijc0LjEiID5DaGlsZDwv
dGV4dD4KICA8dGV4dCB4bWw6c3BhY2U9InByZXNlcnZlIiB4PSI1MDEuMTUwMDIiIHk9Ijc0LjEi
ID5dPC90ZXh0PgogIDx0ZXh0IHhtbDpzcGFjZT0icHJlc2VydmUiIHg9IjEuOTUiIHk9IjExLjcw
MDAwMSIgPlBhcmVudDogRGVwYXJ0bWVudCAoaWQ6IDEpPC90ZXh0PgogIDx0ZXh0IHhtbDpzcGFj
ZT0icHJlc2VydmUiIHg9IjEuOTUiIHk9IjQyLjkiID5DaGlsZDogIEVuZ2luZWVyIChkZXB0X2lk
OiAxKTwvdGV4dD4KICA8Zz4KICAgIDxsaW5lIHgxPSIyMjYuMjAwMDEiIHkxPSI3LjgiIHgyPSIy
NzYuOSIgeTI9IjcuOCIgY2xhc3M9InNvbGlkIj48L2xpbmU+CiAgICA8bGluZSB4MT0iMjc2Ljki
IHkxPSI3LjgiIHgyPSIyNzYuOSIgeTI9IjM5IiBjbGFzcz0ic29saWQiPjwvbGluZT4KICAgIDxs
aW5lIHgxPSIyNDEuOCIgeTE9IjM5IiB4Mj0iMjc2LjkiIHkyPSIzOSIgY2xhc3M9InNvbGlkIj48
L2xpbmU+CiAgPC9nPgo8L3N2Zz4K
" alt="" />
<figcaption>Figure 16–16: Referential Integrity through
Cascades</figcaption>
</figure>

**Specification:**

Construct a `ForeignKeyConstraint` descriptor linking two `Table`
instances. Implement enforcement interceptors within `insert_record` and
`delete_records` supporting both `RESTRICT` and `CASCADE` policies.

``` cpp
enum class ReferentialAction { Restrict, Cascade, SetNull };

struct ForeignKeyConstraint {
    std::string child_col;
    Table& parent_table;
    std::string parent_pk_col;
    ReferentialAction on_delete;
};

class ConstraintManager {
public:
    void register_fk(Table& child_table, ForeignKeyConstraint fk);
    void validate_child_insert(const Table& child_table, const Row& row);
    void handle_parent_delete(Table& parent_table, const Row& deleted_row);
private:
    std::unordered_map<std::string, std::vector<ForeignKeyConstraint>> m_fk_registry;
};
```

**Verification Tasks:**

1.  **Invalid Foreign Key Rejection:** Attempt to insert an employee
    with `dept_id = 999` when the department table contains only IDs
    $\{1, 2, 3\}$. Verify that the manager rejects the insert with a
    `ConstraintViolationException`.
2.  **Restrict Policy Enforcement:** Attempt to delete department
    `id = 1` while active child engineer records reference it under a
    `RESTRICT` policy. Verify that the parent deletion is aborted and
    the table remains unmodified.
3.  **Cascading Child Pruning:** Configure a foreign key with `CASCADE`.
    Delete department `id = 1`. Verify that the parent record is deleted
    and that all child engineer records with `dept_id = 1` are expunged
    from the child table and its indices.